# 券商分點 V1.6 量化研究模型 — 15 個子策略獨立 Dashboard（依《優化建議》逐條升級）

依《券商分點六大層級》文件逐條實作，每一條策略輸出成**一個獨立 HTML 網頁**，另附總覽頁 `index.html`。

| # | 子策略 | 對應文件章節 |
|---|---|---|
| 01 | 分點買賣超（1/3/5/10/20 日累積） | 一 |
| 02 | 持續性主力・分點強度分數 0～100 | 二 |
| 03 | 主力成本區估算 | 三 |
| 04 | 主力成本密集區（成本分布） | 四 |
| 05 | 隔日沖券商辨識・隔日沖比例 | 五 |
| 06 | 長線／短線主力類型分類 | 六 |
| 07 | 分點集中度 | 七 |
| 08 | 買賣家數差 | 八 |
| 09 | 吸籌偵測 | 九 |
| 10 | 出貨偵測 | 十 |
| 11 | 主力反轉訊號（轉空／轉多） | 十一 |
| 12 | 主力行為矩陣 | 十二 |
| 13 | 主力籌碼 Score | 十三 |
| 14 | 分點量化選股漏斗 | 十四 |
| 15 | 十大分點因子 × 回測驗證 | 十五 + V1.0 架構 |

**V1.1 優化對照**（參數集中在「1. 參數設定」的 V1.1 區塊）

| # | 優化內容 | 主要參數 |
|---|---|---|
| S01 | EMA 加權累積買超＋產業板塊合計（族群建倉偵測） | `EMA_SPANS`、`SECTOR_MIN_STOCKS` |
| S02 | 動態流動性門檻：依 60 日均量百分位在 0.4%～0.1% 間內插（全模型「有效買超」共用） | `MAT_RATIO_ILLIQ/LIQUID`、`MAT_MIN_LOTS` |
| S03 | 庫存扣減可選 AVG／FIFO／LIFO＋支撐效力時間衰減 | `COST_METHOD`、`SUPPORT_HALFLIFE` |
| S04 | 加權高斯核密度估計（KDE）取代 14 格直方圖，標示波峰數 | `KDE_GRID`、`KDE_BW_ADJ` |
| S05 | T+1 隔日沖＋2／3 日內淨賣回 → 隔日沖／隔兩日沖／三日短波段特徵庫 | `DT_HORIZON`、`DT_CLEAR_RATIO` |
| S06 | K-Means 機器分群＋歷史跟單超額報酬評估各群 | `N_CLUSTERS` |
| S07 | 集中度分母改為「成交量 − 當沖量」實質留倉量（TWSE 當沖統計） | `DAYTRADE_YEARS` |
| S08 | 全市場分點買進／賣出家數＋前 15 大淨流向（富邦個股主力進出頁） | `BREADTH_STOCKS`、`BREADTH_DAYS` |
| S09 | ⑥隱性吸籌：收黑但長下影線、跌不下去（無五檔歷史時的 K 線代理） | `ABSORB_SHADOW` |
| S10 | 散戶接盤：融資暴增、零股量創高 → 出貨警報(散戶接盤) | `MARGIN_SURGE`、`ODD_LOT_DAYS` |
| S11 | 防守線改為 Chandelier ATR 移動停損／主力成本密集區上下緣 | `REV_BREAK_MODE`、`ATR_N`、`ATR_MULT` |
| S12 | PCA 主成分降維＋因子相關矩陣 | `PCA_COMPONENTS`、`PCA_FIT_DAYS` |
| S13 | Random Forest walk-forward 機器學習版 Score（無前視），列出學到的權重與樣本外 IC | `ML_ENABLE`、`ML_HORIZON`、`ML_RETRAIN` |
| S14 | Z-Score 軟性評分排序（隔日沖仍硬性排除），列出「僅差一關」個股 | `SOFT_Z_PASS` |
| S15 | 跳動點價差＋平方根市場衝擊滑價，依 100 萬／1000 萬／5000 萬資金各算淨值；規則版 vs ML 版對照 | `CAPITAL_LEVELS`、`IMPACT_K` |

**資料來源**：富邦 eBroker 分點進出（zgb0，張數＋金額兩種模式逐日抓取）、yfinance 股價、TWSE T86 三大法人、TWSE MI_MARGN 融資；V1.1 新增 TWSE 當沖統計（TWTB4U）、零股成交、ISIN 產業別、富邦個股主力進出（zco，全市場家數）。

**使用方式**：`執行階段 → 全部執行`，第一格會要求授權 Google 雲端硬碟（快取存在 `MyDrive/broker_dashboards/_cache`）。

**資料範圍**：5 個分點 × 近 3 年（約 735 交易日 × 2 模式 ≈ 7,350 頁）＋ TWSE 法人/融資 3 年。第一次抓取約 **2～2.5 小時**（V1.4 上櫃法人/融資與上市同時並行抓取，不增加總時間）（分點約 40 分、法人融資約 55 分、V1.3 當沖改抓滿 3 年約 55 分、全市場家數 80 檔×20 日約 15 分）。請保持分頁開啟；若中途斷線，直接重新「全部執行」，已抓的部分從雲端硬碟快取讀取，只補缺的。之後每天重跑只需幾分鐘。

完成後最後一格會打包 zip 下載，也可用 `show_dashboard(編號)` 直接在 Colab 內預覽。

> ⚠️ 分點資料是「交易通路」不是投資人身分；每個分點每日只列出買超／賣超各前 50 名個股，未上榜的個股視為 0。所有分數皆為研究用途，不構成投資建議。

## V1.6：每日排程自動執行（GitHub Actions）

- 本 notebook 同時支援 **Colab 手動執行**（不變）與 **GitHub Actions 每日排程**（環境變數 `BROKER_RUN_MODE=ci`）。
- 排程模式：不掛載雲端硬碟、輸出到 `BROKER_OUTPUT_DIR`；跑完寫出 `build_info.json`（資料日期、覆蓋率、自我測試結果）。
- 無人值守時，**自我測試失敗或最新交易日完全沒有分點資料會讓工作失敗**（不發佈錯誤的網頁；GitHub 會寄失敗通知信）。
- 排程與部署設定見 repo 內 `README.md` 與 `.github/workflows/broker-daily.yml`。

## V1.5 優化（接續 V1.4）

- **除權息成本調整**：分點買進價先換到「還原價尺度」再算庫存成本、成本區與成本密集區，並與還原收盤比較。除息後不會再把「股價掉了一個股息」誤判成跌破主力成本（影響 S03、S04、S11、S12～S15 的 DIST／成本支撐）。
- **Chandelier 停損改用最高／最低價**（還原後），符合原始定義；缺高低價時退回收盤。
- **Top50 截尾診斷**：統計每個分點「買超或賣超列滿 50 檔」的日子比例（觸頂時未上榜的交易被當成 0），顯示在頁首並列出各分點。
- **存活者偏誤診斷**：分點交易過、但 yfinance 抓不到股價（多為下市／轉板）的股票數與其淨買賣量占比，顯示在 S15。
- **t 值改用「間隔 ≥ h 日」的不重疊抽樣**（S13 IC、S15 因子／IC），訊號稀疏時不再丟掉有效樣本。
- **因子統計也排除進場日漲停（買不到）的訊號**，與組合回測一致。
- S04 最新日無股價（停牌）的個股不再被誤判為「成本區內」。
- 網頁表格與選單的文字一律跳脫（HTML escape），股名含特殊字元不會破版。
- `OFFLINE_HTML = True` 可把 Chart.js 內嵌進每頁，手機或離線也能看圖。
- 每個執行緒各用一個連線 Session（requests.Session 非執行緒安全）。

## V1.4 優化（接續 V1.3）

**資料與一致性**
- 上櫃股補抓 TPEx 外資／投信／融資（`TPEX_CHIPS`），與 TWSE 並行抓取；TPEx 連不上時自動退回中性分，不中斷。
- 成本定義統一：`SUPPORT_HALFLIFE` 預設 = `COST_HALFLIFE`；S03 個股列新增「Score 用成本（引擎）」欄，看得到 Score 實際用的成本。
- 每頁頁首顯示分點日覆蓋率；最新交易日若有分點缺資料會警告。
- 已驗證的「當日無交易」分點頁（連續兩次空頁且日期正確）會快取，不再每次重抓。
- 修正 S03 說明與程式不符、摘要 S08 欄位標錯。

**研究方法**
- S15：Benjamini-Hochberg FDR 多重檢定（`FDR_Q`）、樣本內 vs 保留期（`HOLDOUT_DAYS`）、持有天數 × 檔數參數敏感度表。
- S15 組合回測：漲停買不到改買次一名、跌停賣不掉順延出場（`LIMIT_TH`），滑價依淨值複利後的資金計算。
- S06：機器分群的跟單績效拆前／後段，兩段皆正才標為優勢群（避免樣本內挑群）。
- S13：最近一次模型在未訓練過的已實現樣本上計算 permutation importance 與 AUC。

**效能與維護**
- 股價快取改為增量更新：只補最近幾天；偵測到除權息（還原價改寫）的個股才整檔重抓。
- 參數檢查（`check_params`）；改了參數卻沒重跑因子引擎時，各頁輸出會警告。
- 新增自我測試格（`RUN_SELF_TESTS`）：庫存法、KDE、無前視、分點缺漏加總等。

**未處理（需外部資料）**：下市股股價（存活者偏誤）、交易所五檔/內外盤歷史、上櫃當沖量（仍以估計補值）。

## V1.3 修正（接續 V1.2）

1. **分點日缺漏不再污染合計**：`NET_T`／`BUY_T`／`SELL_T`／`AMTN_T`／`POSQ_T`／`POSW_T` 改為「有任一分點有資料就加總」（`min_count=1`），單一分點某日抓取失敗不會讓當日合計變 NaN；`ACTIVE20` 不再把 NaN 當成有交易。
2. **Python 3.11 以下相容**：修正 Cell 5 巢狀同種引號的 f-string。
3. **當沖量與回看期一致**：`DAYTRADE_YEARS` 預設 = `LOOKBACK_YEARS`，避免集中度序列在一年前出現「扣／不扣當沖」的結構斷層；若當沖資料仍未涵蓋全期會印出警告。
4. **門檻集中管理**：隔日沖（`DT_FLAG_RATIO`、`DT_PAIR_RATIO`）與集中度（`CONC_TIER_HIGH`、`CONC_TIER_MID`、`CONC_SCORE_FULL`）全部移到參數區，各頁程式與說明文字都讀同一個參數。**數值與 V1.2 相同，結果不變**，只是改參數終於會全域生效。
5. **打包只含 HTML**：zip 不再包進 `_cache`（數千個快取檔），檔案小、打包快。

## V1.2 審查修正版：使用前必讀

- 本版依原始程式做資料時間軸、缺漏與局部策略修正；**不是經三年真實資料重新驗證的獲利優化版**。
- 原附件無儲存輸出與快取，本次未下載全部市場資料；測試為確定性邊界案例及介面相容性，不是市場績效。
- 部分模組是描述、風控或排序工具，不是15個互相獨立的交易策略。
- 仍需真實分點/股價快取、point-in-time股票池、來源發布時間、上市櫃補齊、除權息庫存調整、漲跌停/停牌可成交性以及樣本外測試。
- Top50未列示仍以零作截尾代理，無法還原未知交易；新增覆蓋旗標不等於消除截尾偏誤。
- 交易日曆取得失敗會停止；舊空/無單位分點快取需重抓；股價快取使用新版本與股票池雜湊。
- 訊號於t收盤後形成、t+1收盤進場，是保守研究假設，不是t+1開盤成交模型。
- 原文「無前視」只應解讀為設計目標；本次修正沒有認證整個系統已消除全部偏誤。參數仍待驗證，勿直接實盤。
- 分點是交易通路，不代表單一主力；僅供研究分析，非個人化投資建議。

原始 cell 編號可對照報告；此說明新增於原第1格後，其餘格順序不變。


In [ ]:
!pip -q install yfinance scikit-learn scipy

In [ ]:
# ============ 1. 參數設定 ============
import os, re, json, time, math, glob, shutil, warnings, datetime as dt
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import requests
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

# 追蹤分點：名稱 -> (a 參數, b 參數)
BRANCHES = {
    '元大-竹北':   ('9800', '0039003800310042'),
    '永豐金-市政': ('9A00', '0039004100390057'),
    '凱基-竹科':   ('9200', '9272'),
    '富邦-員林':   ('9600', '9672'),
    '國票-台中':   ('7790', '0037003700390076'),
}
BASE_URL = 'https://fubon-ebrokerdj.fbs.com.tw/z/zg/zgb/zgb0.djhtm'

LOOKBACK_YEARS  = 3       # 分點資料抓取年數（約 735 個交易日；5 分點 × 2 模式 ≈ 7,350 頁）
INST_YEARS      = 3       # TWSE 外資/投信/融資抓取年數（每日 2 次請求、約 4.5 秒/日；設 0 = 不抓）
MAX_WORKERS     = 4       # 分點頁並行數（勿調太高，避免被擋）
REQUEST_SLEEP   = 0.35    # 每次請求後休息秒數
USE_DEMO_DATA   = False   # True = 使用模擬資料（僅供測試版面，所有數字無意義）
SAVE_TO_DRIVE   = True    # 3 年資料首次抓取約 1～1.5 小時，快取存雲端硬碟，斷線重跑只補缺的部分

UNIVERSE_MODE   = 'ALL'   # 'ALL' = 追蹤分點交易過的所有上市櫃普通股；'0050' = 只看 0050 成分股
UNIVERSE_0050 = ['2330','2317','2454','2308','2382','2891','2881','2412','2882','2303','3711','2886',
                 '2884','1216','2357','2885','2892','3034','2345','2890','5880','2327','3231','2880',
                 '3008','2002','1303','2603','2379','6669','3017','2887','4938','1301','2301','2395',
                 '3045','2912','5871','2883','6505','1101','4904','2207','3661','2059','3653','1590',
                 '2615','6919']   # 成分股會調整，請依最新公告自行更新

# 策略門檻
COST_WINDOW      = 20     # 成本估算短視窗（日）
COST_LONG_WINDOW = 60     # 成本估算中期視窗（日）
DENSITY_WINDOW   = 100    # 成本密集區（S04）使用的最近交易日數
DT_MIN_LOTS      = 50     # 隔日沖事件：單日淨買超至少幾張
DT_ALERT_RATIO   = 0.90   # 隔日沖比例警示（文件建議 90%）
CONC_HIGH        = 0.10   # S15 因子②「集中度 ≥ 門檻」使用的門檻
CONC_MID         = 0.05   # S07 輕度集中下限、S14 漏斗集中度門檻
MATERIAL_RATIO   = 0.002  # 「有效買超」門檻：淨買超 ≥ 同期成交量 0.2%（過濾 1、2 張的雜訊買超）
COST_HALFLIFE    = 20     # 因子引擎「庫存成本」半衰期（日）：買進加權、賣出扣減、逐日衰減
DT_PENALTY       = True   # 主力籌碼 Score：隔日沖比例 ≥ DT_FLAG_RATIO 的個股，5 日買超與集中度不給分
# V1.3 門檻集中管理（數值同 V1.2）
DT_FLAG_RATIO    = 0.50   # 股票層級隔日沖率 ≥ 此值：S13 扣分、S14 硬性排除、S15 因子⑤、摘要共振排除
DT_PAIR_RATIO    = 0.70   # 分點×個股配對 T+1 平均出清 ≥ 此值：判定為「隔日沖」（S05 類型與警示、S06 分類）
CONC_TIER_HIGH   = 0.30   # S07 等級：5 日集中度 ≥ 30% 高度集中
CONC_TIER_MID    = 0.15   # S07 等級：≥ 15% 中度集中；摘要「集中度」共振門檻
CONC_SCORE_FULL  = 0.15   # S13 Score：5 日集中度達此值給滿分 12
TC_ROUNDTRIP     = 0.0045 # 回測交易成本（來回）：手續費 0.1425%×2 打折後 ≈0.15% ＋ 證交稅 0.3%
HOLD_DAYS        = 5      # 回測組合持有天數
TOP_N_PORTFOLIO  = 5      # 回測組合每次持有檔數

# ============ V1.1 優化參數（對應《優化建議》S01～S15） ============
# S01 EMA 加權累積與族群建倉
EMA_SPANS        = (5, 20)        # EMA 加權累積買超的跨度（近期權重較高）
SECTOR_MIN_STOCKS = 3             # 同產業至少幾檔同時「有效買超」才視為族群建倉
# S02 動態流動性門檻（取代固定 0.2%）：依 60 日均量在股票池中的百分位，線性內插有效買超比例
MAT_RATIO_LIQUID = 0.001          # 流動性最高（大型權值股）：淨買超 ≥ 同期成交量 0.1%
MAT_RATIO_ILLIQ  = 0.004          # 流動性最低（小型股）：淨買超 ≥ 同期成交量 0.4%
MAT_MIN_LOTS     = 20             # 有效買超最低張數（避免極低量股 1、2 張就成立）
# S03 庫存扣減法與時間衰減
COST_METHOD      = 'AVG'          # 'AVG' = 按比例減碼（原法）、'FIFO' = 先買先賣、'LIFO' = 後買先賣
SUPPORT_HALFLIFE = COST_HALFLIFE  # V1.4：與因子引擎同一半衰期（S03 顯示的成本 ≈ Score 使用的成本邏輯）
# S04 核密度估計
KDE_GRID         = 200            # KDE 價格網格點數
KDE_BW_ADJ       = 1.0            # 頻寬倍數（>1 更平滑、<1 更細）
# S05 多日出清特徵
DT_HORIZON       = 3              # 「短沖」判定看 T+1～T+N 日內出清比例
DT_CLEAR_RATIO   = 0.80           # N 日內累積賣出 ≥ 買進的 80% 視為該天期短沖
# S06 機器分群
N_CLUSTERS       = 6              # K-Means 群數
# S07 實質留倉成交量（扣除當沖量）
DAYTRADE_YEARS   = LOOKBACK_YEARS # V1.3：與分點回看期一致，避免集中度序列斷層（每日 1 次請求；0 = 不抓）
# S08 全市場買賣家數差（富邦個股主力進出 zco 頁）
BREADTH_STOCKS   = 80             # 對追蹤分點最活躍的前 N 檔抓全市場分點家數（0 = 關閉，只用 5 分點）
BREADTH_DAYS     = 20             # 抓最近幾個交易日
# S09 隱性吸籌（無五檔歷史時以 K 線下影線/賣壓日不跌作代理）
ABSORB_SHADOW    = 0.35           # 10 日平均下影線比例門檻
# S10 散戶接盤
MARGIN_SURGE     = 0.05           # 融資 5 日增幅 ≥ 5% 視為融資暴增
ODD_LOT_DAYS     = 60             # TWSE 零股成交量抓取天數（0 = 不抓）
# S11 防守線
REV_BREAK_MODE   = 'ATR_ZONE'     # 'MA10'（原法）、'ATR'、'ZONE'、'ATR_ZONE'（ATR 移動停損 或 成本密集區下緣，任一跌破）
ATR_N, ATR_MULT  = 14, 3.0        # Chandelier 停損：近 22 日最高價 − 3 × ATR14
# S12 PCA
PCA_COMPONENTS   = 3
PCA_FIT_DAYS     = 120            # 以最近 N 日的橫斷面標準化資料堆疊擬合 PCA
# S13 機器學習版 Score（walk-forward，無前視）
ML_ENABLE        = True
ML_HORIZON       = 10             # 標籤：未來 10 日同日超額報酬 > 0
ML_RETRAIN       = 40             # 每 40 個交易日重新訓練一次（擴張視窗）
ML_MIN_TRAIN     = 120            # 至少累積 120 個交易日樣本才開始預測
# S14 Z-Score 軟性評分
SOFT_Z_PASS      = 0.35           # 平均 Z 分數 ≥ 0.35 即入選（另保留「隔日沖」為硬性排除）
# S15 滑價模型
CAPITAL_LEVELS   = (1_000_000, 10_000_000, 50_000_000)   # 回測投入資金（元），各算一條淨值
IMPACT_K         = 0.8            # 市場衝擊係數：衝擊 ≈ K × 日波動 × √(下單張數 ÷ 20 日均量)
MAX_PARTICIPATION = 0.10          # 單檔下單超過 20 日均量 10% 時標示流動性不足

# ============ V1.4 新增參數 ============
TPEX_CHIPS       = True           # 上櫃股補抓 TPEx 外資/投信/融資（與 TWSE 並行）；False = 上櫃維持中性分
FDR_Q            = 0.10           # S15 多重檢定：Benjamini-Hochberg 偽發現率門檻
HOLDOUT_DAYS     = 120            # S15 保留期：最後 N 個交易日只用來驗證（調參時請勿看這段結果）
SENS_HOLD        = (5, 10, 20)    # S15 參數敏感度：持有天數
SENS_TOPN        = (3, 5, 10)     # S15 參數敏感度：持股檔數
LIMIT_TH         = 0.095          # 漲跌停判定：原始收盤漲跌幅 ≥ 9.5% 視為漲停（買不到）／跌停（賣不掉）
RUN_SELF_TESTS   = True           # 執行最後的自我測試格
# ============ V1.5 新增參數 ============
OFFLINE_HTML     = False          # True = 把 Chart.js 內嵌進每頁（每頁 +約 200KB），離線／手機直接開也能看圖
TRUNC_ROWS       = 50             # 富邦分點頁每日買超／賣超各列前 50 名：某側列滿即視為截尾觸頂
# ============ V1.6 排程模式（GitHub Actions）：以環境變數覆寫，Colab 手動執行不受影響 ============
RUN_MODE         = os.environ.get('BROKER_RUN_MODE', 'colab')   # 'ci' = 無人值守排程
if RUN_MODE == 'ci':
    SAVE_TO_DRIVE = False                                       # 排程機器沒有雲端硬碟；快取由 Actions cache 保存
    USE_DEMO_DATA = os.environ.get('BROKER_DEMO', '0') == '1'   # 手動觸發時可選模擬資料測流程
    OFFLINE_HTML  = os.environ.get('BROKER_OFFLINE_HTML', '0') == '1'

def check_params():
    """V1.4：參數合理性檢查，錯誤設定直接停止，避免產出看似正常但無意義的報表。"""
    errs = []
    ratio = dict(DT_ALERT_RATIO=DT_ALERT_RATIO, DT_CLEAR_RATIO=DT_CLEAR_RATIO, DT_FLAG_RATIO=DT_FLAG_RATIO, DT_PAIR_RATIO=DT_PAIR_RATIO,
                 CONC_MID=CONC_MID, CONC_HIGH=CONC_HIGH, CONC_TIER_MID=CONC_TIER_MID, CONC_TIER_HIGH=CONC_TIER_HIGH, CONC_SCORE_FULL=CONC_SCORE_FULL,
                 MAT_RATIO_LIQUID=MAT_RATIO_LIQUID, MAT_RATIO_ILLIQ=MAT_RATIO_ILLIQ, FDR_Q=FDR_Q, LIMIT_TH=LIMIT_TH, MAX_PARTICIPATION=MAX_PARTICIPATION)
    errs += [f'{k}={v} 應介於 0～1' for k, v in ratio.items() if not (0 < v <= 1)]
    if not CONC_MID < CONC_TIER_MID < CONC_TIER_HIGH: errs.append('需 CONC_MID < CONC_TIER_MID < CONC_TIER_HIGH')
    if MAT_RATIO_LIQUID > MAT_RATIO_ILLIQ: errs.append('MAT_RATIO_LIQUID 應 ≤ MAT_RATIO_ILLIQ（大型股門檻較低）')
    for k in ('COST_HALFLIFE', 'SUPPORT_HALFLIFE', 'HOLD_DAYS', 'TOP_N_PORTFOLIO', 'ML_HORIZON', 'ML_RETRAIN', 'N_CLUSTERS', 'COST_WINDOW', 'COST_LONG_WINDOW'):
        if not (isinstance(globals()[k], (int, float)) and globals()[k] > 0): errs.append(f'{k} 必須 > 0')
    if DAYTRADE_YEARS and DAYTRADE_YEARS < LOOKBACK_YEARS: errs.append(f'DAYTRADE_YEARS({DAYTRADE_YEARS}) < LOOKBACK_YEARS({LOOKBACK_YEARS})：集中度序列會有斷層（若刻意如此可忽略）')
    if HOLDOUT_DAYS < 0 or HOLDOUT_DAYS > 245 * LOOKBACK_YEARS / 2: errs.append('HOLDOUT_DAYS 應介於 0 ～ 總期間一半')
    if REV_BREAK_MODE not in ('MA10', 'ATR', 'ZONE', 'ATR_ZONE'): errs.append('REV_BREAK_MODE 只能是 MA10／ATR／ZONE／ATR_ZONE')
    if COST_METHOD not in ('AVG', 'FIFO', 'LIFO'): errs.append('COST_METHOD 只能是 AVG／FIFO／LIFO')
    hard = [e for e in errs if '可忽略' not in e]
    for e in errs: print('⚠️ 參數：' + e)
    if hard: raise ValueError('參數設定錯誤，請修正後重跑：' + '；'.join(hard))
check_params()

# 參數簽章：因子引擎執行時記錄；之後若改了參數只重跑單一策略格，save_page 會警告
_PARAM_NAMES = [k for k, v in list(globals().items()) if k.isupper() and not k.startswith('_') and isinstance(v, (int, float, str, bool, tuple, list, dict))]
def _param_sig():
    import hashlib
    return hashlib.md5(repr([(k, globals().get(k)) for k in _PARAM_NAMES]).encode()).hexdigest()

TPE = dt.timezone(dt.timedelta(hours=8))
OUTPUT_DIR = os.environ.get('BROKER_OUTPUT_DIR') or ('/content/broker_dashboards' if os.path.exists('/content') else os.path.abspath('broker_dashboards'))
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        _local_cache = os.path.join(OUTPUT_DIR, '_cache')
        OUTPUT_DIR = '/content/drive/MyDrive/broker_dashboards'
        os.makedirs(os.path.join(OUTPUT_DIR, '_cache'), exist_ok=True)
        if os.path.isdir(_local_cache):   # 把先前存在 Colab 本機的快取搬進雲端硬碟，不必重抓
            n = 0
            for fn in os.listdir(_local_cache):
                dst = os.path.join(OUTPUT_DIR, '_cache', fn)
                if not os.path.exists(dst):
                    shutil.copy2(os.path.join(_local_cache, fn), dst); n += 1
            if n:
                print(f'已將本機快取 {n} 個檔案複製到雲端硬碟')
    except ImportError:
        print('非 Colab 環境，快取存在本機')
CACHE_DIR = os.path.join(OUTPUT_DIR, '_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
START_DATE = (pd.Timestamp(dt.datetime.now(TPE).date()) - pd.DateOffset(years=LOOKBACK_YEARS)).date()
INST_START = (pd.Timestamp(dt.datetime.now(TPE).date()) - pd.DateOffset(years=INST_YEARS)).date() if INST_YEARS else None
_today = pd.Timestamp(dt.datetime.now(TPE).date())
DAYTRADE_START = (_today - pd.DateOffset(years=DAYTRADE_YEARS)).date() if DAYTRADE_YEARS else None
print('輸出資料夾：', OUTPUT_DIR)

In [ ]:
# ============ 2. 交易日曆 ============
def get_trading_dates(start):
    """start 之後的台股交易日（以加權指數實際交易日為準）"""
    now = dt.datetime.now(TPE)
    dates = None
    try:
        import yfinance as yf
        h = yf.Ticker('^TWII').history(start=(pd.Timestamp(start) - pd.Timedelta(days=10)).strftime('%Y-%m-%d'), auto_adjust=False)
        if len(h):
            dates = [pd.Timestamp(d).tz_localize(None).normalize() if pd.Timestamp(d).tzinfo else pd.Timestamp(d).normalize() for d in h.index]
    except Exception as e:
        print('yfinance 取得交易日失敗，改用平日曆：', e)
    if not dates:
        raise RuntimeError('無法取得真實交易日曆：停止執行，不以平日假冒交易日。請重試或提供經驗證的交易日。')
    dates = sorted({d for d in dates if d.date() >= start})
    # 盤後分點資料約 17:00 後才完整，之前不納入今天
    if dates and dates[-1].date() == now.date() and now.hour < 17:
        dates = dates[:-1]
    return [d.date() for d in dates]

In [ ]:
# ============ 3. 富邦分點頁爬蟲（zgb0） ============
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36',
    'Referer': 'https://fubon-ebrokerdj.fbs.com.tw/',
})
import threading
_TL = threading.local()
def _sess():
    """V1.5：每個執行緒各自一個 requests.Session（Session 非執行緒安全），沿用 SESSION 的標頭"""
    s = getattr(_TL, 's', None)
    if s is None:
        s = requests.Session(); s.headers.update(SESSION.headers); _TL.s = s
    return s
RE_GEN  = re.compile(r"GenLink2stk\(\s*'(?:AS)?([0-9A-Za-z]+)'\s*,\s*'([^']*)'\s*\)")
RE_LINK = re.compile(r"Link2Stk\(\s*'([0-9A-Za-z]+)'\s*\)[^>]*>([^<]*)<")
RE_TR   = re.compile(r'<tr[^>]*>(.*?)</tr>', re.S | re.I)
RE_TD   = re.compile(r'<td[^>]*>(.*?)</td>', re.S | re.I)
RE_DATE = re.compile(r'資料日期[：:]\s*(\d{8})')
FLOW_COLS = ['code', 'name', 'buy', 'sell', 'net']

def _num(s):
    s = re.sub(r'<[^>]+>', '', s).replace(',', '').replace('&nbsp;', '').strip()
    try:
        return float(s)
    except ValueError:
        return None

def parse_zgb(html):
    """解析分點進出頁：回傳 (DataFrame[code,name,buy,sell,net], 資料日期 yyyymmdd)"""
    out = []
    for tr in RE_TR.findall(html):
        tds = RE_TD.findall(tr)
        i = 0
        while i < len(tds):
            cell, code, name = tds[i], None, ''
            m = RE_GEN.search(cell)
            if m:
                code, name = m.group(1).upper(), m.group(2)
            else:
                m2 = RE_LINK.search(cell)
                if m2:
                    code, name = m2.group(1).upper(), m2.group(2).strip()
                    if name.upper().startswith(code):
                        name = name[len(code):]
            if code and i + 3 < len(tds):
                b, s, n = _num(tds[i + 1]), _num(tds[i + 2]), _num(tds[i + 3])
                if None not in (b, s, n):
                    out.append((code, name.strip(), b, s, n))
                    i += 4
                    continue
            i += 1
    df = pd.DataFrame(out, columns=FLOW_COLS).drop_duplicates('code')
    text = re.sub(r'<[^>]+>', ' ', html)
    m = RE_DATE.search(text)
    hdr = re.search(r'買進\s*(張數|金額)', text)
    u = re.search(r'單位[：:]\s*(張|股|仟元|千元|元)', text)
    if hdr:
        unit = 'lots' if hdr.group(1) == '張數' else 'amt'
    elif u:
        unit = 'lots' if u.group(1) in ('張', '股') else 'amt'
    else:
        unit = None
    return df, (m.group(1) if m else None), unit

def fetch_branch_day(a, b, date, mode):
    """mode: 'B' = 張數, 'E' = 金額(仟元)。回傳 (df, 狀態, 頁面實際單位)"""
    path = os.path.join(CACHE_DIR, f'zgb_{a}_{b}_{date:%Y%m%d}_{mode}.csv')
    upath = path + '.unit'
    # V1.2: 舊 nodata 可能是暫時性錯頁，不當成永久不可查。
    if os.path.exists(path):
        meta = open(upath).read().split() if os.path.exists(upath) else []
        unit = meta[0] if meta else None
        cached = pd.read_csv(path, dtype={'code': str})
        if unit in ('lots', 'amt') and not cached.empty:
            return cached, 'cache', unit
        if unit in ('lots', 'amt') and 'empty' in meta[1:]:
            return pd.DataFrame(columns=FLOW_COLS), 'cache_empty', unit   # V1.4：已驗證的當日無交易頁
        # 舊缺單位或空快取不能認證，繼續重抓。
    d = f'{date.year}-{date.month}-{date.day}'
    url = f'{BASE_URL}?a={a}&b={b}&c={mode}&e={d}&f={d}'
    last_err, empty_hits = None, 0
    for attempt in range(3):
        try:
            r = _sess().get(url, timeout=25)
            r.raise_for_status()
            html = r.content.decode('cp950', errors='ignore')
            df, ddate, unit = parse_zgb(html)
            time.sleep(REQUEST_SLEEP)
            if not ddate or ddate != date.strftime('%Y%m%d'):
                return pd.DataFrame(columns=FLOW_COLS), f"date_mismatch:{ddate or 'missing'}", unit
            if unit in ('lots', 'amt') and df.empty:
                # V1.4：日期與單位都正確、但沒有任何個股 → 可能真的當日無交易；連續兩次一致才認定並快取
                empty_hits += 1
                if empty_hits >= 2:
                    if date < dt.datetime.now(TPE).date():
                        df.to_csv(path, index=False)
                        with open(upath, 'w') as f:
                            f.write(f'{unit} empty')
                    return df, 'empty', unit
                raise ValueError('空頁，再確認一次')
            if unit not in ('lots', 'amt'):
                raise ValueError('分點頁缺少可驗證單位；不快取為成功')
            if date < dt.datetime.now(TPE).date() or dt.datetime.now(TPE).hour >= 18:
                df.to_csv(path, index=False)
                with open(upath, 'w') as f:
                    f.write(unit or '')
            return df, 'ok', unit
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    return pd.DataFrame(columns=FLOW_COLS), f'error:{last_err}', None

def probe_earliest(dates):
    """二分搜尋富邦可查到的最早交易日（約 10 次請求），避免對查不到的舊日期發出上千次請求"""
    a, b = list(BRANCHES.values())[0]
    ok = lambda d: fetch_branch_day(a, b, d, 'E')[1] in ('ok', 'cache')
    if ok(dates[0]):
        return dates
    lo, hi = 0, len(dates) - 1
    if not ok(dates[hi]):
        print('⚠️ 連最近一日都查不到資料，請檢查網路或網址參數')
        return dates
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if ok(dates[mid]): hi = mid
        else: lo = mid
    print(f'⚠️ 富邦最早只提供到 {dates[hi]} 的單日分點資料，抓取區間自動縮短為 {len(dates) - hi} 個交易日')
    return dates[hi:]

def collect_flows(dates):
    tasks = [(bn, a, b, d, m) for bn, (a, b) in BRANCHES.items() for d in dates for m in ('B', 'E')]
    res, stat, units = {}, Counter(), {}
    print(f'抓取分點資料：{len(BRANCHES)} 分點 × {len(dates)} 日 × 2 模式 = {len(tasks)} 次（已快取的會跳過）')
    t_start = time.time()
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = {ex.submit(fetch_branch_day, a, b, d, m): (bn, d, m) for bn, a, b, d, m in tasks}
        for k, fu in enumerate(as_completed(futs), 1):  # noqa
            bn, d, m = futs[fu]
            df, st, unit = fu.result()
            # 以頁面實際標示的單位為準（不信任請求參數）
            if unit is not None:
                units[(bn, d, m)] = unit
                if (m == 'B' and unit == 'amt') or (m == 'E' and unit == 'lots'):
                    stat['參數單位對調'] += 1
            res[(bn, d, m)] = df
            stat[st.split(':')[0]] += 1
            if k % 250 == 0 or k == len(tasks):
                el = time.time() - t_start
                eta = el / k * (len(tasks) - k)
                print(f'  {k}/{len(tasks)}  {dict(stat)}  已用 {el / 60:.1f} 分，預估剩 {eta / 60:.1f} 分')
    default_unit = {'B': 'lots', 'E': 'amt'}
    n_sw = stat.get('參數單位對調', 0)
    if n_sw:
        print(f'ℹ️ {n_sw} 頁的實際單位與請求參數相反（富邦 c=B 為金額、c=E 為張數），已依頁面標示的單位自動對調，資料正確。')
        if n_sw > len(tasks) * 0.5:
            default_unit = {'B': 'amt', 'E': 'lots'}   # 舊快取沒有單位標記時也採用對調
    if stat.get('date_mismatch', 0) > len(tasks) * 0.5:
        print('⚠️ 多數日期回傳的資料日期與請求不符，網站可能更改了日期參數格式，請檢查 URL。')
    rows = []
    for bn in BRANCHES:
        for d in dates:
            # 依頁面「實際單位」分配，不依請求參數（富邦 c=B 實為金額、c=E 實為張數）
            pages = [(res.get((bn, d, m)), units.get((bn, d, m)) or default_unit[m]) for m in ('B', 'E')]
            pages = [(df_, u) for df_, u in pages if df_ is not None and not df_.empty]
            lb = next((df_ for df_, u in pages if u == 'lots'), None)
            la = next((df_ for df_, u in pages if u == 'amt'), None)
            if lb is None and la is None:
                continue
            lb = (lb if lb is not None else pd.DataFrame(columns=FLOW_COLS)).rename(columns={'buy': 'buy_lots', 'sell': 'sell_lots', 'net': 'net_lots'})
            la = (la if la is not None else pd.DataFrame(columns=FLOW_COLS)).rename(columns={'buy': 'buy_amt', 'sell': 'sell_amt', 'net': 'net_amt', 'name': 'name2'})
            mg = pd.merge(lb, la, on='code', how='outer')
            mg['name'] = mg['name'].fillna(mg['name2'])
            mg['date'] = pd.Timestamp(d)
            mg['branch'] = bn
            rows.append(mg.drop(columns=['name2']))
    if not rows:
        raise RuntimeError('沒有抓到任何分點資料。請確認網路、URL 參數，或先設 USE_DEMO_DATA=True 測試版面。')
    flow = pd.concat(rows, ignore_index=True)
    for c in ['buy_lots', 'sell_lots', 'net_lots', 'buy_amt', 'sell_amt', 'net_amt']:
        flow[c] = pd.to_numeric(flow[c], errors='coerce')
    flow['code'] = flow['code'].astype(str)
    print(f'分點資料完成：{len(flow):,} 筆，{flow.code.nunique()} 檔，{flow.date.nunique()} 個交易日')
    # 檢查網站是否忽略日期參數（每天資料都一樣）
    same, tot = 0, 0
    for bn, g in flow.groupby('branch'):
        sig = g.groupby('date').apply(lambda x: hash(tuple(sorted(zip(x['code'], x['net_lots'].fillna(x['net_amt']))))))
        same += int((sig == sig.shift(1)).sum()); tot += max(len(sig) - 1, 0)
    if tot and same / tot > 0.3:
        print(f'🚨 有 {same}/{tot} 個交易日的資料與前一日完全相同，網站可能忽略了日期參數。請把這段輸出貼給我。')
    else:
        print(f'  日期檢查通過：相鄰交易日資料相同的比例 {same / tot if tot else 0:.1%}')
    return flow

In [ ]:
# ============ 4. 股價（yfinance）與 TWSE 法人/融資 ============
def _naive(idx):
    idx = pd.to_datetime(idx)
    if getattr(idx, 'tz', None) is not None:
        idx = idx.tz_localize(None)
    return idx.normalize()

def get_twse_listed():
    """上市代號清單（TWSE OpenAPI）；不在清單內者視為上櫃。失敗回傳 None。"""
    path = os.path.join(CACHE_DIR, f'twse_listed_{dt.date.today():%Y%m%d}.json')
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            return set(json.load(f))
    try:
        j = _sess().get('https://openapi.twse.com.tw/v1/exchangeReport/STOCK_DAY_ALL', timeout=30).json()
        codes = sorted({x['Code'].strip() for x in j if x.get('Code')})
        if len(codes) > 500:
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(codes, f)
            return set(codes)
    except Exception as e:
        print('  取得上市清單失敗，改用 .TW → .TWO 逐一嘗試：', e)
    return None

def fetch_prices(codes, start):
    """yfinance 下載開高低收、還原收盤價（含除權息）與成交量（張）。
    V1.4 增量快取：同一份快取每天只補最近 PRICE_OVERLAP 個日曆日；重疊期間還原價或收盤若被改寫（除權息調整）
    → 該檔整段重抓，避免新舊還原尺度混用。新出現的代號也整段抓。"""
    import logging, yfinance as yf
    logging.getLogger('yfinance').setLevel(logging.CRITICAL)
    codes = sorted(c for c in codes if re.fullmatch(r'[1-9]\d{3}|00\d{2,4}[A-Z]?', c) and not c.endswith('B'))
    FIELDS = ('Close', 'Volume', 'Adj Close', 'High', 'Low', 'Open')
    PRICE_OVERLAP = 20
    path = os.path.join(CACHE_DIR, 'prices_v5.pkl')
    start_ts, today = pd.Timestamp(start), dt.date.today()
    store = None
    if os.path.exists(path):
        try:
            store = pd.read_pickle(path)
            if store['F']['Close'].index.min() > start_ts + pd.Timedelta(days=10):
                print('  股價快取起始日晚於需求，整段重抓'); store = None
        except Exception as e:
            print('  股價快取無法讀取，整段重抓：', e); store = None
    def out_of(st):
        F = st['F']
        cols = [c for c in codes if c in F['Close'].columns]
        res = []
        for k in FIELDS:
            X = F[k].reindex(columns=cols)
            X = X[X.index >= start_ts]
            res.append(X / 1000.0 if k == 'Volume' else X)
        return tuple(res)
    if store and store.get('asof') == today and set(codes) <= set(store['F']['Close'].columns) | set(store.get('missing', ())):
        print(f"股價使用今日快取：{store['F']['Close'].shape[1]} 檔")
        return out_of(store)

    listed = get_twse_listed()
    sfx = dict(store.get('sfx', {})) if store else {}

    def grab(tickers, label, st):
        """回傳 {代號: DataFrame(FIELDS)}；被限流的批次自動退避重試"""
        got, todo, rnd = {}, list(tickers), 0
        while todo and rnd < 5:
            if rnd:
                wait = 20 * rnd
                print(f'    {label} 第 {rnd + 1} 輪重試 {len(todo)} 檔（等待 {wait} 秒避開限流）')
                time.sleep(wait)
            for i in range(0, len(todo), 40):
                chunk = todo[i:i + 40]
                try:
                    data = yf.download(chunk, start=st, auto_adjust=False, progress=False, group_by='ticker', threads=False)
                except Exception:
                    continue
                for t in chunk:
                    try:
                        sub = data[t] if isinstance(data.columns, pd.MultiIndex) else data
                        sub = sub.dropna(subset=['Close'])
                    except Exception:
                        continue
                    if len(sub):
                        s = sub.copy(); s.index = _naive(s.index)
                        if 'Adj Close' not in s:
                            s['Adj Close'] = s['Close']
                        c = t.split('.')[0]
                        got[c] = s[list(FIELDS)]; sfx[c] = '.' + t.split('.')[1]
                time.sleep(1.5)
            todo = [t for t in todo if t.split('.')[0] not in got]
            rnd += 1
        return got

    def tick_of(c):
        if c in sfx: return c + sfx[c]
        if listed is not None: return c + ('.TW' if c in listed else '.TWO')
        return c + '.TW'

    def grab_full(cs, label):
        got = grab([tick_of(c) for c in cs], label, start)
        miss = [c for c in cs if c not in got]
        if miss:   # 清單可能與實際掛牌不同步（剛轉板等），換後綴再試一次
            for c in miss: sfx.pop(c, None)
            alt = [c + ('.TWO' if tick_of(c).endswith('.TW') else '.TW') for c in miss]
            got.update(grab(alt, '換後綴', start))
        return got

    old = {} if store is None else store['F']
    known = [c for c in codes if store is not None and c in old['Close'].columns and old['Close'][c].notna().any()]
    new_codes = [c for c in codes if c not in known]
    frames = {}
    if known:
        last = old['Close'].index.max()
        inc_start = (last - pd.Timedelta(days=PRICE_OVERLAP)).strftime('%Y-%m-%d')
        print(f'  股價增量更新：{len(known)} 檔自 {inc_start} 起；新代號 {len(new_codes)} 檔整段下載')
        inc = grab([tick_of(c) for c in known], '增量', inc_start)
        refetch = []
        for c in known:
            s = inc.get(c)
            o = pd.DataFrame({k: old[k][c] for k in FIELDS}).dropna(subset=['Close'])
            if s is None:                           # 這幾天抓不到（停牌或限流）→ 沿用舊資料
                frames[c] = o; continue
            ov = o.index.intersection(s.index)
            if len(ov) == 0:
                refetch.append(c); continue
            dev = max(float((s.loc[ov, 'Adj Close'] / o.loc[ov, 'Adj Close'] - 1).abs().max()),
                      float((s.loc[ov, 'Close'] / o.loc[ov, 'Close'] - 1).abs().max()))
            if not np.isfinite(dev) or dev > 5e-4:  # 還原價被改寫 → 除權息，整段重抓
                refetch.append(c); continue
            frames[c] = pd.concat([o[o.index < s.index.min()], s]).sort_index()
        if refetch:
            print(f'  {len(refetch)} 檔偵測到除權息/價格改寫，整段重抓')
        new_codes += refetch
    else:
        print(f'  下載股價（整段）：{len(new_codes)} 檔')
    if new_codes:
        frames.update(grab_full(new_codes, '整段'))
    F = {k: pd.DataFrame({c: f[k] for c, f in frames.items()}).sort_index() for k in FIELDS}
    missing = [c for c in codes if c not in frames]
    stocks_missing = [c for c in missing if re.fullmatch(r'[1-9]\d{3}', c)]
    print(f'股價完成：{F["Close"].shape[1]} / {len(codes)} 檔。缺普通股 {len(stocks_missing)} 檔'
          + (f'：{", ".join(stocks_missing[:30])}{" …" if len(stocks_missing) > 30 else ""}' if stocks_missing else ''))
    st = dict(F=F, sfx=sfx, asof=today, missing=missing)
    if len(stocks_missing) <= max(10, 0.02 * len(codes)):
        pd.to_pickle(st, path)
    else:
        print('  ⚠️ 缺漏偏多（可能仍在限流），本次不寫入快取；稍候重跑此格即可補齊。')
    return out_of(st)

def _twse_json(url, cache_name):
    path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    for attempt in range(3):
        try:
            r = _sess().get(url, timeout=25)
            j = r.json()
            time.sleep(2.0)
            if j.get('stat', '').upper() == 'OK':
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(j, f, ensure_ascii=False)
            return j
        except Exception:
            time.sleep(10 * (attempt + 1))   # TWSE 限流時回傳 HTML，退避久一點
    return {}

def _toint(x):
    try:
        return float(str(x).replace(',', '').strip())
    except ValueError:
        return np.nan

def fetch_twse_chips(dates):
    """外資、投信買賣超（張）與融資餘額（張），上市股票。上櫃另由 fetch_tpex_chips 補（V1.4）。"""
    FOR, TRU, MAR = {}, {}, {}
    print(f'抓取 TWSE 法人與融資：{len(dates)} 日（已快取的會跳過）')
    t0_ = time.time()
    for k_, d in enumerate(dates, 1):
        if k_ % 50 == 0:
            print(f'  {k_}/{len(dates)}  已用 {(time.time() - t0_) / 60:.1f} 分')
        ds = d.strftime('%Y%m%d')
        j = _twse_json(f'https://www.twse.com.tw/rwd/zh/fund/T86?date={ds}&selectType=ALLBUT0999&response=json', f't86_{ds}.json')
        if j.get('data'):
            f = j['fields']
            ic = f.index('證券代號')
            ifo = next((i for i, x in enumerate(f) if '外' in x and '買賣超' in x), None)
            itr = next((i for i, x in enumerate(f) if '投信買賣超' in x), None)
            FOR[pd.Timestamp(d)] = {r[ic].strip(): _toint(r[ifo]) / 1000 for r in j['data']} if ifo is not None else {}
            TRU[pd.Timestamp(d)] = {r[ic].strip(): _toint(r[itr]) / 1000 for r in j['data']} if itr is not None else {}
        j = _twse_json(f'https://www.twse.com.tw/rwd/zh/marginTrading/MI_MARGN?date={ds}&selectType=ALL&response=json', f'margin_{ds}.json')
        tables = j.get('tables') or ([{'fields': j.get('fields', []), 'data': j.get('data', [])}] if j.get('data') else [])
        for t in tables:
            f = t.get('fields') or []
            if ('代號' in f or '股票代號' in f) and len(t.get('data') or []) > 100 and '今日餘額' in f:
                ic = f.index('代號') if '代號' in f else f.index('股票代號')
                im = f.index('今日餘額')
                MAR[pd.Timestamp(d)] = {r[ic].strip(): _toint(r[im]) for r in t['data']}
                break
    mk = lambda D: pd.DataFrame(D).T.sort_index() if D else pd.DataFrame()
    F_, T_, M_ = mk(FOR), mk(TRU), mk(MAR)
    print(f'法人資料 {len(F_)} 日、融資資料 {len(M_)} 日')
    return F_, T_, M_

# ---------- V1.4：TPEx 上櫃 外資／投信／融資 ----------
TPEX_BASE = 'https://www.tpex.org.tw/www/zh-tw'
TPEX_HEADERS = {'Referer': 'https://www.tpex.org.tw/zh-tw/index.html'}

def _tpex_json(path_, params, cache_name):
    """TPEx 新版端點（/www/zh-tw/...，回傳 tables[{fields,data}]）；有資料才快取"""
    fp = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(fp):
        with open(fp, encoding='utf-8') as f:
            return json.load(f)
    for attempt in range(3):
        try:
            r = _sess().get(f'{TPEX_BASE}/{path_}', params={**params, 'response': 'json'}, headers=TPEX_HEADERS, timeout=25)
            j = r.json()
            time.sleep(2.0)
            if any(t.get('data') for t in (j.get('tables') or [])):
                with open(fp, 'w', encoding='utf-8') as f:
                    json.dump(j, f, ensure_ascii=False)
            return j
        except Exception:
            time.sleep(8 * (attempt + 1))
    return None

def _tpex_table(j):
    for t in (j or {}).get('tables') or []:
        if t.get('data'):
            return [str(x).strip() for x in t.get('fields') or []], t['data']
    return [], []

# insti/dailyTrade 的 24 欄只能依位置讀（JSON 不帶群組表頭）：
#   2-4 外資及陸資(不含外資自營商) 5-7 外資自營商 8-10 外資及陸資合計 11-13 投信 14-16/17-19/20-22 自營商 23 三大法人合計
# 外資取第 4 欄（不含外資自營商），與 TWSE T86「外陸資買賣超(不含外資自營商)」口徑一致；每列以三大法人合計做加總檢查
TPEX_COL_FOR, TPEX_COL_FOR_ALL, TPEX_COL_TRU, TPEX_COL_DEA, TPEX_COL_TOT = 4, 10, 13, 22, 23

def fetch_tpex_chips(dates):
    """上櫃外資、投信買賣超（張）與融資餘額（張）。連續 3 日完全取不到 → 視為此網路無法連線 TPEx，停止並退回中性分。"""
    FOR, TRU, MAR = {}, {}, {}
    if not dates:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    fails, bad_rows = 0, 0
    print(f'抓取 TPEx 上櫃法人與融資：{len(dates)} 日（已快取的會跳過）')
    for k_, d in enumerate(dates, 1):
        if k_ % 100 == 0:
            print(f'  TPEx {k_}/{len(dates)}')
        ds, dq = d.strftime('%Y%m%d'), d.strftime('%Y/%m/%d')
        ji = _tpex_json('insti/dailyTrade', {'type': 'Daily', 'sect': 'AL', 'date': dq}, f'tpex_inst_{ds}.json')
        jm = _tpex_json('margin/balance', {'date': dq}, f'tpex_margin_{ds}.json')
        fi, di = _tpex_table(ji)
        if len(fi) == 24 and fi[23].startswith('三大法人'):
            fo, tr = {}, {}
            for r in di:
                v = [_toint(r[i]) for i in (TPEX_COL_FOR_ALL, TPEX_COL_TRU, TPEX_COL_DEA, TPEX_COL_TOT)]
                if all(np.isfinite(v)) and abs(v[0] + v[1] + v[2] - v[3]) > 1:
                    bad_rows += 1; continue           # 欄位位置與預期不符的列不採用
                code = str(r[0]).strip()
                fo[code] = _toint(r[TPEX_COL_FOR]) / 1000; tr[code] = _toint(r[TPEX_COL_TRU]) / 1000
            FOR[pd.Timestamp(d)], TRU[pd.Timestamp(d)] = fo, tr
        fm, dm = _tpex_table(jm)
        if '資餘額' in fm:
            im = fm.index('資餘額')
            MAR[pd.Timestamp(d)] = {str(r[0]).strip(): _toint(r[im]) for r in dm}
        fails = 0 if (ji or jm) else fails + 1
        if fails >= 3:
            print('⚠️ TPEx 連續 3 日無回應（部分網路會被 TPEx 擋），停止抓取；上櫃股外資/融資維持中性分。')
            break
    if bad_rows:
        print(f'  ⚠️ TPEx 法人有 {bad_rows} 列加總檢查不符，已略過（若比例很高，代表 TPEx 改了欄位順序）')
    mk = lambda D: pd.DataFrame(D).T.sort_index() if D else pd.DataFrame()
    F_, T_, M_ = mk(FOR), mk(TRU), mk(MAR)
    print(f'TPEx 法人資料 {len(F_)} 日（{F_.shape[1]} 檔）、融資資料 {len(M_)} 日')
    return F_, T_, M_

def merge_chips(a, b):
    """TWSE 與 TPEx 代號不重疊；同代號以 TWSE 為準"""
    if not len(b): return a
    if not len(a): return b
    return a.combine_first(b)

# ---------- V1.1 新增資料源 ----------
def _twse_tables(j):
    """TWSE rwd JSON 可能是 {fields,data} 或 {tables:[{fields,data}]}，統一成 list"""
    t = list(j.get('tables') or [])
    if j.get('data'):
        t.append({'fields': j.get('fields', []), 'data': j.get('data', [])})
    return [x for x in t if x.get('fields') and x.get('data')]

def fetch_industry():
    """S01 族群建倉：上市/上櫃產業別（TWSE ISIN 公開頁，big5）。失敗回傳 {}，族群分析標為「未分類」。"""
    path = os.path.join(CACHE_DIR, f'industry_{dt.date.today():%Y%m}.json')
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    out = {}
    for mode in (2, 4):   # 2 = 上市、4 = 上櫃
        try:
            r = _sess().get(f'https://isin.twse.com.tw/isin/C_public.jsp?strMode={mode}', timeout=40)
            html = r.content.decode('cp950', errors='ignore')
            for tr in RE_TR.findall(html):
                tds = [re.sub(r'<[^>]+>', '', x).replace('&nbsp;', ' ').strip() for x in RE_TD.findall(tr)]
                if len(tds) >= 5:
                    m = re.match(r'^([0-9A-Z]{4,6})[\s　]+', tds[0])
                    if m and tds[4]:
                        out[m.group(1)] = tds[4]
            time.sleep(1.0)
        except Exception as e:
            print('  產業別抓取失敗：', e)
    if len(out) > 500:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(out, f, ensure_ascii=False)
    print(f'產業別：{len(out)} 檔')
    return out

def fetch_daytrade_vol(dates):
    """S07 實質留倉量：TWSE 當日沖銷交易標的成交股數（張）。只有上市股票；上櫃與未抓日期之後以估計值補。"""
    DTV = {}
    if not dates:
        return pd.DataFrame()
    print(f'抓取 TWSE 當沖成交量：{len(dates)} 日（已快取的會跳過）')
    for k_, d in enumerate(dates, 1):
        if k_ % 50 == 0:
            print(f'  {k_}/{len(dates)}')
        ds = d.strftime('%Y%m%d')
        j = _twse_json(f'https://www.twse.com.tw/rwd/zh/dayTrading/TWTB4U?date={ds}&selectType=All&response=json', f'daytrade_{ds}.json')
        for t in _twse_tables(j):
            f = t['fields']
            if '證券代號' in f and len(t['data']) > 50:
                iq = next((i for i, x in enumerate(f) if '成交股數' in x), None)
                if iq is None:
                    continue
                ic = f.index('證券代號')
                DTV[pd.Timestamp(d)] = {r[ic].strip(): _toint(r[iq]) / 1000 for r in t['data']}
                break
    out = pd.DataFrame(DTV).T.sort_index() if DTV else pd.DataFrame()
    print(f'當沖量資料 {len(out)} 日')
    return out

ODD_LOT_URL = 'https://www.twse.com.tw/rwd/zh/afterTrading/TWT53U?date={ds}&selectType=ALL&response=json'   # 零股交易行情單（若 TWSE 改版請更新）
def fetch_odd_lot(dates):
    """S10 散戶接盤：零股成交股數。端點若改版會自動回傳空表，條件標 N/A，不影響其他計算。"""
    ODD = {}
    for d in dates:
        ds = d.strftime('%Y%m%d')
        j = _twse_json(ODD_LOT_URL.format(ds=ds), f'oddlot_{ds}.json')
        for t in _twse_tables(j):
            f = t['fields']
            if '證券代號' in f:
                iq = next((i for i, x in enumerate(f) if '成交股數' in x), None)
                if iq is not None:
                    ic = f.index('證券代號')
                    ODD[pd.Timestamp(d)] = {r[ic].strip(): _toint(r[iq]) for r in t['data']}
                    break
    out = pd.DataFrame(ODD).T.sort_index() if ODD else pd.DataFrame()
    print(f'零股資料 {len(out)} 日' + ('（端點無回應或格式不同，S10 零股條件將顯示 N/A）' if not len(out) and dates else ''))
    return out

ZCO_URL = 'https://fubon-ebrokerdj.fbs.com.tw/z/zc/zco/zco.djhtm'
def parse_zco(html):
    """富邦個股主力進出頁：買進家數、賣出家數、前 15 大買超/賣超合計（張）"""
    text = re.sub(r'<[^>]+>', ' ', html).replace('&nbsp;', ' ')
    text = re.sub(r'\s+', ' ', text)
    g = lambda pat: (lambda m: _toint(m.group(1)) if m else np.nan)(re.search(pat, text))
    res = dict(nbuy=g(r'買進家數\s*([\d,]+)'), nsell=g(r'賣出家數\s*([\d,]+)'),
               top_buy=g(r'合計買超張數\s*([\-\d,\.]+)'), top_sell=g(r'合計賣超張數\s*([\-\d,\.]+)'))
    return res

def fetch_market_breadth(codes, dates):
    """S08：全市場（所有分點）買進家數／賣出家數，取代只用 5 個追蹤分點的家數差"""
    rows, bad = [], 0
    tasks = [(c, d) for c in codes for d in dates]
    print(f'抓取全市場分點家數：{len(codes)} 檔 × {len(dates)} 日 = {len(tasks)} 頁（已快取的會跳過）')
    def one(c, d):
        path = os.path.join(CACHE_DIR, f'zco_{c}_{d:%Y%m%d}.json')
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        ds = f'{d.year}-{d.month}-{d.day}'
        for attempt in range(3):
            try:
                r = _sess().get(f'{ZCO_URL}?a={c}&e={ds}&f={ds}', timeout=25)
                res = parse_zco(r.content.decode('cp950', errors='ignore'))
                time.sleep(REQUEST_SLEEP)
                res = {k: (None if not np.isfinite(v) else v) for k, v in res.items()}
                if res.get('nbuy') is not None and d < dt.datetime.now(TPE).date():
                    with open(path, 'w') as f:
                        json.dump(res, f)
                return res
            except Exception:
                time.sleep(1.5 * (attempt + 1))
        return {}
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = {ex.submit(one, c, d): (c, d) for c, d in tasks}
        for k, fu in enumerate(as_completed(futs), 1):
            c, d = futs[fu]
            r = fu.result() or {}
            if r.get('nbuy') is None:
                bad += 1
            rows.append(dict(code=c, date=pd.Timestamp(d), **{k_: r.get(k_) for k_ in ('nbuy', 'nsell', 'top_buy', 'top_sell')}))
            if k % 400 == 0:
                print(f'  {k}/{len(tasks)}')
    df = pd.DataFrame(rows)
    if tasks and bad > 0.5 * len(tasks):
        print(f'⚠️ 全市場家數有 {bad}/{len(tasks)} 頁解析不到「買進家數／賣出家數」，網頁格式可能不同；S08 將退回只用 5 個追蹤分點。請把這段輸出貼給我。')
        return pd.DataFrame()
    print(f'全市場家數完成：{len(tasks) - bad}/{len(tasks)} 頁成功')
    return df


In [ ]:
# ============ 5. 模擬資料（僅 USE_DEMO_DATA=True 時使用） ============
def make_demo_data(n_days):
    rng = np.random.default_rng(7)
    all_days = list(pd.bdate_range(end=pd.Timestamp.today().normalize() - pd.Timedelta(days=1), periods=n_days + 60))
    dates = all_days[-n_days:]
    nm = {'2330': '台積電', '2317': '鴻海', '2454': '聯發科', '2308': '台達電', '2382': '廣達', '2303': '聯電',
          '3231': '緯創', '2603': '長榮', '3017': '奇鋐', '6669': '緯穎', '2357': '華碩', '3661': '世芯-KY',
          '2379': '瑞昱', '3008': '大立光', '2345': '智邦', '6488': '環球晶', '3529': '力旺', '5274': '信驊',
          '2449': '京元電子', '3443': '創意', '8046': '南電', '3037': '欣興', '2368': '金像電', '6223': '旺矽',
          '4966': '譜瑞-KY', '3324': '雙鴻', '2059': '川湖', '1513': '中興電', '1519': '華城', '6515': '穎崴'}
    codes = list(nm)
    close, vol = {}, {}
    for c in codes:
        p0 = rng.uniform(40, 900); drift = rng.normal(0.0005, 0.001)
        r = rng.normal(drift, 0.022, len(all_days))
        close[c] = pd.Series(p0 * np.exp(np.cumsum(r)), index=all_days)
        vol[c] = pd.Series(rng.uniform(2000, 40000) * rng.lognormal(0, 0.35, len(all_days)), index=all_days)
    C, V = pd.DataFrame(close), pd.DataFrame(vol)
    styles = ['swing', 'random', 'daytrade', 'trend', 'churn']
    rows = []
    for bi, bn in enumerate(BRANCHES):
        style = styles[bi % 5]
        focus = rng.choice(codes, 10, replace=False)
        pending = {}
        for ti, d in enumerate(dates):
            for c in focus:
                p = C.loc[d, c]
                if style == 'daytrade':
                    buy = float(rng.integers(150, 1500)) if rng.random() < 0.3 else 0.0
                    sell = pending.pop(c, 0) * rng.uniform(0.85, 1.0)
                    if buy:
                        pending[c] = buy
                elif style in ('swing', 'trend'):
                    phase = (ti // 15) % 2 == 0 if style == 'swing' else True
                    buy = float(rng.integers(50, 600)) if rng.random() < 0.7 else 0.0
                    sell = float(rng.integers(0, 200))
                    if not phase:
                        buy, sell = sell, buy
                elif style == 'churn':
                    buy = float(rng.integers(100, 800)); sell = buy * rng.uniform(0.8, 1.2)
                else:
                    buy = float(rng.integers(0, 400)) if rng.random() < 0.5 else 0.0
                    sell = float(rng.integers(0, 400)) if rng.random() < 0.5 else 0.0
                buy, sell = round(buy), round(sell)
                if buy == 0 and sell == 0:
                    continue
                pb = p * rng.uniform(0.99, 1.01); ps = p * rng.uniform(0.99, 1.01)
                rows.append(dict(date=d, branch=bn, code=c, name=nm[c], buy_lots=buy, sell_lots=sell, net_lots=buy - sell,
                                 buy_amt=round(buy * pb), sell_amt=round(sell * ps), net_amt=round(buy * pb - sell * ps)))
    flow = pd.DataFrame(rows)
    chip_days = dates
    FOR = pd.DataFrame(rng.normal(0, 800, (len(chip_days), len(codes))), index=chip_days, columns=codes)
    TRU = pd.DataFrame(rng.normal(0, 200, (len(chip_days), len(codes))), index=chip_days, columns=codes)
    MAR = pd.DataFrame(np.cumsum(rng.normal(0, 150, (len(chip_days), len(codes))), axis=0) + 20000, index=chip_days, columns=codes)
    return [d.date() for d in dates], flow, C, V, FOR, TRU, MAR
def make_demo_extras(C, V, flow):
    """V1.1 新資料源的模擬版：開高低、產業、當沖量、零股、全市場家數"""
    rng = np.random.default_rng(11)
    sh = pd.DataFrame(rng.uniform(0.002, 0.02, C.shape), index=C.index, columns=C.columns)
    O = C.shift(1).fillna(C) * (1 + rng.normal(0, 0.005, C.shape))
    H = np.maximum(O, C) * (1 + sh); Lo = np.minimum(O, C) * (1 - sh * rng.uniform(0.5, 2, C.shape))
    secs = ['半導體業', '電腦及週邊設備業', '電子零組件業', '航運業', '電機機械']
    IND = {c: secs[i % len(secs)] for i, c in enumerate(C.columns)}
    DTV = (V * pd.DataFrame(rng.uniform(0.15, 0.45, V.shape), index=V.index, columns=V.columns)).iloc[-245:]
    ODD = (V * 1000 * 0.01 * pd.DataFrame(rng.lognormal(0, 0.4, V.shape), index=V.index, columns=V.columns)).iloc[-ODD_LOT_DAYS:] if ODD_LOT_DAYS else pd.DataFrame()
    days = sorted(flow['date'].unique())[-BREADTH_DAYS:]
    rows = [dict(code=c, date=pd.Timestamp(d), nbuy=int(rng.integers(150, 600)), nsell=int(rng.integers(150, 600)),
                 top_buy=float(rng.integers(500, 8000)), top_sell=float(rng.integers(500, 8000))) for c in C.columns for d in days]
    return H, Lo, O, IND, DTV, ODD, pd.DataFrame(rows)


In [ ]:
# ============ 6. 執行資料抓取 ============
t0 = time.time()
if USE_DEMO_DATA:
    DATES, FLOW, CLOSE_ALL, VOL_ALL, FOR_RAW, TRU_RAW, MAR_RAW = make_demo_data(len(pd.bdate_range(START_DATE, dt.date.today())))
    ADJ_ALL = CLOSE_ALL.copy()
    HIGH_ALL, LOW_ALL, OPEN_ALL, INDUSTRY, DTV_RAW, ODD_RAW, MKT_BREADTH = make_demo_extras(CLOSE_ALL, VOL_ALL, FLOW)
    print('⚠️ 使用模擬資料')
else:
    DATES = get_trading_dates(START_DATE)
    print(f'交易日區間：{DATES[0]} ～ {DATES[-1]}（{len(DATES)} 日）')
    DATES = probe_earliest(DATES)
    FLOW = collect_flows(DATES)
    CLOSE_ALL, VOL_ALL, ADJ_ALL, HIGH_ALL, LOW_ALL, OPEN_ALL = fetch_prices(sorted(FLOW['code'].unique()), start=(pd.Timestamp(DATES[0]) - pd.Timedelta(days=70)).strftime('%Y-%m-%d'))
    _chip_days = [d for d in DATES if INST_START and d >= INST_START]
    # V1.4：TWSE 與 TPEx 是不同主機，兩邊各自單線程並行抓取，不增加任一主機的請求頻率
    with ThreadPoolExecutor(2) as _ex:
        _fw = _ex.submit(fetch_twse_chips, _chip_days)
        _fo = _ex.submit(fetch_tpex_chips, _chip_days if TPEX_CHIPS else [])
        FOR_RAW, TRU_RAW, MAR_RAW = _fw.result()
        _TF, _TT, _TM = _fo.result()
    FOR_RAW, TRU_RAW, MAR_RAW = merge_chips(FOR_RAW, _TF), merge_chips(TRU_RAW, _TT), merge_chips(MAR_RAW, _TM)
    # ---- V1.1 新增資料 ----
    INDUSTRY = fetch_industry()                                                                  # S01 族群
    DTV_RAW = fetch_daytrade_vol([d for d in DATES if DAYTRADE_START and d >= DAYTRADE_START])   # S07 當沖量
    ODD_RAW = fetch_odd_lot(DATES[-ODD_LOT_DAYS:] if ODD_LOT_DAYS else [])                       # S10 零股
    MKT_BREADTH = pd.DataFrame()                                                                 # S08 全市場家數
    if BREADTH_STOCKS:
        _recent = FLOW[pd.to_datetime(FLOW['date']) >= pd.Timestamp(DATES[-min(BREADTH_DAYS, len(DATES))])]
        _act = _recent[_recent['code'].astype(str).str.fullmatch(r'[1-9]\d{3}')].groupby('code')['net_lots'].apply(lambda x: x.abs().sum())
        MKT_BREADTH = fetch_market_breadth(list(_act.sort_values(ascending=False).index[:BREADTH_STOCKS]), DATES[-BREADTH_DAYS:])
FLOW.to_csv(os.path.join(OUTPUT_DIR, 'broker_flow_raw.csv'), index=False, encoding='utf-8-sig')
print(f'資料準備完成，耗時 {time.time() - t0:.0f} 秒')
FLOW.head()


In [ ]:
# ============ 7. 因子引擎（所有策略共用） ============
def rs(P, n):
    return P.rolling(n, min_periods=1).sum()

def consec_pos(P):
    a = P.values > 0
    out = np.zeros(a.shape, dtype=float)
    for i in range(a.shape[0]):
        out[i] = np.where(a[i], (out[i - 1] + 1) if i > 0 else 1, 0)
    return pd.DataFrame(out, index=P.index, columns=P.columns)

def rsi(C, n=14):
    d = C.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    out = 100 - 100 / (1 + up / dn.replace(0, np.nan))
    out = out.mask((dn == 0) & (up > 0), 100.0)
    out = out.mask((up == 0) & (dn > 0), 0.0)
    return out.mask((up == 0) & (dn == 0), 50.0)

FL = FLOW.copy()
FL['date'] = pd.to_datetime(FL['date']).dt.normalize()
FL['code'] = FL['code'].astype(str).str.upper()
CLOSE_ALL.index = _naive(CLOSE_ALL.index); VOL_ALL.index = _naive(VOL_ALL.index); ADJ_ALL.index = _naive(ADJ_ALL.index)
IDX = pd.DatetimeIndex(sorted(pd.to_datetime(DATES).normalize().unique()))
if len(IDX) == 0:
    raise RuntimeError('交易日序列為空')
FLOW_DATE_COVERAGE = float(FL['date'].nunique() / len(IDX))
BR = [b for b in BRANCHES if b in set(FL['branch'])]
NAMES = FL.dropna(subset=['name']).sort_values('date').groupby('code')['name'].last().to_dict()
CODES = sorted(set(FL['code']) & set(CLOSE_ALL.columns))
# V1.5 存活者偏誤診斷：分點交易過但抓不到股價的普通股（多為下市／轉板），量化被排除的部位有多大
_com = FL['code'].str.fullmatch(r'[1-9]\d{3}')
_np_codes = sorted(set(FL.loc[_com, 'code']) - set(CLOSE_ALL.columns))
_absn = FL.loc[_com].assign(a=lambda x: x['net_lots'].abs().fillna(x['net_amt'].abs() / 100))
SURV = dict(n=len(_np_codes), n_all=int(FL.loc[_com, 'code'].nunique()),
            share=float(_absn.loc[_absn['code'].isin(_np_codes), 'a'].sum() / max(_absn['a'].sum(), 1e-9)),
            codes=[f'{c} {NAMES.get(c, "")}'.strip() for c in _absn[_absn['code'].isin(_np_codes)].groupby('code')['a'].sum().sort_values(ascending=False).index[:15]])
if SURV['n']:
    print(f"ℹ️ 存活者偏誤：{SURV['n']}/{SURV['n_all']} 檔普通股有分點交易但無股價（多為下市／轉板），占分點淨買賣量 {SURV['share']:.1%}，未納入分析")
# V1.5 Top50 截尾診斷：某分點某日的買超（或賣超）列滿 TRUNC_ROWS 檔 → 未上榜的交易被當成 0
_sg = FL['net_lots'].fillna(FL['net_amt'])
_side = FL.assign(b=_sg > 0, s=_sg < 0).groupby(['branch', 'date'])[['b', 's']].sum()
_hit = (_side['b'] >= TRUNC_ROWS) | (_side['s'] >= TRUNC_ROWS)
TRUNC_BR = _hit.groupby(level=0).mean().reindex(BR).fillna(0.0).to_dict()
TRUNC_ALL = float(_hit.mean()) if len(_hit) else 0.0
print('Top50 截尾觸頂率：' + '、'.join(f'{b} {v:.0%}' for b, v in TRUNC_BR.items()) + ('  ⚠️ 觸頂率高的分點，未上榜交易被低估（成本、隔日沖比例、集中度偏差較大）' if max(TRUNC_BR.values(), default=0) > 0.3 else ''))
FL = FL[FL['code'].isin(CODES)].copy()
CA = CLOSE_ALL[CODES].sort_index()  # 缺價不是可成交的前值
VA = VOL_ALL.reindex(columns=CODES).sort_index()
# 還原收盤價（含除權息）：報酬、均線、RSI、回測一律用還原價，避免除息日被誤判為下跌
AA = ADJ_ALL.reindex(index=CA.index, columns=CODES)
AA = AA.where(AA.notna(), CA)
_ohl = []
for X in (HIGH_ALL, LOW_ALL, OPEN_ALL):
    X = X.copy(); X.index = _naive(X.index)
    _ohl.append(X.reindex(index=CA.index, columns=CODES))
HA, LA_, OA = _ohl  # 先規整時區，再對齊日期

# ---- 張數／金額 單位自動檢查（對舊快取也有效）----
px = CA.reindex(IDX).stack()
px.index.names = ['date', 'code']
FL = FL.merge(px.rename('px').reset_index(), on=['date', 'code'], how='left')
DIAG = {'rows': len(FL)}
_both = FL['buy_lots'].notna() & FL['buy_amt'].notna() & (FL['buy_lots'] > 0) & FL['px'].notna()
DIAG['both_rows'] = int(_both.sum())
if _both.sum() >= 30:
    _same = (FL.loc[_both, 'buy_lots'] == FL.loc[_both, 'buy_amt']).mean()
    _ratio = (FL.loc[_both, 'buy_amt'] / FL.loc[_both, 'buy_lots'] / FL.loc[_both, 'px']).median()
    DIAG['same_ratio'], DIAG['price_ratio'] = float(_same), float(_ratio)
    _inv = (FL.loc[_both, 'buy_lots'] / FL.loc[_both, 'buy_amt'] / FL.loc[_both, 'px']).median()
    if 0.6 < _inv < 1.6 and not (0.6 < _ratio < 1.6):
        print('🔧 偵測到張數與金額欄位對調（舊快取）→ 自動交換')
        for c in ('buy', 'sell', 'net'):
            FL[c + '_lots'], FL[c + '_amt'] = FL[c + '_amt'].copy(), FL[c + '_lots'].copy()
        DIAG['fix'] = '張數↔金額對調'
    elif _same > 0.8:
        print('🔧 偵測到「張數」欄位其實是金額（兩模式數字相同）→ 張數改用 金額÷收盤價 推估')
        for c in ('buy', 'sell', 'net'):
            FL[c + '_lots'] = np.nan
        DIAG['fix'] = '張數頁回傳金額'
    elif 0.0004 < _ratio < 0.0025:
        print('🔧 偵測到張數單位為「股」→ 自動除以 1000 換算成張')
        for c in ('buy', 'sell', 'net'):
            FL[c + '_lots'] = FL[c + '_lots'] / 1000
        DIAG['fix'] = '股→張'
    elif 400 < _ratio < 2500:
        print('🔧 偵測到金額單位為「元」→ 自動除以 1000 換算成仟元')
        for c in ('buy', 'sell', 'net'):
            FL[c + '_amt'] = FL[c + '_amt'] / 1000
        DIAG['fix'] = '元→仟元'
    elif not (0.6 < _ratio < 1.6):
        print(f'⚠️ 買進均價/收盤價 中位數 = {_ratio:.4f}，單位異常且無法自動判斷，成本將全部改用收盤價推估。')
        DIAG['fix'] = '無法判斷，全部推估'
        FL['buy_amt_bad'] = True

# 只有張數或只有金額時，用收盤價互相推估；買進均價只在兩者皆有時計算
has_both = FL['buy_lots'].notna() & FL['buy_amt'].notna() & ('buy_amt_bad' not in FL.columns)
for s_ in ('buy', 'sell', 'net'):
    FL[s_ + '_lots'] = FL[s_ + '_lots'].fillna(FL[s_ + '_amt'] / FL['px'])
    FL[s_ + '_amt'] = FL[s_ + '_amt'].fillna(FL[s_ + '_lots'] * FL['px'])
FL['avg_buy'] = np.where(has_both & (FL['buy_lots'] > 0), FL['buy_amt'] / FL['buy_lots'].where(FL['buy_lots'] > 0), np.nan)
_dev = (FL['avg_buy'] / FL['px'] - 1).abs()
DIAG['avg_valid'] = int((FL['avg_buy'].notna() & (_dev <= 0.2)).sum())
DIAG['avg_outlier'] = int((_dev > 0.2).sum())
FL.loc[_dev > 0.2, 'avg_buy'] = np.nan          # 均價偏離收盤 >20% 視為異常
# 成本用價：實際買進均價優先，缺值時用當日收盤價推估
FL['cost_px'] = FL['avg_buy'].fillna(FL['px'])
FL['cost_src'] = np.where(FL['avg_buy'].notna(), '實際均價', np.where(FL['px'].notna(), '收盤推估', None))
# V1.5 除權息調整：買進價 × 當日（還原價 ÷ 原始價）→ 換到還原價尺度，之後一律與還原收盤 CJ 比較。
# 最新交易日還原價 = 原始價，所以顯示的成本即「以今天價格尺度表示」的成本。
_af = (AA / CA).replace([np.inf, -np.inf], np.nan).reindex(IDX).stack()
_af.index.names = ['date', 'code']
FL = FL.merge(_af.rename('adjf').reset_index(), on=['date', 'code'], how='left')
FL['cost_raw'] = FL['cost_px']
FL['cost_px'] = FL['cost_px'] * FL['adjf'].fillna(1.0)
DIAG['adj_rows'] = int((FL['adjf'].fillna(1.0).sub(1).abs() > 1e-4).sum())
FL = FL.dropna(subset=['net_lots'])
FL['posq'] = np.where(FL['cost_px'].notna(), FL['net_lots'].clip(lower=0), 0.0)
FL['posw'] = FL['posq'] * FL['cost_px'].fillna(0)
FL['posw2'] = FL['posq'] * FL['cost_px'].fillna(0) ** 2     # S11 成本區上下緣（加權標準差）用
FL['buy_lots'] = FL['buy_lots'].fillna(0); FL['sell_lots'] = FL['sell_lots'].fillna(0)
_pos = FL['net_lots'] > 0
DIAG['netbuy_rows'] = int(_pos.sum())
DIAG['real_share'] = float((FL.loc[_pos, 'cost_src'] == '實際均價').mean()) if _pos.any() else 0.0
print(f"成本資料：淨買超 {DIAG['netbuy_rows']:,} 筆，其中實際均價 {DIAG['real_share']:.0%}、收盤推估 {1 - DIAG['real_share']:.0%}")

def panel(field, branch=None):
    d = FL if branch is None else FL[FL['branch'] == branch]
    p = d.pivot_table(index='date', columns='code', values=field, aggfunc='sum')
    out = p.reindex(index=IDX, columns=CODES).fillna(0.0)
    # Top50 未上榜仍採零代理；整個分點日無紀錄則不視為零。
    observed_days = pd.Series(IDX.isin(d['date'].unique()), index=IDX)
    return out.where(observed_days, axis=0)

OBSERVED_BRANCH_COUNT = FL.groupby(['date', 'code'])['branch'].nunique().unstack().reindex(index=IDX, columns=CODES).fillna(0)
OBSERVED_BRANCH_FRACTION = OBSERVED_BRANCH_COUNT / max(len(BR), 1)
# V1.4：分點日覆蓋率（哪些分點在哪些交易日有資料），顯示於每頁頁首
BR_OBS = pd.DataFrame({b: IDX.isin(FL.loc[FL['branch'] == b, 'date'].unique()) for b in BR}, index=IDX)
COV_ALL = float(BR_OBS.values.mean()) if len(BR) else 0.0
COV_LAST = int(BR_OBS.iloc[-1].sum()) if len(BR) else 0
_never = [b for b in BRANCHES if b not in BR]
if _never:
    print(f'⚠️ 以下分點整段期間都沒有資料，未納入：{"、".join(_never)}')
if COV_LAST < len(BR):
    print(f'⚠️ 最新交易日 {IDX[-1]:%Y-%m-%d} 只有 {COV_LAST}/{len(BR)} 個分點有資料（缺：{"、".join(b for b in BR if not BR_OBS[b].iloc[-1])}）；'
          '今日訊號以有資料的分點計算，請留意或稍後重跑補抓。')
print(f'分點日覆蓋率 {COV_ALL:.1%}')
NET  = {b: panel('net_lots', b) for b in BR}
BUY  = {b: panel('buy_lots', b) for b in BR}
SELL = {b: panel('sell_lots', b) for b in BR}
AMTN = {b: panel('net_amt', b) for b in BR}
POSQ = {b: panel('posq', b) for b in BR}
POSW = {b: panel('posw', b) for b in BR}
def _bsum(D):
    """V1.3：跨分點加總。任一分點有資料即加總（缺漏分點視為未知、不拖累整天），全部分點皆缺才為 NaN。"""
    return pd.concat(list(D.values())).groupby(level=0).sum(min_count=1).reindex(index=IDX, columns=CODES)
NET_T, BUY_T, SELL_T = _bsum(NET), _bsum(BUY), _bsum(SELL)
CPX = {b: FL[FL['branch'] == b].pivot_table(index='date', columns='code', values='cost_px', aggfunc='mean').reindex(index=IDX, columns=CODES) for b in BR}
NB5  = {b: NET[b].rolling(5, min_periods=1).sum() for b in BR}     # 各分點滾動買超（預先算好，3 年資料下避免重複計算）
NB20 = {b: NET[b].rolling(20, min_periods=1).sum() for b in BR}
AMTN_T, POSQ_T, POSW_T = _bsum(AMTN), _bsum(POSQ), _bsum(POSW)
POSW2_T = panel('posw2')

# 價格與技術指標（先在完整價格序列上算，再對齊分點日期）
C = CA.reindex(IDX)                 # 原始收盤：與分點成本（原始成交價）比較時使用
CJ = AA.reindex(IDX)                # 還原收盤：報酬、均線、型態判斷使用
VOLD = VA.reindex(IDX)
MA5, MA10, MA20 = [AA.rolling(n).mean().reindex(IDX) for n in (5, 10, 20)]
HH20 = AA.rolling(20).max().reindex(IDX)
RSI14 = rsi(AA).reindex(IDX)
RET5, RET10, RET20 = [(AA / AA.shift(n) - 1).reindex(IDX) for n in (5, 10, 20)]
VOLA = AA.pct_change().rolling(60, min_periods=20).std().reindex(IDX)      # 日報酬波動度（60 日）
VOL5, VOL10, VOL20 = [VA.rolling(n, min_periods=max(3, n // 2)).sum().reindex(IDX) for n in (5, 10, 20)]
VR = (VOL5 / 5) / (VOL20 / 20)

# ---- S02 優化：動態流動性門檻 ----
# 依 60 日均量在普通股中的百分位，於 MAT_RATIO_ILLIQ（最低流動性）～ MAT_RATIO_LIQUID（最高流動性）之間線性內插
ADV60 = VA.rolling(60, min_periods=20).mean().reindex(IDX)
ADV20 = VA.rolling(20, min_periods=10).mean().reindex(IDX)
_common = [c for c in CODES if re.fullmatch(r'[1-9]\d{3}', c)]
LIQP = ADV60[_common].rank(axis=1, pct=True).reindex(columns=CODES)
MATR = (MAT_RATIO_ILLIQ + (MAT_RATIO_LIQUID - MAT_RATIO_ILLIQ) * LIQP).fillna(MATERIAL_RATIO)
def material(N, V):
    return N.notna() & V.notna() & (V > 0) & (N > 0) & (N >= np.maximum(MATR * V, MAT_MIN_LOTS))

# ---- S07 優化：實質留倉成交量 = 成交量 − 當沖量 ----
# 有 TWSE 當沖資料的日子直接扣除；沒有的日子（上櫃、或超出 DAYTRADE_YEARS）用該股歷史當沖比例中位數，再不行用全市場中位數估計
if len(DTV_RAW):
    _dtv = DTV_RAW.copy(); _dtv.index = _naive(_dtv.index)
    _dtv = _dtv.reindex(index=VA.index, columns=CODES)
    DT_SHARE = (_dtv / VA.replace(0, np.nan)).clip(0, 0.95)
else:
    DT_SHARE = pd.DataFrame(np.nan, index=VA.index, columns=CODES)
# 僅用 t-1 以前已知資訊補值，不用未來一年資料改寫前兩年。
_med_stock = DT_SHARE.expanding(min_periods=1).median().shift(1)
_med_cross = _med_stock.median(axis=1)
DT_SHARE_F = DT_SHARE.fillna(_med_stock)
DT_SHARE_F = DT_SHARE_F.apply(lambda s: s.fillna(_med_cross)).fillna(0.0)
DT_OBSERVED = DT_SHARE.notna()
DT_ESTIMATED = DT_SHARE.isna() & DT_SHARE_F.ne(0)
DT_SRC = 'TWSE 當沖成交量' if np.isfinite(DT_SHARE.values).any() else '無當沖資料（未扣除）'
# V1.3：當沖資料未涵蓋全期時，早期集中度未扣當沖、晚期有扣，時間序列不可比 → 警告
_dt_first = DT_SHARE.reindex(IDX).dropna(how='all').index
if len(_dt_first) and _dt_first[0] > IDX[min(20, len(IDX) - 1)]:
    print(f'⚠️ 當沖資料自 {_dt_first[0]:%Y-%m-%d} 起（分點自 {IDX[0]:%Y-%m-%d}），之前的集中度未扣當沖，'
          'S13 集中度分數與 S15 因子②回測會有斷層；請確認 DAYTRADE_YEARS = LOOKBACK_YEARS 並重跑資料抓取。')
RVA = VA * (1 - DT_SHARE_F)                                   # 實質留倉成交量（張）
RVOL1 = RVA.reindex(IDX)
RVOL5, RVOL20 = [RVA.rolling(n, min_periods=max(3, n // 2)).sum().reindex(IDX) for n in (5, 20)]
DTSH = DT_SHARE_F.reindex(IDX)

LBN = min(60, len(IDX))
N1 = NET_T
N5, N10, N20, N60 = rs(NET_T, 5), rs(NET_T, 10), rs(NET_T, 20), rs(NET_T, LBN)
# 有效買超：淨買超且 ≥ 同期成交量 MATERIAL_RATIO（過濾零星買超）
MAT5, MAT10, MAT20 = [material(N, V) for N, V in ((N5, VOL5), (N10, VOL10), (N20, VOL20))]
# ---- S01 優化：EMA 加權累積（近期權重高，降低 20 天前單筆大單的干擾）----
ENET = {n: NET_T.ewm(span=n, adjust=False).mean() * n for n in EMA_SPANS}
CONS = consec_pos(NET_T)
AMT5 = rs(AMTN_T, 5); AMT5P = AMT5.shift(5)
# 集中度分母改用「實質留倉成交量」（扣除當沖量）；保留原始版本 CONC*_RAW 供對照
CONC1 = NET_T / RVOL1.replace(0, np.nan)
CONC5 = N5 / RVOL5.replace(0, np.nan)
CONC20 = N20 / RVOL20.replace(0, np.nan)
CONC1_RAW, CONC5_RAW, CONC20_RAW = NET_T / VOLD.replace(0, np.nan), N5 / VOL5.replace(0, np.nan), N20 / VOL20.replace(0, np.nan)
BUYERS = sum((NET[b] > 0).astype(int) for b in BR)
SELLERS = sum((NET[b] < 0).astype(int) for b in BR)
BUYERS5 = sum((NB5[b] > 0).astype(int) for b in BR)
SELLERS5 = sum((NB5[b] < 0).astype(int) for b in BR)
BRD, BRD5 = BUYERS - SELLERS, BUYERS5 - SELLERS5

# 庫存成本（含賣出扣減）：各分點逐日 買進→加權平均成本、賣出→扣減庫存（成本不變）、庫存按半衰期衰減，
# 再以各分點剩餘庫存加權合計。避免「買進後已賣光，仍被當成成本支撐」的問題。
def inventory_cost(halflife):
    """淨流量庫存代理：庫存張數與支撐時間權重分離，非真實帳戶持股。"""
    if halflife <= 0:
        raise ValueError('halflife 必須大於0')
    lam = 0.5 ** (1 / halflife)
    T, N = len(IDX), len(CODES)
    num, support_den, physical = [np.zeros((T, N)) for _ in range(3)]
    for branch in BR:
        net = NET[branch].values
        cp = CPX[branch].values
        cp = np.where(np.isfinite(cp), cp, CJ.values)   # V1.5：還原價尺度
        qty, wqty, wcost = np.zeros(N), np.zeros(N), np.zeros(N)
        for t in range(T):
            wqty *= lam; wcost *= lam  # 只衰減支撐影響力，不憑空減少張數
            nt = net[t]
            buying = (nt > 0) & np.isfinite(cp[t])
            added = np.where(buying, nt, 0.0)
            qty += added; wqty += added
            wcost += np.where(buying, added * cp[t], 0.0)
            selling = np.where(np.isfinite(nt) & (nt < 0), -nt, 0.0)
            remain = np.maximum(qty - selling, 0.0)
            frac = np.divide(remain, qty, out=np.zeros(N), where=qty > 0)
            wqty *= frac; wcost *= frac; qty = remain
            num[t] += wcost; support_den[t] += wqty; physical[t] += qty
    cost = np.divide(num, support_den, out=np.full((T, N), np.nan), where=support_den > 1e-12)
    return pd.DataFrame(cost, index=IDX, columns=CODES), pd.DataFrame(physical, index=IDX, columns=CODES)

COST, INV_POS = inventory_cost(COST_HALFLIFE)
DIST = CJ / COST - 1   # V1.5：成本與股價同為還原價尺度（除息不會被當成跌破成本）

# 隔日沖比例（股票層級，已知資訊：事件在 t-k，於 t 收盤後確認）
# S05 優化：同時計算 T+1 與 T+1～T+N 日內累積出清比例，捕捉「隔兩日沖」「三日短波段」
_ev = NET_T >= DT_MIN_LOTS
def _clear_ratio(k, net=False):
    """k=1、net=False：原定義（隔日賣出張數 ÷ 事件日買進張數）。
    net=True：k 日內「淨賣回」比例 = max(0, −Σ淨買超(t+1..t+k)) ÷ 事件日淨買超——不會把每天雙向進出的活躍分點誤判成短沖"""
    if net:
        s_ = -sum(NET_T.shift(-j) for j in range(1, k + 1))
        r_ = (s_.clip(lower=0) / NET_T.where(NET_T > 0)).where(_ev)
    else:
        s_ = sum(SELL_T.shift(-j) for j in range(1, k + 1))
        r_ = (s_ / BUY_T.replace(0, np.nan)).where(_ev)
    return r_.clip(upper=2).shift(k).rolling(LBN, min_periods=1).mean()
DTR1 = _clear_ratio(1)
DTRH = _clear_ratio(DT_HORIZON, net=True) if DT_HORIZON > 1 else DTR1
DTR = np.fmax(DTR1, DTRH)   # 隔日沖扣分／排除：T+1 隔日沖 或 N 日內淨賣回，取較高者（抓得到 T+2、T+3 出場的短線客）

# 反轉訊號
n5p, n10p, n20p = N5.shift(1), N10.shift(1), N20.shift(1)
_mat_prev = np.maximum(DT_MIN_LOTS, MATR.shift(1) * VOL5.shift(1).fillna(0))   # 先前的累積量要有份量，避免 1、2 張就觸發
# S11 優化：防守線改用 ATR 移動停損（Chandelier）與主力成本密集區上下緣，降低震盪市跌破 10MA 的假訊號
_tr = pd.concat([(HA - LA_), (HA - CA.shift(1)).abs(), (LA_ - CA.shift(1)).abs()]).groupby(level=0).max()
ATR = _tr.rolling(ATR_N, min_periods=ATR_N // 2).mean()
_adjf = (AA / CA).replace([np.inf, -np.inf], np.nan).fillna(1.0)                # 原始價 ATR 換算成還原價尺度
_hadj = (HA * _adjf).where(HA.notna(), AA); _ladj = (LA_ * _adjf).where(LA_.notna(), AA)   # V1.5：Chandelier 用最高／最低價（還原）
STOP_LONG = (_hadj.rolling(22, min_periods=5).max() - ATR_MULT * ATR * _adjf).shift(1).reindex(IDX)    # 多單移動停損
STOP_SHORT = (_ladj.rolling(22, min_periods=5).min() + ATR_MULT * ATR * _adjf).shift(1).reindex(IDX)   # 空方反壓線
_q60, _w60, _w260 = rs(POSQ_T, 60), rs(POSW_T, 60), rs(POSW2_T, 60)
_zm = (_w60 / _q60.replace(0, np.nan)).where(_q60 >= DT_MIN_LOTS)
_zs = np.sqrt((_w260 / _q60.replace(0, np.nan) - _zm ** 2).clip(lower=0))
ZONE_LO, ZONE_HI = _zm - 0.674 * _zs, _zm + 0.674 * _zs                          # 近 60 日買進價 ≈P25～P75（V1.5：還原價尺度）
def _brk(dn):
    ma = (CJ < MA10) if dn else (CJ > MA10)
    stop = STOP_LONG if dn else STOP_SHORT
    atr = ((CJ < stop) if dn else (CJ > stop)).where(stop.notna(), ma)          # 沒有高低價時退回 10MA
    zl = ZONE_LO if dn else ZONE_HI
    zone = ((CJ < zl) if dn else (CJ > zl)) & zl.notna()
    return {'MA10': ma, 'ATR': atr, 'ZONE': zone.where(zl.notna(), atr), 'ATR_ZONE': atr | zone}[REV_BREAK_MODE].astype(bool)
BRK_DN, BRK_UP = _brk(True), _brk(False)
BEAR = (n5p > 0) & (n10p > 0) & (n20p > 0) & (n5p >= _mat_prev) & (NET_T < 0) & (NET_T.abs() >= 0.3 * n5p) & BRK_DN
BULL = (n5p < 0) & (n10p < 0) & (n20p < 0) & (n5p.abs() >= _mat_prev) & (NET_T > 0) & (NET_T >= 0.3 * n5p.abs()) & BRK_UP

# 分點強度分數（文件第二層：股票層級＝追蹤分點合計）
ACTIVE20 = rs((NET_T.fillna(0) != 0).astype(int), 20) > 0   # V1.3：NaN 不算有交易
STR = (20 * MAT5 + 20 * MAT10 + 20 * MAT20 + 15 * CONS.clip(upper=5) / 5
       + 10 * ((AMT5 > AMT5P) & (AMT5 > 0)) + 15 * (RET5 > 0)).where(ACTIVE20)

# 法人／融資
FOR_D = FOR_RAW.reindex(index=IDX, columns=CODES) if len(FOR_RAW) else pd.DataFrame(np.nan, index=IDX, columns=CODES)
TRU_D = TRU_RAW.reindex(index=IDX, columns=CODES) if len(TRU_RAW) else pd.DataFrame(np.nan, index=IDX, columns=CODES)
MAR_D = MAR_RAW.reindex(index=IDX, columns=CODES) if len(MAR_RAW) else pd.DataFrame(np.nan, index=IDX, columns=CODES)
FOR5 = FOR_D.rolling(5, min_periods=5).sum()
TRU5 = TRU_D.rolling(5, min_periods=5).sum()
FORCONS = consec_pos(FOR_D.fillna(0)).where(FOR_D.notna())
MARCHG5 = MAR_D - MAR_D.shift(5)

# 股票池
UNIV = [c for c in CODES if re.fullmatch(r'[1-9]\d{3}', c) and C[c].notna().any()]
if UNIVERSE_MODE == '0050':
    UNIV = [c for c in UNIV if c in UNIVERSE_0050]
# 主力籌碼 Score（文件第十三層，權重 18/20/12/8/15/10/8/9）
def _pr(P):
    """只在股票池內、且淨買超 > 0 的個股之間做百分位排名（排除 ETF、債券等干擾）"""
    return P[UNIV].where(P[UNIV] > 0).rank(axis=1, pct=True).reindex(columns=CODES).fillna(0)
S_A = 18 * _pr(N5)
S_B = 20 * _pr(N20)
S_C = (12 * (CONC5 / CONC_SCORE_FULL).clip(0, 1)).where(N5 > 0, 0).fillna(0)
if DT_PENALTY:   # 隔日沖主導的買超不是布局：5 日買超與集中度不給分
    _dt = DTR.fillna(0) >= DT_FLAG_RATIO
    S_A, S_C = S_A.where(~_dt, 0), S_C.where(~_dt, 0)
S_D = 8 * BRD5.clip(lower=0) / max(len(BR), 1)
S_E = pd.DataFrame(np.select([(DIST >= 0) & (DIST <= 0.08), ((DIST >= -0.03) & (DIST < 0)) | ((DIST > 0.08) & (DIST <= 0.15)), (DIST > 0.15) & (DIST <= 0.25)],
                             [15, 10, 5], 0), index=IDX, columns=CODES).where(COST.notna(), 0)
S_F = pd.DataFrame(np.where(FOR5.isna(), 5, np.where((FOR5 > 0) & (FORCONS >= 3), 10, np.where(FOR5 > 0, 6, 0))), index=IDX, columns=CODES)
S_G = pd.DataFrame(np.where(MARCHG5.isna(), 4, np.where(MARCHG5 < 0, 8, 0)), index=IDX, columns=CODES)
S_H = 3 * (CJ > MA20) + 3 * (MA5 > MA20) + 3 * (CJ >= 0.98 * HH20)
SCORE = (S_A + S_B + S_C + S_D + S_E + S_F + S_G + S_H).where(C.notna() & ACTIVE20)

# 前瞻報酬（回測用：訊號日收盤後確認 → 隔日收盤進場）
ENT = AA.shift(-1)       # 還原價：除權息不會被算成虧損
FWD = {h: (AA.shift(-(1 + h)) / ENT - 1).reindex(IDX) for h in (5, 10, 20)}
_adj_low = LA_ * (AA / CA).replace([np.inf, -np.inf], np.nan)
MAE = {h: (_adj_low[::-1].rolling(h, min_periods=h).min()[::-1].shift(-2) / ENT - 1).clip(upper=0).reindex(IDX) for h in (5, 10, 20)}

L = IDX[-1]
print(f'引擎完成：{len(IDX)} 交易日、{len(CODES)} 檔有股價、股票池 {len(UNIV)} 檔、最新日 {L:%Y-%m-%d}')
ENGINE_SIG = _param_sig()   # V1.4：記錄引擎執行時的參數，供 save_page 偵測「改參數未重跑引擎」

In [ ]:
# ============ 8. HTML 版型與共用函式 ============
CSS = r'''
:root{--bg:#0a0e13;--pn:#111821;--bd:#1f2a36;--tx:#d7dee7;--mu:#7d8b99;--ac:#f0b429;--up:#ff4d4f;--dn:#22c55e;--bl:#4ea1ff}
*{box-sizing:border-box}html{color-scheme:dark}
body{margin:0;background:var(--bg);color:var(--tx);font-family:"Noto Sans TC","PingFang TC","Microsoft JhengHei",system-ui,sans-serif;font-size:14px;line-height:1.6}
header{padding:20px 24px 16px;border-bottom:1px solid var(--bd);background:linear-gradient(180deg,#121a24,#0a0e13)}
header a{color:var(--mu);text-decoration:none;font-size:13px}header a:hover{color:var(--ac)}
.no{display:inline-block;background:var(--ac);color:#111;font-weight:800;border-radius:6px;padding:1px 8px;margin-right:8px;font-family:ui-monospace,Menlo,Consolas,monospace}
h1{font-size:22px;margin:8px 0 4px}h2{font-size:16px;margin:0 0 10px;color:#e8edf3}h3{font-size:14px;margin:0 0 8px;color:var(--ac)}
.sub{color:var(--mu);margin:0}.meta{color:var(--mu);font-size:12px;margin-top:8px;font-family:ui-monospace,Menlo,Consolas,monospace}
.demo{background:#3a1d00;color:#ffb74d;padding:10px 24px;font-weight:700;border-bottom:1px solid #5a3200}
main{padding:18px 24px;display:flex;flex-direction:column;gap:16px;max-width:1500px;margin:0 auto}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}
.card{background:var(--pn);border:1px solid var(--bd);border-radius:10px;padding:14px;min-width:0}
.method{background:#0d141c;border:1px solid var(--bd);border-left:3px solid var(--ac);border-radius:8px;padding:12px 16px;color:#b9c4cf;font-size:13px}
.method ul{margin:4px 0;padding-left:20px}.method code{color:var(--ac)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px}
.kpi{background:var(--pn);border:1px solid var(--bd);border-radius:10px;padding:10px 14px}
.kl{color:var(--mu);font-size:12px}.kv{font-size:22px;font-weight:700;font-family:ui-monospace,Menlo,Consolas,monospace}
.tw{overflow:auto;max-height:560px;border:1px solid var(--bd);border-radius:8px}
table{border-collapse:collapse;width:100%;font-size:13px}
th{position:sticky;top:0;background:#16202b;color:#9fb0c0;font-weight:600;text-align:left;padding:7px 8px;cursor:pointer;white-space:nowrap;border-bottom:1px solid var(--bd);z-index:1}
td{padding:5px 8px;border-bottom:1px solid #16202a;white-space:nowrap}
tr:hover td{background:#152030}
td.num{text-align:right;font-family:ui-monospace,Menlo,Consolas,monospace}
.up{color:var(--up)}.dn{color:var(--dn)}.mu{color:var(--mu)}.ok{color:var(--up);font-weight:700}.ng{color:#4b5866}
.pill{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;font-weight:600}
.pill.u{background:rgba(255,77,79,.15);color:#ff7a7c}.pill.d{background:rgba(34,197,94,.15);color:#4ade80}.pill.n{background:rgba(240,180,41,.15);color:#f5c451}.pill.g{background:#1d2732;color:#9fb0c0}
.tabs{display:flex;flex-wrap:wrap;gap:6px}.tabs button,select{background:#16202b;color:var(--tx);border:1px solid var(--bd);border-radius:6px;padding:6px 12px;font:inherit;cursor:pointer}
.tabs button.on{background:var(--ac);color:#111;border-color:var(--ac);font-weight:700}
.ch{position:relative;height:300px}.ch.sm{height:180px}.ch.lg{height:420px}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-bottom:10px}
.mini{display:grid;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:12px}
.note{color:var(--mu);font-size:12px}
footer{padding:16px 24px 28px;color:var(--mu);font-size:12px;border-top:1px solid var(--bd);max-width:1500px;margin:0 auto}
@media(max-width:640px){main,header{padding-left:12px;padding-right:12px}.kv{font-size:18px}}
'''

JSLIB = r'''
const UP='#ff4d4f',DN='#22c55e',AC='#f0b429',BL='#4ea1ff',MU='#7d8b99',GR='#1a2430';
const fmtN=(v,d=0)=>v==null||!isFinite(v)?'—':Number(v).toLocaleString('zh-TW',{minimumFractionDigits:d,maximumFractionDigits:d});
const fmtP=(v,d=1)=>v==null||!isFinite(v)?'—':(v*100).toFixed(d)+'%';
const cls=v=>v>0?'up':v<0?'dn':'';
const sc=v=>v>=0?UP:DN;
const $=id=>document.getElementById(id);
const esc=s=>String(s).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
function kpis(id,items){$(id).innerHTML=items.map(x=>`<div class="kpi"><div class="kl">${x[0]}</div><div class="kv ${x[2]||''}">${x[1]}</div></div>`).join('')}
function table(id,cols,rows,opt={}){
  const el=$(id);let sk=opt.sort||null,sd=opt.asc?1:-1;const H={};
  cols.forEach(c=>{if(c.heat){const v=rows.map(r=>r[c.k]).filter(x=>x!=null&&isFinite(x));H[c.k]=v.length?[Math.min(...v),Math.max(...v)]:[0,1]}});
  const cell=(c,r)=>{const v=r[c.k];let s,cl='',st='',num=['n','p','f'].includes(c.f);
    if(c.f==='n')s=fmtN(v,c.d||0);else if(c.f==='p')s=fmtP(v,c.d??1);else if(c.f==='f')s=fmtN(v,c.d??2);
    else if(c.f==='tag')s=v?`<span class="pill ${r[c.tc]||c.tone||'n'}">${esc(v)}</span>`:'<span class=mu>—</span>';
    else if(c.f==='ck')s=v===true?'<span class=ok>✓</span>':v===false?'<span class=ng>✗</span>':'<span class=mu>N/A</span>';
    else s=(v==null||v==='')?'—':esc(v);
    if(c.c)cl=cls(v);
    if(c.heat&&v!=null&&isFinite(v)){const[lo,hi]=H[c.k];
      if(c.heat==='pos'){const a=hi>lo?(v-lo)/(hi-lo)*0.5:0.25;st=`background:rgba(240,180,41,${a.toFixed(2)})`}
      else{const m=Math.max(Math.abs(lo),Math.abs(hi))||1;const a=Math.min(Math.abs(v)/m,1)*0.5;st=`background:${v>=0?'rgba(255,77,79,':'rgba(34,197,94,'}${a.toFixed(2)})`}}
    return `<td class="${num?'num ':''}${cl}" style="${st}">${s}</td>`};
  const render=()=>{let rs=[...rows];if(sk)rs.sort((a,b)=>{const x=a[sk],y=b[sk];if(x==null)return 1;if(y==null)return -1;return(x>y?1:x<y?-1:0)*sd});
    el.innerHTML=`<div class="tw" style="${opt.h?'max-height:'+opt.h+'px':''}"><table><thead><tr>${cols.map(c=>`<th data-k="${c.k}" title="${c.tip||''}">${c.t}${sk===c.k?(sd>0?' ▲':' ▼'):''}</th>`).join('')}</tr></thead><tbody>${rs.length?rs.map(r=>`<tr>${cols.map(c=>cell(c,r)).join('')}</tr>`).join(''):`<tr><td colspan="${cols.length}" class="mu">（無符合資料）</td></tr>`}</tbody></table></div>`;
    el.querySelectorAll('th').forEach(th=>th.onclick=()=>{const k=th.dataset.k;if(sk===k)sd=-sd;else{sk=k;sd=-1}render()})};
  render()}
const CH={};
function mk(id,cfg){if(typeof Chart==='undefined'){$(id).parentNode.innerHTML='<p class=note>圖表函式庫未載入（需要網路）</p>';return}
  if(CH[id])CH[id].destroy();const o=cfg.options||{};o.responsive=true;o.maintainAspectRatio=false;o.animation=false;
  o.plugins=Object.assign({legend:{labels:{color:'#9fb0c0',boxWidth:12}}},o.plugins||{});
  if(!o.scales&&cfg.type!=='doughnut'&&cfg.type!=='pie')o.scales={x:ax(),y:ax()};
  cfg.options=o;CH[id]=new Chart($(id),cfg);return CH[id]}
function ax(e){return Object.assign({ticks:{color:MU,maxRotation:0,autoSkip:true},grid:{color:GR}},e||{})}
function sel(id,opts,cb){const s=$(id);s.innerHTML=opts.map(o=>`<option value="${esc(o[0])}">${esc(o[1])}</option>`).join('');s.onchange=()=>cb(s.value);if(opts.length)cb(opts[0][0])}
function tabs(id,names,cb){const el=$(id);el.innerHTML=names.map((n,i)=>`<button data-i="${i}" class="${i?'':'on'}">${n}</button>`).join('');
  el.querySelectorAll('button').forEach(b=>b.onclick=()=>{el.querySelectorAll('button').forEach(x=>x.classList.remove('on'));b.classList.add('on');cb(names[+b.dataset.i])});if(names.length)cb(names[0])}
function cum(a){let s=0;return a.map(v=>s+=(v||0))}
'''

PAGE = r'''<!doctype html>
<html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>__TITLE__</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>__CSS__</style></head><body>
<header><a href="index.html">← 回總覽</a><h1><span class="no">S__NUM__</span>__TITLE__</h1><p class="sub">__SUB__</p>
<div class="meta">資料日期 __DATE__ ｜ 期間 __RANGE__ ｜ 追蹤分點 __BR__ ｜ 分點日覆蓋 __COV__ ｜ Top50 觸頂 __TRUNC__ ｜ 產生 __GEN__</div></header>
__DEMO__
<main><section class="method"><h3>策略邏輯（對應文件第 __CH__ 層）</h3>__METHOD__</section>
__BODY__</main>
<footer>限制說明：分點資料為「交易通路」而非最終投資人身分；每分點每日只列出買超、賣超各前 50 名，未上榜者以 0 作截尾代理，並非證實零交易；整個分點日缺漏保留缺失。成本與庫存僅是淨流量代理、初始庫存未知；買進價已依除權息換算為還原價尺度（以最新交易日價格表示）。V1.2 為程式稽核修正候選，尚未通過真實資料樣本外績效驗證。外資／投信／融資：上市取自 TWSE、上櫃取自 TPEx（TPEx 抓取失敗時上櫃顯示 N/A 並在評分中給中性分）；上櫃當沖量為估計值。本頁僅供研究，不構成投資建議。</footer>
<script>const D=__DATA__;
__JSLIB__
__JS__</script></body></html>'''

PAGES = []
PAGE_DATA = {}   # 各頁資料，供 00_summary.html 彙整

def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return None if not np.isfinite(o) else round(float(o), 4)
    if isinstance(o, (pd.Timestamp, dt.date)):
        return o.strftime('%Y-%m-%d')
    if o is pd.NA or o is pd.NaT:
        return None
    return o

_CHARTJS = {}
def _chartjs_inline():
    """V1.5：下載一次 Chart.js 4.4.1 並快取；失敗則維持 CDN 連結"""
    if 'js' not in _CHARTJS:
        fp = os.path.join(CACHE_DIR, 'chart.umd.4.4.1.min.js')
        try:
            if not os.path.exists(fp):
                r = _sess().get('https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js', timeout=30)
                r.raise_for_status()
                with open(fp, 'w', encoding='utf-8') as f:
                    f.write(r.text)
            _CHARTJS['js'] = open(fp, encoding='utf-8').read()
        except Exception as e:
            print('  Chart.js 內嵌失敗，維持 CDN：', e); _CHARTJS['js'] = None
    return _CHARTJS['js']

def nonoverlap(s, h):
    """V1.5：t 值用的不重疊抽樣——依時間順序挑選彼此間隔 ≥ h 個交易日的觀測值（訊號稀疏時不丟樣本）"""
    s = pd.Series(s).dropna()
    if not len(s):
        return s
    pos = IDX.get_indexer(pd.DatetimeIndex(s.index))
    keep, last = [], -10 ** 9
    for i, p in enumerate(pos):
        if p >= 0 and p - last >= h:
            keep.append(i); last = p
    return s.iloc[keep]

def tstat_of(s):
    s = pd.Series(s).dropna()
    return float(s.mean() / s.std(ddof=1) * np.sqrt(len(s))) if len(s) >= 5 and s.std(ddof=1) > 0 else None

def save_page(num, slug, chapter, title, sub, method, body, js, data, headline=''):
    if _param_sig() != ENGINE_SIG:
        print(f'⚠️ S{num:02d}：參數在因子引擎（Cell 9）執行後被修改，本頁仍用舊因子計算 → 請從 Cell 9 起重跑')
    html = PAGE
    rep = {'__TITLE__': title, '__NUM__': f'{num:02d}', '__SUB__': sub, '__CH__': chapter,
           '__DATE__': f'{L:%Y-%m-%d}', '__RANGE__': f'{IDX[0]:%Y-%m-%d} ～ {IDX[-1]:%Y-%m-%d}（{len(IDX)} 日）',
           '__BR__': '、'.join(BR), '__COV__': f'{COV_ALL:.1%}（最新日 {COV_LAST}/{len(BR)}）', '__TRUNC__': f'{TRUNC_ALL:.0%}', '__GEN__': dt.datetime.now(TPE).strftime('%Y-%m-%d %H:%M'),
           '__DEMO__': '<div class="demo">⚠️ DEMO 模擬資料：本頁數字為隨機產生，僅供檢查版面，請設 USE_DEMO_DATA=False 重跑。</div>' if USE_DEMO_DATA else '',
           '__METHOD__': method, '__BODY__': body, '__CSS__': CSS, '__JSLIB__': JSLIB, '__JS__': js,
           '__DATA__': json.dumps(clean(data), ensure_ascii=False, allow_nan=False).replace('</', '<\\/')}
    for k in ['__CSS__', '__TITLE__', '__NUM__', '__SUB__', '__CH__', '__DATE__', '__RANGE__', '__BR__', '__COV__', '__TRUNC__', '__GEN__', '__DEMO__', '__METHOD__', '__BODY__', '__JSLIB__', '__JS__', '__DATA__']:
        html = html.replace(k, rep[k])
    if OFFLINE_HTML:
        _js = _chartjs_inline()
        if _js:
            html = html.replace('<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>', '<script>' + _js.replace('</script', '<\\/script') + '</script>')
    PAGE_DATA[num] = data
    fn = f'{num:02d}_{slug}.html'
    with open(os.path.join(OUTPUT_DIR, fn), 'w', encoding='utf-8') as f:
        f.write(html)
    global PAGES
    PAGES = [p for p in PAGES if p['num'] != num] + [dict(num=num, file=fn, title=title, sub=sub, headline=headline, chapter=chapter)]
    print(f'✅ S{num:02d} {title} → {fn}  ({len(html) / 1024:.0f} KB)  {headline}')
    return fn

def stk(c):
    return f'{c} {NAMES.get(c, "")}'.strip()

def lv(P, c=None):
    v = P.iloc[-1] if c is None else P[c].iloc[-1]
    return v

def f_(x):
    try:
        x = float(x)
        return x if np.isfinite(x) else None
    except (TypeError, ValueError):
        return None

TONE = {'強勢主力布局': 'u', '偏多': 'u', '中性': 'n', '偏空': 'd', '主力撤退/弱勢': 'd',
        '被套/防守': 'd', '成本附近': 'n', '已有獲利': 'u', '獲利了結風險': 'n',
        '低於成本區': 'd', '成本區內': 'n', '接近成本上緣': 'n', '突破成本區': 'u',
        '波段主力': 'u', '趨勢主力': 'u', '隔日沖': 'n', '短線主力': 'n', '高周轉': 'g', '持續賣方': 'd', '不明顯': 'g',
        '高度集中': 'u', '中度集中': 'u', '輕度集中': 'n', '分散': 'g', '賣方集中': 'd',
        '買盤廣泛': 'u', '賣盤廣泛': 'd', '買方集中、賣方分散': 'u', '賣方集中、買方分散': 'd', '分歧': 'g',
        '吸籌': 'u', '疑似吸籌': 'n', '出貨警報': 'd', '疑似出貨': 'n', '主力轉空': 'd', '籌碼反轉(轉多)': 'u',
        '強': 'u', '偏強': 'u', '普通': 'n', '弱': 'g',
        '隔兩日沖': 'n', '三日短波段': 'n', '非短沖': 'g', '族群建倉': 'u', '出貨警報(散戶接盤)': 'd', '軟性入選': 'n', '全數通過': 'u'}
def tone(s):
    return TONE.get(s, 'n')
# ---- V1.1 共用：橫斷面排名常態化 Z 分數（對極端值穩健；S12 PCA、S13 ML、S14 軟性評分用）----
try:
    from scipy.special import ndtri as _ndtri
except ImportError:
    _ndtri = lambda p: (np.asarray(p) - 0.5) * 3.4641     # 無 scipy 時以均勻分布標準化近似
def rank_z(s):
    """s：一列 Series（同一天、多檔股票）→ 排名轉常態分數，NaN 保持 NaN"""
    s = pd.Series(s, dtype=float)
    ok = s.notna()
    out = pd.Series(np.nan, index=s.index)
    n = int(ok.sum())
    if n >= 3:
        out[ok] = _ndtri((s[ok].rank() - 0.5) / n)
    return out

def panel_z(P, mask=None):
    """整個面板逐日做 rank_z（向量化）"""
    X = P if mask is None else P.where(mask)
    R = X.rank(axis=1)
    n = R.notna().sum(axis=1).replace(0, np.nan)
    Q = R.sub(0.5).div(n, axis=0)
    return pd.DataFrame(_ndtri(Q.clip(1e-6, 1 - 1e-6).values), index=P.index, columns=P.columns).where(R.notna())


In [ ]:
# ============ S01 分點買賣超 ============
def build_s01():
    br = {}
    for b in BR:
        fb = FL[(FL['branch'] == b) & (FL['date'] == L)]
        rowify = lambda df: [dict(stk=stk(r.code), buy=r.buy_lots, sell=r.sell_lots, net=r.net_lots, amt=r.net_amt, avg=r.avg_buy) for r in df.itertuples()]
        P = NET[b]
        R = pd.DataFrame({f'n{n}': rs(P, n).iloc[-1] for n in (1, 3, 5, 10, 20)})
        R['cons'] = consec_pos(P).iloc[-1]
        for n in EMA_SPANS:   # EMA 加權累積：近期權重高，20 天前的單筆大單影響遞減
            R[f'e{n}'] = (P.ewm(span=n, adjust=False).mean() * n).iloc[-1]
        R = R[(P.iloc[-20:] != 0).any()]
        R = R.loc[R['n20'].abs().sort_values(ascending=False).index[:40]]
        rows = [dict(code=c, stk=stk(c), **r.to_dict()) for c, r in R.iterrows()]
        br[b] = dict(top_buy=rowify(fb[fb['net_lots'] > 0].nlargest(15, 'net_lots')),
                     top_sell=rowify(fb[fb['net_lots'] < 0].nsmallest(15, 'net_lots')),
                     roll=rows, series={c: P[c].iloc[-20:].tolist() for c in R.index})
    # ---- 產業板塊合計：偵測分點是否對同一族群多檔同步建倉 ----
    # 產業分類缺失時不可把不同股票併成同一「未分類」族群，否則會製造
    # 虛假的族群建倉；未知標的只排除於族群判定，不改動原始買賣超欄位。
    _known_ind = set(INDUSTRY) if isinstance(INDUSTRY, dict) else set()
    _ind_complete = (not UNIV) or (bool(_known_ind) and all(c in _known_ind for c in UNIV))
    sect = []
    for b, P5, P20 in [(b, NB5[b], NB20[b]) for b in BR] + [('追蹤分點合計', N5, N20)]:
        Pn = NET[b] if b in NET else NET_T
        e20 = (Pn.ewm(span=20, adjust=False).mean() * 20).iloc[-1]
        mat = material(P5, VOL5).iloc[-1]
        df = pd.DataFrame(dict(n5=P5.iloc[-1], n20=P20.iloc[-1], e20=e20, mat=mat)).reindex(UNIV)
        df['sec'] = [INDUSTRY.get(c) if isinstance(INDUSTRY, dict) else None for c in df.index]
        df = df[df['sec'].notna()]
        df = df[(df.n20 != 0) | (df.n5 != 0)]
        for sec, g in df.groupby('sec'):
            m = g[g.mat]
            if not len(m) and g.n20.sum() <= 0:
                continue
            top = m.sort_values('n5', ascending=False).index[:6]
            sect.append(dict(branch=b, sec=sec, nstk=len(m), n5=g.n5.sum(), n20=g.n20.sum(), e20=g.e20.sum(),
                             stks='、'.join(stk(c) for c in top) or '—', t='族群建倉' if len(m) >= SECTOR_MIN_STOCKS else None, tc='u'))
    sect.sort(key=lambda r: (-(r['t'] is not None), -r['nstk'], -r['n5']))
    days = IDX[-10:]
    top = N5.iloc[-1][UNIV].abs().sort_values(ascending=False).index[:20]
    heat = [dict(stk=stk(c), **{f'd{i}': NET_T.loc[d, c] for i, d in enumerate(days)}, s=NET_T.loc[days, c].sum()) for c in top]
    data = dict(br=br, heat=heat, sect=sect[:150], ind=_ind_complete, mins=SECTOR_MIN_STOCKS, spans=list(EMA_SPANS), hd=[d.strftime('%m/%d') for d in days], sd=[d.strftime('%m/%d') for d in IDX[-20:]])
    body = '''
<div class="row"><div class="tabs" id="tb"></div></div>
<div class="kpis" id="k"></div>
<div class="grid"><div class="card"><h2>今日買超前 15</h2><div id="t1"></div></div><div class="card"><h2>今日賣超前 15</h2><div id="t2"></div></div></div>
<div class="card"><h2>多日累積買賣超（不要只看一天）</h2><p class="note">依 20 日累積買賣超絕對值排序；點欄位排序。</p><div id="t3"></div></div>
<div class="card"><div class="row"><h2 style="margin:0">逐日買賣超 × 累積</h2><select id="s1"></select></div><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>追蹤分點合計：近 10 日逐日淨買超熱度（張）</h2><div id="t4"></div></div>
<div class="card"><h2>產業板塊合計：族群性建倉偵測</h2><p class="note" id="sn"></p><div id="t5"></div></div>'''
    js = r'''
const cs=[{k:'stk',t:'股票'},{k:'buy',t:'買進(張)',f:'n'},{k:'sell',t:'賣出(張)',f:'n'},{k:'net',t:'淨買超(張)',f:'n',c:1},{k:'amt',t:'淨額(仟元)',f:'n',c:1},{k:'avg',t:'買進均價',f:'f'}];
function drawBr(b){const B=D.br[b];
 kpis('k',[['今日買超列示檔數(前15)',B.top_buy.length],['今日賣超列示檔數(前15)',B.top_sell.length],
  ['前15買超合計(張)',fmtN(B.top_buy.reduce((a,x)=>a+x.net,0)),'up'],['前15賣超合計(張)',fmtN(B.top_sell.reduce((a,x)=>a+x.net,0)),'dn']]);
 table('t1',cs,B.top_buy,{h:420});table('t2',cs,B.top_sell,{h:420});
 table('t3',[{k:'stk',t:'股票'},{k:'n1',t:'1日',f:'n',c:1,heat:1},{k:'n3',t:'3日',f:'n',c:1,heat:1},{k:'n5',t:'5日',f:'n',c:1,heat:1},{k:'n10',t:'10日',f:'n',c:1,heat:1},{k:'n20',t:'20日',f:'n',c:1,heat:1}].concat(D.spans.map(n=>({k:'e'+n,t:'EMA'+n+'加權',f:'n',c:1,heat:1,tip:'指數加權累積：越近的買超權重越高'}))).concat([{k:'cons',t:'連續買超天數',f:'n'}]),B.roll);
 sel('s1',B.roll.map(r=>[r.code,r.stk]),c=>{const s=B.series[c];mk('c1',{data:{labels:D.sd,datasets:[
  {type:'bar',label:'單日淨買超(張)',data:s,backgroundColor:s.map(sc)},
  {type:'line',label:'累積(張)',data:cum(s),borderColor:AC,pointRadius:0,yAxisID:'y1'}]},
  options:{scales:{x:ax(),y:ax(),y1:ax({position:'right',grid:{display:false}})}}})});}
tabs('tb',Object.keys(D.br),drawBr);
const hc=[{k:'stk',t:'股票'}].concat(D.hd.map((d,i)=>({k:'d'+i,t:d,f:'n',c:1,heat:1}))).concat([{k:'s',t:'10日合計',f:'n',c:1}]);
table('t4',hc,D.heat);
$('sn').textContent=(D.ind?'':'⚠️ 產業別資料未完整涵蓋股票池；未知標的未納入族群判定。')+`同一分點在同產業有 ≥${D.mins} 檔「有效買超」（5 日）標示為族群建倉。`;
table('t5',[{k:'branch',t:'分點'},{k:'sec',t:'產業'},{k:'t',t:'判定',f:'tag',tc:'tc'},{k:'nstk',t:'有效買超檔數',f:'n',heat:'pos'},{k:'n5',t:'5日淨買(張)',f:'n',c:1},{k:'n20',t:'20日淨買(張)',f:'n',c:1},{k:'e20',t:'EMA20加權',f:'n',c:1},{k:'stks',t:'主要個股'}],D.sect,{sort:null});'''
    method = '''<ul><li>逐日抓取每個分點的買超／賣超排行（張數＋金額），計算 1／3／5／10／20 日累積淨買超。</li>
<li>文件重點：「連續 3→5→10→20 日」的累積買超，比單日突然大買更有研究價值；同時列出連續買超天數。</li>
<li>買進均價 = 買進金額(仟元) ÷ 買進張數，單位為元／股。</li>
<li><b>V1.1 優化</b>：① 新增 <b>EMA 加權累積</b>（span 5／20，累積量 = EMA × span），近期買超權重較高，20 天前的單筆異常大單影響隨時間遞減；② 新增<b>產業板塊合計</b>：依 TWSE 產業別加總，同一分點在同產業有多檔同時「有效買超」即標示族群建倉。</li></ul>'''
    tot = sum(len(v['top_buy']) for v in br.values())
    return save_page(1, 'net_buy_sell', '一', '分點買賣超', '誰在買？誰在賣？從單日到 20 日累積', method, body, js, data, f'今日共 {tot} 筆分點買超、{sum(1 for r in sect if r["t"])} 個族群建倉')

build_s01()

In [ ]:
# ============ S02 持續性主力・分點強度分數 ============
def strength_frame(Pn, Pa):
    n5, n10, n20 = [rs(Pn, n).iloc[-1] for n in (5, 10, 20)]
    cons = consec_pos(Pn).iloc[-1]
    a5 = rs(Pa, 5); amt_inc = (a5.iloc[-1] > a5.shift(5).iloc[-1]) & (a5.iloc[-1] > 0)
    ret5 = RET5.iloc[-1].reindex(Pn.columns)
    df = pd.DataFrame(dict(n5=n5, n10=n10, n20=n20, cons=cons, amt_inc=amt_inc, ret5=ret5,
                           bdays=(Pn.iloc[-20:] > 0).sum(), sdays=(Pn.iloc[-20:] < 0).sum()))
    v5, v10, v20 = VOL5.iloc[-1].reindex(Pn.columns), VOL10.iloc[-1].reindex(Pn.columns), VOL20.iloc[-1].reindex(Pn.columns)
    mr = MATR.iloc[-1].reindex(Pn.columns)                                 # 動態流動性門檻（每檔不同）
    thr = lambda v: np.maximum(mr * v, MAT_MIN_LOTS)
    df['mr'] = mr; df['liq'] = LIQP.iloc[-1].reindex(Pn.columns)
    df['s5'] = 20 * ((df.n5 > 0) & (df.n5 >= thr(v5)))                    # 有效買超才計分
    df['s10'] = 20 * ((df.n10 > 0) & (df.n10 >= thr(v10)))
    df['s20'] = 20 * ((df.n20 > 0) & (df.n20 >= thr(v20)))
    df['scons'] = 15 * df.cons.clip(upper=5) / 5; df['samt'] = 10 * df.amt_inc; df['spx'] = 15 * (df.ret5 > 0)
    df['score'] = df[['s5', 's10', 's20', 'scons', 'samt', 'spx']].sum(axis=1)
    return df[(Pn.iloc[-20:] != 0).any()]

def grade(s):
    return '強' if s >= 80 else '偏強' if s >= 60 else '普通' if s >= 40 else '弱'

def build_s02():
    pairs = []
    for b in BR:
        df = strength_frame(NET[b], AMTN[b])
        df = df[df.index.isin(UNIV)]
        for c, r in df.iterrows():
            pairs.append(dict(branch=b, stk=stk(c), **r.to_dict(), g=grade(r.score), gt=tone(grade(r.score))))
    pairs.sort(key=lambda x: (-x['score'], -x['n20']))
    stock = strength_frame(NET_T, AMTN_T)
    stock = stock[stock.index.isin(UNIV)].sort_values(['score', 'n20'], ascending=False)
    srows = [dict(stk=stk(c), **r.to_dict(), g=grade(r.score), gt=tone(grade(r.score))) for c, r in stock.head(60).iterrows()]
    hist = np.histogram([p['score'] for p in pairs], bins=10, range=(0, 100))[0].tolist()
    brs = [dict(branch=b, n=sum(1 for p in pairs if p['branch'] == b), avg=np.mean([p['score'] for p in pairs if p['branch'] == b] or [np.nan]),
                strong=sum(1 for p in pairs if p['branch'] == b and p['score'] >= 70)) for b in BR]
    data = dict(pairs=pairs[:150], stock=srows, hist=hist, brs=brs)
    body = '''
<div class="kpis" id="k"></div>
<div class="grid"><div class="card"><h2>分點×個股 強度分數分布</h2><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>各分點強勢配對數（≥70 分）</h2><div class="ch"><canvas id="c2"></canvas></div></div></div>
<div class="card"><h2>個股強度排名（追蹤分點合計）</h2><div id="t1"></div></div>
<div class="card"><h2>分點 × 個股 強度排名（前 150）</h2><div id="t2"></div></div>'''
    js = r'''
kpis('k',[['評分配對數',D.pairs.length],['≥80 分配對',D.pairs.filter(p=>p.score>=80).length,'up'],['≥80 分個股',D.stock.filter(p=>p.score>=80).length,'up'],['最高分個股',D.stock.length?D.stock[0].stk:'—']]);
mk('c1',{type:'bar',data:{labels:['0-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-100'],datasets:[{label:'配對數',data:D.hist,backgroundColor:D.hist.map((_,i)=>i>=8?UP:i>=6?AC:MU)}]},options:{plugins:{legend:{display:false}}}});
mk('c2',{type:'bar',data:{labels:D.brs.map(b=>b.branch),datasets:[{label:'≥70 分',data:D.brs.map(b=>b.strong),backgroundColor:UP},{label:'評分配對數',data:D.brs.map(b=>b.n),backgroundColor:'#2a3a4c'}]}});
const cols=[{k:'stk',t:'股票'},{k:'score',t:'強度分數',f:'n',heat:'pos'},{k:'g',t:'等級',f:'tag',tc:'gt'},{k:'n5',t:'5日',f:'n',c:1},{k:'n10',t:'10日',f:'n',c:1},{k:'n20',t:'20日',f:'n',c:1},{k:'cons',t:'連買天數',f:'n'},{k:'bdays',t:'20日買超天數',f:'n'},{k:'sdays',t:'20日賣超天數',f:'n'},{k:'ret5',t:'5日漲跌',f:'p',c:1},{k:'liq',t:'流動性百分位',f:'p',d:0},{k:'mr',t:'有效買超門檻',f:'p',d:2,tip:'淨買超需 ≥ 同期成交量此比例'},
 {k:'s5',t:'+5日',f:'n'},{k:'s10',t:'+10日',f:'n'},{k:'s20',t:'+20日',f:'n'},{k:'scons',t:'+連續',f:'n'},{k:'samt',t:'+金額增',f:'n'},{k:'spx',t:'+股價同漲',f:'n'}];
table('t1',cols,D.stock);table('t2',[{k:'branch',t:'分點'}].concat(cols),D.pairs);'''
    method = f'''不問「今天誰買最多」，而問「最近 20 天，哪些分點持續站在買方」。評分（滿分 100）：
<ul><li>5 日買超 <code>+20</code>、10 日買超 <code>+20</code>、20 日買超 <code>+20</code>（須為<b>有效買超</b>）</li>
<li><b>V1.1 動態流動性門檻</b>：不再固定 0.2%。依個股 60 日均量在普通股中的百分位，於 <code>{MAT_RATIO_ILLIQ:.1%}</code>（流動性最低）～ <code>{MAT_RATIO_LIQUID:.1%}</code>（流動性最高）間線性內插，且至少 <code>{MAT_MIN_LOTS}</code> 張。
大型權值股成交量大，0.1% 已是可觀部位；小型股需 0.4% 才不會被零星單誤觸。此門檻同時套用到全模型的「有效買超」（S09、S11、S14、S15）。</li>
<li>連續買超 <code>+15</code>（連續天數 ÷ 5，最多 5 天滿分）</li><li>近 5 日買超金額 &gt; 前 5 日 <code>+10</code></li><li>股價 5 日同步上漲 <code>+15</code></li></ul>'''
    n80 = int((stock['score'] >= 80).sum())
    return save_page(2, 'branch_strength', '二', '持續性主力・分點強度分數', '從「看資料」變成量化選股因子（0～100）', method, body, js, data, f'{n80} 檔強度 ≥80')

build_s02()


In [ ]:
# ============ S03 主力成本區估算 ============
def wq(v, w, q):
    v, w = np.asarray(v, float), np.asarray(w, float)
    o = np.argsort(v); v, w = v[o], w[o]
    c = np.cumsum(w) / w.sum()
    return float(np.interp(q, c, v))

def cost_status(d):
    if d is None or not np.isfinite(d):
        return None
    return '被套/防守' if d < -0.05 else '成本附近' if d <= 0.03 else '已有獲利' if d <= 0.10 else '獲利了結風險'

def diagnose_cost():
    """S03/S04 資料鏈檢查：每一步剩多少筆，哪一步斷掉一目了然"""
    if len(IDX):
        start = IDX[-min(COST_WINDOW, len(IDX))]
        w = FL[FL['date'] >= start]
    else:
        w = FL.iloc[0:0]
    steps = [
        ('分點資料總筆數', len(FL)),
        ('張數＋金額 兩者皆有', DIAG.get('both_rows')),
        ('有效買進均價（偏離收盤 ≤20%）', DIAG.get('avg_valid')),
        ('均價異常被排除', DIAG.get('avg_outlier')),
        ('淨買超筆數（全期）', DIAG.get('netbuy_rows')),
        (f'近 {COST_WINDOW} 日淨買超筆數', int((w['net_lots'] > 0).sum())),
        (f'近 {COST_WINDOW} 日淨買超且在股票池', int(((w['net_lots'] > 0) & w['code'].isin(UNIV)).sum())),
        (f'近 {COST_WINDOW} 日可估成本', int(((w['posq'] > 0) & w['code'].isin(UNIV)).sum())),
    ]
    print('── S03 成本資料鏈檢查 ──')
    for k, v in steps:
        print(f'  {k:<26} {v if v is not None else "—":>10}')
    if 'same_ratio' in DIAG:
        print(f"  張數=金額 比例 {DIAG['same_ratio']:.1%}｜均價/收盤 中位數 {DIAG['price_ratio']:.3f}（正常約 1.0）｜自動修正：{DIAG.get('fix', '無需修正')}")
    print(f"  實際均價占比 {DIAG.get('real_share', 0):.0%}（其餘以收盤價推估）")

def _inventory(dates, nets, pxs, srcs):
    """逐日庫存法（V1.1）：以「批次」記錄每筆買進的張數、價格、日期。
    賣出扣減：AVG＝各批按比例減碼（原法，成本不變）／FIFO＝先扣最早買進／LIFO＝先扣最近買進；庫存歸零→重置。
    時間衰減：每批的支撐權重 = 0.5^(持有天數 / SUPPORT_HALFLIFE)，套牢越久支撐越弱。"""
    T_end = len(IDX) - 1
    tix = IDX.get_indexer(pd.DatetimeIndex(dates))
    books = {m: [] for m in ('AVG', 'FIFO', 'LIFO')}
    bought, sold, real_q = 0.0, 0.0, 0.0
    bp, bw, bdays = [], [], set()
    for t, n_, p, src in zip(tix, nets, pxs, srcs):
        if n_ > 0 and np.isfinite(p):
            for lots in books.values():
                lots.append([n_, p, t])
            bought += n_; bp.append(p); bw.append(n_); bdays.add(t)
            real_q += n_ if src == '實際均價' else 0
        elif n_ < 0:
            sold += -n_
            for m, lots in books.items():
                tot = sum(l[0] for l in lots)
                if tot <= 0:
                    continue
                q = -n_
                if m == 'AVG':
                    f = max(tot - q, 0) / tot
                    for l in lots: l[0] *= f
                else:
                    it = range(len(lots)) if m == 'FIFO' else range(len(lots) - 1, -1, -1)
                    for i in it:
                        take = min(lots[i][0], q); lots[i][0] -= take; q -= take
                        if q <= 0: break
                # 保留小於 1 張的正庫存；來源換算與跨日扣減可產生小數張，
                # 不應以「小於 1」誤判為歸零。僅移除數值誤差級的殘量。
                lots[:] = [l for l in lots if l[0] > 1e-9]
    if bought <= 0:
        return None
    def summ(lots):
        q = np.array([l[0] for l in lots]); p = np.array([l[1] for l in lots]); age = T_end - np.array([l[2] for l in lots])
        if not len(q) or q.sum() <= 1e-9:
            return dict(pos=0.0, raw=None, cost=None, sup=None, age=None)
        w = q * 0.5 ** (age / SUPPORT_HALFLIFE)
        return dict(pos=float(q.sum()), raw=float((q * p).sum() / q.sum()), cost=float((w * p).sum() / w.sum()),
                    sup=float(w.sum() / q.sum()), age=float((q * age).sum() / q.sum()))
    S = {m: summ(l) for m, l in books.items()}
    k = dict(S[COST_METHOD])
    k.update(avg=S['AVG']['raw'], fifo=S['FIFO']['raw'], lifo=S['LIFO']['raw'], bought=bought, sold=sold, out=min(sold / bought, 9.99),
             lo=wq(bp, bw, 0.25), hi=wq(bp, bw, 0.75), real=real_q / bought, days=len(bdays))
    return k

S03_BODY = '''
<div class="kpis" id="k"></div>
<div id="empty"></div>
<div class="grid"><div class="card"><h2>各分點持倉狀態分布</h2><div class="ch"><canvas id="c0"></canvas></div></div>
<div class="card"><h2>資料品質</h2><div id="dq"></div></div></div>
<div class="card"><div class="row"><h2 style="margin:0">成本 vs 股價</h2><select id="s1"></select></div>
<div class="grid"><div class="ch lg"><canvas id="c1"></canvas></div><div><h3>最近買賣事件（庫存計算依據）</h3><div id="t2"></div><p class="note" id="nt"></p></div></div></div>
<div class="card"><div class="row"><h2 style="margin:0">成本總表</h2><div class="tabs" id="tb"></div></div><div id="t1"></div></div>'''

S03_JS = r"""
const P=D.pairs,S=D.stocks;const cnt=(R,s)=>R.filter(r=>r.st===s).length;
kpis('k',[['有庫存配對',P.filter(r=>r.st!=='已出清').length],['已出清',cnt(P,'已出清')],['被套/防守',cnt(P,'被套/防守'),'dn'],['成本附近',cnt(P,'成本附近')],['已有獲利',cnt(P,'已有獲利'),'up'],['獲利了結風險',cnt(P,'獲利了結風險'),'up'],['實際均價占比',fmtP(D.diag.real_share,0)]]);
if(!P.length){$('empty').innerHTML=`<div class="card" style="border-color:#5a3200"><h2>⚠️ 沒有可估算成本的資料</h2><p class="note">淨買超筆數 ${fmtN(D.diag.netbuy_rows)}；張數金額皆有 ${fmtN(D.diag.both_rows)}；自動修正：${D.diag.fix||'無'}。請把 Colab 中「S03 成本資料鏈檢查」的輸出貼回來。</p></div>`}
const pal={'被套/防守':DN,'成本附近':AC,'已有獲利':UP,'獲利了結風險':'#ff8a65','已出清':'#3b4a5a'};
mk('c0',{type:'bar',data:{labels:D.br,datasets:D.sts.map(s=>({label:s,data:D.by_br[s],backgroundColor:pal[s]}))},options:{indexAxis:'y',scales:{x:ax({stacked:true}),y:ax({stacked:true})}}});
const d=D.diag;table('dq',[{k:'k',t:'項目'},{k:'v',t:'數值'}],[
 {k:'分點資料筆數',v:fmtN(d.rows)},{k:'張數＋金額 皆有',v:fmtN(d.both_rows)},{k:'有效買進均價',v:fmtN(d.avg_valid)},{k:'均價異常排除',v:fmtN(d.avg_outlier)},
 {k:'淨買超筆數',v:fmtN(d.netbuy_rows)},{k:'實際均價占比',v:fmtP(d.real_share,0)},{k:'均價/收盤 中位數',v:d.price_ratio!=null?fmtN(d.price_ratio,3):'—'},{k:'單位自動修正',v:d.fix||'無需修正'}],{h:340,sort:null});
const cols=[{k:'branch',t:'分點'},{k:'stk',t:'股票'},{k:'st',t:'狀態',f:'tag',tc:'stc'},{k:'q20',t:`${D.win}日剩餘庫存(張)`,f:'n'},{k:'b20',t:`${D.win}日買進(張)`,f:'n'},{k:'out20',t:'出清比例',f:'p',d:0},{k:'cost20',t:`${D.win}日支撐成本(衰減)`,f:'f',tip:'依持有天數衰減加權後的成本，狀態判斷用此值'},{k:'raw20',t:`${D.win}日庫存成本(${D.meth})`,f:'f'},{k:'sup',t:'支撐效力',f:'p',d:0,heat:'pos',tip:'剩餘庫存經時間衰減後的有效比例'},{k:'age',t:'平均持有天數',f:'f',d:1},
 {k:'lo',t:'成本區P25',f:'f'},{k:'hi',t:'成本區P75',f:'f'},{k:'px',t:'現價',f:'f'},{k:'d20',t:`現價/${D.win}日成本`,f:'p',c:1,heat:1},
 {k:'qall',t:`${D.lwin}日剩餘庫存`,f:'n'},{k:'costall',t:`${D.lwin}日支撐成本(衰減)`,f:'f'},{k:'cavg',t:`${D.lwin}日 AVG`,f:'f'},{k:'cfifo',t:`${D.lwin}日 FIFO`,f:'f'},{k:'clifo',t:`${D.lwin}日 LIFO`,f:'f'},{k:'dall',t:`現價/${D.lwin}日成本`,f:'p',c:1,heat:1},{k:'n20',t:'20日淨買超',f:'n',c:1},{k:'days',t:'買超天數',f:'n'},{k:'real',t:'實際均價占比',f:'p',d:0},{k:'eng',t:'Score用成本(引擎)',f:'f',tip:'S12～S15 實際使用：全期淨流量、半衰期加權、跨分點合計（僅個股列）'},{k:'deng',t:'現價/引擎成本',f:'p',c:1}];
tabs('tb',['分點 × 個股','個股（追蹤分點合計）'],t=>table('t1',cols,t.startsWith('分點')?P:S));
const opts=Object.keys(D.det).map(k=>{const r=P.concat(S).find(x=>x.key===k)||{};return[k,`${r.branch}｜${r.stk}｜${r.st||'—'}`]});
sel('s1',opts,k=>{const x=D.det[k],line=v=>x.d.map(_=>v);
 const ds=[{type:'line',label:'收盤價',data:x.px,borderColor:BL,pointRadius:0,yAxisID:'y',order:1}];
 if(x.c20!=null)ds.push({type:'line',label:D.win+'日成本',data:line(x.c20),borderColor:AC,borderDash:[6,4],pointRadius:0,yAxisID:'y',order:2});
 if(x.call!=null)ds.push({type:'line',label:D.lwin+'日成本',data:line(x.call),borderColor:'#b07cff',borderDash:[2,3],pointRadius:0,yAxisID:'y',order:3});
 ds.push({type:'line',label:'成本區上緣',data:line(x.hi),borderColor:'rgba(240,180,41,.35)',borderWidth:1,pointRadius:0,yAxisID:'y',fill:false,order:5});
 ds.push({type:'line',label:'成本區(P25~P75)',data:line(x.lo),borderColor:'rgba(240,180,41,.35)',borderWidth:1,backgroundColor:'rgba(240,180,41,.10)',pointRadius:0,yAxisID:'y',fill:'-1',order:6});
 ds.push({type:'bar',label:'淨買超(張)',data:x.net,backgroundColor:x.net.map(v=>v>=0?'rgba(255,77,79,.4)':'rgba(34,197,94,.4)'),yAxisID:'y1',order:9});
 mk('c1',{data:{labels:x.d,datasets:ds},options:{scales:{x:ax(),y:ax({position:'left'}),y1:ax({position:'right',grid:{display:false}})}}});
 table('t2',[{k:'date',t:'日期'},{k:'br',t:'分點'},{k:'q',t:'淨買賣(張)',f:'n',c:1},{k:'pr',t:'買進價(原始)',f:'f'},{k:'p',t:'還原後',f:'f',tip:'依除權息換算到今日價格尺度'},{k:'src',t:'來源'}],x.buys,{h:320,sort:null});
 $('nt').textContent=`庫存法（${D.meth}，支撐半衰期 ${D.hl} 日）：買進加權、賣出依 ${D.meth} 扣減庫存。${D.win}日支撐成本 ${fmtN(x.c20,2)}；${D.lwin}日支撐成本 ${fmtN(x.call,2)}（— 表示已出清）`});"""

def build_s03():
    diagnose_cost()
    s20, s60 = IDX[-min(COST_WINDOW, len(IDX))], IDX[-min(COST_LONG_WINDOW, len(IDX))]
    fw = FL[(FL['date'] >= s60) & FL['code'].isin(UNIV)].sort_values('date')
    last_px = CJ.iloc[-1]   # V1.5：成本為還原價尺度，與還原收盤比較（最新日兩者相同）
    inv = {}
    for (b, c), g in fw.groupby(['branch', 'code'], sort=False):
        g20 = g[g['date'] >= s20]
        k60 = _inventory(g['date'].values, g['net_lots'].values, g['cost_px'].values, g['cost_src'].values)
        k20 = _inventory(g20['date'].values, g20['net_lots'].values, g20['cost_px'].values, g20['cost_src'].values) if len(g20) else None
        if k60:
            inv[(b, c)] = (k20, k60)

    def mkrow(branch, c, k20, k60):
        cur = f_(last_px.get(c))
        d20 = cur / k20['cost'] - 1 if (cur and k20 and k20['cost']) else None
        d60 = cur / k60['cost'] - 1 if (cur and k60 and k60['cost']) else None
        st = cost_status(d20 if d20 is not None else d60) or '已出清'
        base = k20 or k60
        Pn20 = NB20[branch] if branch in NB20 else N20
        return dict(key=f'{branch if branch in NB20 else "合計"}|{c}', branch=branch, code=c, stk=stk(c),
                    q20=k20['pos'] if k20 else 0, b20=k20['bought'] if k20 else 0, out20=k20['out'] if k20 else None,
                    cost20=k20['cost'] if k20 else None, lo=base['lo'], hi=base['hi'],
                    qall=k60['pos'], costall=k60['cost'], px=cur, d20=d20, dall=d60,
                    n20=float(Pn20[c].iloc[-1]), real=base['real'], days=base['days'], st=st, stc=tone(st) if st != '已出清' else 'g',
                    raw20=k20['raw'] if k20 else None, sup=(k20 or k60)['sup'], age=(k20 or k60)['age'],
                    cavg=k60['avg'], cfifo=k60['fifo'], clifo=k60['lifo'])

    pairs = [mkrow(b, c, k20, k60) for (b, c), (k20, k60) in inv.items()]
    # 個股層級：各分點庫存相加，成本以剩餘庫存加權
    agg = {}
    for (b, c), (k20, k60) in inv.items():
        agg.setdefault(c, []).append((k20, k60))
    def merge(ks):
        ks = [k for k in ks if k]
        if not ks:
            return None
        pos = sum(k['pos'] for k in ks); bought = sum(k['bought'] for k in ks)
        wavg = lambda f, w=lambda k: k['pos']: (sum(w(k) * k[f] for k in ks if k[f] is not None) / sum(w(k) for k in ks if k[f] is not None)) if pos > 1e-9 else None
        sw = lambda k: k['pos'] * (k['sup'] or 0)
        return dict(pos=pos, cost=wavg('cost', sw) if sum(sw(k) for k in ks) > 0 else None, raw=wavg('raw'), avg=wavg('avg'), fifo=wavg('fifo'), lifo=wavg('lifo'),
                    sup=wavg('sup'), age=wavg('age'), bought=bought, out=min(sum(k['sold'] for k in ks) / bought, 9.99), sold=sum(k['sold'] for k in ks),
                    lo=min(k['lo'] for k in ks), hi=max(k['hi'] for k in ks),
                    real=sum(k['real'] * k['bought'] for k in ks) / bought, days=max(k['days'] for k in ks))
    stocks = []
    for c, lst in agg.items():
        k60 = merge([x[1] for x in lst]); k20 = merge([x[0] for x in lst])
        if k60:
            _r = mkrow('追蹤分點合計', c, k20, k60)
            _r['eng'] = f_(COST[c].iloc[-1]); _r['deng'] = f_(DIST[c].iloc[-1])   # V1.4：Score／漏斗／回測實際使用的成本
            stocks.append(_r)
    key = lambda r: (r['st'] == '已出清', -r['q20'], -r['qall'])
    pairs.sort(key=key); stocks.sort(key=key)
    pairs, stocks = pairs[:150], stocks[:80]
    det, dd = {}, IDX[-60:]
    for r in pairs[:40] + stocks[:20]:
        b, c = r['key'].split('|')
        g = fw[fw['code'] == c]
        if b != '合計':
            g = g[g['branch'] == b]
        Pn = NET[b] if b != '合計' else NET_T
        g = g.sort_values('date', ascending=False).head(40)
        det[r['key']] = dict(d=[x.strftime('%m/%d') for x in dd], px=CJ[c].reindex(dd).tolist(), net=Pn[c].reindex(dd).tolist(),
                             c20=r['cost20'], call=r['costall'], lo=r['lo'], hi=r['hi'],
                             buys=[dict(date=x.strftime('%Y-%m-%d'), br=y, q=z, p=(w if z > 0 else None), pr=(u if z > 0 else None), src=(v if z > 0 else '賣出扣庫存'))
                                   for x, y, z, w, u, v in zip(g['date'], g['branch'], g['net_lots'], g['cost_px'], g['cost_raw'], g['cost_src'])])
    sts = ['被套/防守', '成本附近', '已有獲利', '獲利了結風險', '已出清']
    by_br = {st: [sum(1 for r in pairs if r['branch'] == b and r['st'] == st) for b in BR] for st in sts}
    data = dict(meth=COST_METHOD, hl=SUPPORT_HALFLIFE, pairs=pairs, stocks=stocks, det=det, win=COST_WINDOW, lwin=COST_LONG_WINDOW, br=BR, sts=sts, by_br=by_br, diag=DIAG)
    method = f"""<ul><li><b>庫存法</b>：對每個「分點 × 個股」逐日處理——淨買超日以當日<b>買進均價</b>加入庫存並重算加權平均成本；淨賣超日<b>扣減庫存</b>（成本不變）；庫存歸零則成本重置。分別從近 <code>{COST_WINDOW}</code> 日與近 <code>{COST_LONG_WINDOW}</code> 日起算。</li>
<li>「剩餘庫存」才是仍可能護盤的部位；剩餘庫存歸零才標「已出清」，不再視為成本支撐。「出清比例」= 期間賣出 ÷ 期間買進，僅供參考（Top50 截尾下賣出可能被低估）。</li>
<li><b>V1.4 成本對照</b>：本頁是「近 {COST_WINDOW}／{COST_LONG_WINDOW} 日批次庫存」；S12～S15 的 Score、漏斗與回測使用因子引擎的「全期淨流量庫存」（半衰期 {COST_HALFLIFE} 日）。兩者半衰期已統一，個股列另列「Score用成本(引擎)」欄供對照，差異主要來自起算視窗。</li>
<li><b>V1.1 優化</b>：① 賣出扣減法可選 <code>COST_METHOD</code> = AVG（按比例減碼，原法）／FIFO（先買先賣：較早、通常較低的成本先出場）／LIFO（後買先賣：最近追高的部位先出場）；目前使用 <code>{COST_METHOD}</code>，表格同時列出三種方法的 {COST_LONG_WINDOW} 日成本供對照。
② <b>時間衰減</b>：每批庫存依持有天數給予支撐權重 0.5^(天數 / {SUPPORT_HALFLIFE})，得到「支撐成本」與「支撐效力」；套牢越久的部位支撐越弱，狀態判斷改用支撐成本。</li>
<li>缺少買進均價時（只出現在其中一個排行、或偏離收盤 &gt;20%），改用當日收盤價推估並在「來源」欄標示。成本區 = 期間買進價的加權 P25～P75。</li>
<li>狀態依現價與庫存成本距離：&lt; −5% 被套/防守；−5%～+3% 成本附近；+3%～+10% 已有獲利；&gt; +10% 獲利了結風險（20 日內無庫存時改用 {COST_LONG_WINDOW} 日）。</li></ul>"""
    n = sum(1 for r in pairs if r['st'] == '獲利了結風險')
    return save_page(3, 'cost_zone', '三', '主力成本區估算', '庫存法估算分點持倉成本（含賣出扣減），判斷被套、成本附近或獲利了結風險', method, S03_BODY, S03_JS, data,
                     f'{sum(1 for r in pairs if r["st"] != "已出清")} 組有庫存、{n} 組獲利了結風險' if pairs else '⚠️ 無資料，見頁面診斷')

build_s03()

In [ ]:
# ============ S04 主力成本密集區 ============
def dense_zone(prices, weights, mass=0.60):
    """V1.1：加權高斯核密度估計（KDE）取代固定 14 格直方圖。
    頻寬用 Silverman 法則（有效樣本數 + 穩健標準差），從密度最高峰向兩側擴張；
    區內占比以原始買進張數加權計算，確保「涵蓋 60% 買進張數」不是 KDE 網格近似值。"""
    p, w = np.asarray(prices, float), np.asarray(weights, float)
    ok = np.isfinite(p) & np.isfinite(w) & (w > 0); p, w = p[ok], w[ok]
    W = w.sum()
    if p.size == 0 or not np.isfinite(W) or W <= 0:
        raise ValueError('dense_zone 需要至少一筆有限且權重為正的價格資料')
    mass = float(np.clip(mass, 0.0, 1.0))
    mu = (w * p).sum() / W
    sd = np.sqrt((w * (p - mu) ** 2).sum() / W)
    iqr = wq(p, w, 0.75) - wq(p, w, 0.25)
    sig = min(sd, iqr / 1.34) if iqr > 0 else sd
    neff = W ** 2 / (w ** 2).sum()
    bw = max(1.06 * (sig if sig > 0 else max(mu * 0.01, 1e-6)) * neff ** (-0.2) * KDE_BW_ADJ, mu * 0.002)
    lo, hi = wq(p, w, 0.01) - 3 * bw, wq(p, w, 0.99) + 3 * bw
    # 確保原始觀測落在網格內，讓直接張數涵蓋率可達到要求值。
    lo, hi = min(lo, float(p.min())), max(hi, float(p.max()))
    x = np.linspace(lo, hi, KDE_GRID)
    dens = (w[None, :] * np.exp(-0.5 * ((x[:, None] - p[None, :]) / bw) ** 2)).sum(axis=1) / (W * bw * np.sqrt(2 * np.pi))
    m = dens / dens.sum()
    pk = int(dens.argmax()); Lb = Rb = pk
    # 原始買進張數涵蓋率，而非把 KDE 網格密度當成張數。
    acc = float(w[(p >= x[Lb]) & (p <= x[Rb])].sum() / W)
    while acc < mass and (Lb > 0 or Rb < len(x) - 1):
        left = m[Lb - 1] if Lb > 0 else -1
        right = m[Rb + 1] if Rb < len(x) - 1 else -1
        if right >= left: Rb += 1
        else: Lb -= 1
        acc = float(w[(p >= x[Lb]) & (p <= x[Rb])].sum() / W)
    # 波峰數：密度局部極大且 ≥ 最高峰 25%（多峰代表有多個成本群）
    peaks = int(((dens[1:-1] > dens[:-2]) & (dens[1:-1] >= dens[2:]) & (dens[1:-1] >= 0.25 * dens.max())).sum())
    return x, dens, x[Lb], x[Rb], acc, x[pk], bw, max(peaks, 1)

def build_s04():
    ev = FL[(FL['posq'] > 0) & FL['code'].isin(UNIV) & (FL['date'] >= IDX[-min(DENSITY_WINDOW, len(IDX))])]
    tot = ev.groupby('code')['posq'].sum().sort_values(ascending=False)
    rows, det = [], {}
    for c in tot.index[:40]:
        g = ev[ev['code'] == c]
        if len(g) < 3:
            continue
        x, dens, zl, zh, share, peak, bw, npk = dense_zone(g['cost_px'], g['posq'])
        cur = CJ[c].iloc[-1]
        if not np.isfinite(cur):   # V1.5：最新日無股價（停牌）不判定狀態
            continue
        if cur < zl: st = '低於成本區'
        elif cur > zh: st = '突破成本區'
        elif cur >= zl + 0.8 * (zh - zl): st = '接近成本上緣'
        else: st = '成本區內'
        cs = CJ[c].reindex(IDX[-20:])
        above = cs > zh
        bdays, bret = None, None
        if st == '突破成本區':
            k = len(above) - 1
            while k > 0 and above.iloc[k - 1]:
                k -= 1
            bdays = len(above) - k
            bret = cur / cs.iloc[k] - 1
        rows.append(dict(code=c, stk=stk(c), q=tot[c], zl=zl, zh=zh, share=share, px=cur, pos=(cur - zl) / (zh - zl) if zh > zl else None,
                         st=st, stc=tone(st), bdays=bdays, bret=bret, peak=peak, bw=bw, npk=npk))
        dd = IDX[-60:]
        step = max(1, len(x) // 120)                      # 前端只送 ~120 點，檔案不會太大
        xs, ds = x[::step], dens[::step]
        det[c] = dict(lab=[f'{v:.2f}' for v in xs], h=(ds / ds.max()).round(4).tolist(),
                      inz=[bool(zl - 1e-9 <= v <= zh + 1e-9) for v in xs],
                      cur=int(np.abs(xs - cur).argmin()), d=[x.strftime('%m/%d') for x in dd], px=CJ[c].reindex(dd).tolist(), zl=zl, zh=zh)
    data = dict(rows=rows, det=det)
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><div class="row"><h2 style="margin:0">成本分布</h2><select id="s1"></select></div>
<div class="grid"><div><h3>買進張數核密度（KDE，金色＝成本密集區，紅點＝現價）</h3><div class="ch lg"><canvas id="c1"></canvas></div></div>
<div><h3>股價與成本密集區</h3><div class="ch lg"><canvas id="c2"></canvas></div></div></div></div>
<div class="card"><h2>成本密集區總表</h2><div id="t1"></div></div>'''
    js = r'''
const R=D.rows;const n=s=>R.filter(r=>r.st===s).length;
kpis('k',[['分析個股',R.length],['突破成本區',n('突破成本區'),'up'],['接近成本上緣',n('接近成本上緣')],['成本區內',n('成本區內')],['低於成本區',n('低於成本區'),'dn']]);
table('t1',[{k:'stk',t:'股票'},{k:'q',t:'累積淨買(張)',f:'n'},{k:'zl',t:'密集區下緣',f:'f'},{k:'zh',t:'密集區上緣',f:'f'},{k:'share',t:'區內占比',f:'p'},{k:'peak',t:'密度峰值價',f:'f'},{k:'npk',t:'波峰數',f:'n',tip:'≥2 表示有多個成本群'},{k:'bw',t:'KDE頻寬',f:'f'},{k:'px',t:'現價',f:'f'},{k:'pos',t:'區間位置',f:'p',tip:'0%=下緣 100%=上緣'},{k:'st',t:'狀態',f:'tag',tc:'stc'},{k:'bdays',t:'突破天數',f:'n'},{k:'bret',t:'突破後漲跌',f:'p',c:1}],R);
sel('s1',R.map(r=>[r.code,r.stk+'｜'+r.st]),c=>{const x=D.det[c];
 mk('c1',{type:'line',data:{labels:x.lab,datasets:[
  {label:'成本密集區',data:x.h.map((v,i)=>x.inz[i]?v:null),borderColor:AC,backgroundColor:'rgba(240,180,41,.25)',fill:'origin',pointRadius:0,spanGaps:false},
  {label:'買進密度',data:x.h,borderColor:'#7d8b99',pointRadius:0,borderWidth:1.5},
  {label:'現價',data:x.h.map((v,i)=>i===x.cur?v:null),borderColor:UP,backgroundColor:UP,pointRadius:6,showLine:false}]},
  options:{plugins:{legend:{display:false}},scales:{x:ax({title:{display:true,text:'價位',color:MU}}),y:ax({title:{display:true,text:'相對密度',color:MU}})}}});
 mk('c2',{type:'line',data:{labels:x.d,datasets:[{label:'密集區上緣',data:x.d.map(_=>x.zh),borderColor:AC,borderDash:[5,4],pointRadius:0,fill:false},
  {label:'密集區下緣',data:x.d.map(_=>x.zl),borderColor:AC,borderDash:[5,4],pointRadius:0,fill:'-1',backgroundColor:'rgba(240,180,41,.12)'},
  {label:'收盤價',data:x.px,borderColor:BL,pointRadius:0}]}})});'''
    method = f'''<ul><li>把近 {DENSITY_WINDOW} 個交易日追蹤分點的「淨買超日」買進均價，以淨買超張數為權重。</li>
<li><b>V1.1 優化：核密度估計（KDE）</b>取代固定 14 格直方圖。每筆買進價視為一個高斯核，頻寬依 Silverman 法則自動調整（有效樣本數、穩健標準差，<code>KDE_BW_ADJ</code> 可微調），
高波動股與低波動股都能得到平滑且不失真的成本分布；同時標出「波峰數」，≥2 代表存在多個成本群。</li>
<li>從密度最高峰向兩側擴張，直到涵蓋 60% 買進張數 → 定義為「主力成本密集區」。</li>
<li>狀態：低於成本區／成本區內／接近成本上緣（區間 80% 以上）／突破成本區；突破者追蹤突破天數與突破後漲跌，研究成本區突破後的延續性。</li></ul>'''
    return save_page(4, 'cost_density', '四', '主力成本密集區', '成本分布直方圖與成本區突破延續性', method, body, js, data, f'{sum(1 for r in rows if r["st"] == "突破成本區")} 檔突破成本區')

build_s04()

In [ ]:
# ============ S05 隔日沖券商 ============
def daytrade_stats(window=None):
    """window=None 用全期；給天數則只看最近 window 個「已完成相應出清窗口」的事件日。"""
    pair_rows, brs, alerts, timeline = [], [], [], {}
    KH = max(3, DT_HORIZON)
    # window 只限制事件日；額外保留 KH 日，讓最近事件仍有完整的 T+1～T+KH 結果可計算。
    sl = slice(-(window + KH), None) if window else slice(None)
    for b in BR:
        NETb, BUYb, SELLb = NET[b].iloc[sl], BUY[b].iloc[sl], SELL[b].iloc[sl]
        # 沒有正的買進分母時，比例沒有定義，不列為有效事件。
        ev = (NETb >= DT_MIN_LOTS) & (BUYb > 0)
        valid = {}
        for k_ in range(1, KH + 1):
            m = ev.copy()
            # 整合修正：日期存在不等於數值完整，任一未來分點日缺漏就不算已完成事件。
            for future_day in range(1, k_ + 1):
                m &= NETb.shift(-future_day).notna() & BUYb.shift(-future_day).notna() & SELLb.shift(-future_day).notna()
            # 事件必須有完整的 T+1～T+k 未來資料。
            m.iloc[-k_:] = False
            if window:
                # 對每個 horizon，只保留最近 window 個已完成事件日；尾端 KH 日只是 look-ahead。
                scope = pd.DataFrame(False, index=m.index, columns=m.columns)
                end_i = len(m) - k_
                start_i = max(0, end_i - window)
                if end_i > start_i:
                    scope.iloc[start_i:end_i] = True
                m &= scope
            valid[k_] = m

        nxt = SELLb.shift(-1)
        R = (nxt / BUYb.replace(0, np.nan)).where(valid[1]).clip(upper=2)
        RK = {}                                                # T+1～T+k 日內累積出清比例
        for k_ in range(1, KH + 1):          # k 日內淨賣回比例：max(0, −Σ淨買超(t+1..t+k)) ÷ 事件日淨買超
            sk = (-sum(NETb.shift(-j) for j in range(1, k_ + 1))).clip(lower=0)
            RK[k_] = (sk / NETb.where(NETb > 0)).where(valid[k_]).clip(upper=2)
        R2, R3 = RK[2], RK[3]
        evh, ev2, ev3 = valid[1], valid[2], valid[3]
        tot_buy = BUYb.where(evh).sum().sum(); tot_sell = nxt.where(evh).sum().sum()
        n_ev = int(evh.values.sum()); n_ev2 = int(ev2.values.sum()); n_ev3 = int(ev3.values.sum())
        brs.append(dict(branch=b, events=n_ev, events2=n_ev2, events3=n_ev3,
                        ratio=tot_sell / tot_buy if tot_buy else None,
                        ratio2=float(np.nanmean(R2.values)) if np.isfinite(R2.values).any() else None,
                        ratio3=float(np.nanmean(R3.values)) if np.isfinite(R3.values).any() else None,
                        hit=int((R >= DT_ALERT_RATIO).values.sum()),
                        hitr=(R >= DT_ALERT_RATIO).values.sum() / n_ev if n_ev else None))
        daily = (nxt.where(evh).sum(axis=1) / BUYb.where(evh).sum(axis=1).replace(0, np.nan))
        timeline[b] = daily.iloc[:-1].iloc[-120:].tolist()
        cnt, cnt2, cnt3 = evh.sum(), ev2.sum(), ev3.sum()
        mr, mr2, mr3, h90 = R.mean(), R2.mean(), R3.mean(), (R >= DT_ALERT_RATIO).sum()
        def prof(c):   # 多維短線客特徵：哪一個天期內出清；各天期至少要有 2 個完整事件
            a1, a2, a3 = mr.get(c, np.nan), mr2.get(c, np.nan), mr3.get(c, np.nan)
            if np.isfinite(a1) and a1 >= DT_PAIR_RATIO: return '隔日沖'
            if cnt2.get(c, 0) >= 2 and np.isfinite(a2) and a2 >= DT_CLEAR_RATIO: return '隔兩日沖'
            if cnt3.get(c, 0) >= 2 and np.isfinite(a3) and a3 >= DT_CLEAR_RATIO: return '三日短波段'
            return '非短沖'
        PROF = {c: prof(c) for c in cnt[cnt >= 2].index}
        for c in cnt[cnt >= 2].index:
            if c not in UNIV:
                continue
            pt = PROF[c]
            pair_rows.append(dict(branch=b, code=c, stk=stk(c), ev=int(cnt[c]), ev2=int(cnt2.get(c, 0)), ev3=int(cnt3.get(c, 0)),
                                  mr=mr[c], mr2=mr2.get(c), mr3=mr3.get(c), pt=pt, ptc=tone(pt),
                                  h90=int(h90[c]), hr=h90[c] / cnt[c],
                                  lb=BUYb[c].where(evh[c]).sum()))
        # 警示 1：今日大買 + 歷史隔日沖比例高
        today = NETb.iloc[-1]
        for c in today[today >= DT_MIN_LOTS].index:
            if cnt.get(c, 0) >= 2 and mr.get(c, 0) >= DT_PAIR_RATIO and c in UNIV:
                alerts.append(dict(type='明日賣壓警示', tc='n', branch=b, stk=stk(c), q=today[c], mr=mr[c], ev=int(cnt[c]), note=f'今日買 {today[c]:,.0f} 張，歷史平均隔日賣出 {mr[c]:.0%}'))
            elif cnt.get(c, 0) >= 2 and PROF.get(c) in ('隔兩日沖', '三日短波段') and c in UNIV:
                kk = 2 if PROF[c] == '隔兩日沖' else 3
                rr = (mr2 if kk == 2 else mr3)[c]
                alerts.append(dict(type=f'{kk}日內賣壓警示', tc='n', branch=b, stk=stk(c), q=today[c], mr=rr, ev=int(cnt[c]), note=f'{PROF[c]}型：今日買 {today[c]:,.0f} 張，歷史 {kk} 日內平均出清 {rr:.0%}'))
        # 警示 2：昨日大買、今日賣出 ≥ 90%
        if len(IDX) >= 2:
            yb = BUYb.iloc[-2]; ys = SELLb.iloc[-1]; yev = (NETb.iloc[-2] >= DT_MIN_LOTS) & (yb > 0)
            for c in yev[yev].index:
                if ys[c] / yb[c] >= DT_ALERT_RATIO and c in UNIV:
                    alerts.append(dict(type='隔日沖確認', tc='d', branch=b, stk=stk(c), q=yb[c], mr=ys[c] / yb[c], ev=int(cnt.get(c, 0)), note=f'昨買 {yb[c]:,.0f} 張，今賣 {ys[c]:,.0f} 張'))
        # 警示 3：前日大量淨買、昨今兩日合計淨賣回 ≥ 門檻（且不是隔日就賣完）→ 隔兩日沖確認
        if len(IDX) >= 3:
            n0 = NETb.iloc[-3]; b0 = BUYb.iloc[-3]; e2 = (n0 >= DT_MIN_LOTS) & (b0 > 0); r1_, r2_ = -NETb.iloc[-2], -NETb.iloc[-1]
            for c in e2[e2].index:
                if r1_[c] < DT_ALERT_RATIO * n0[c] and r1_[c] + r2_[c] >= DT_ALERT_RATIO * n0[c] and c in UNIV:
                    alerts.append(dict(type='隔兩日沖確認', tc='d', branch=b, stk=stk(c), q=n0[c], mr=(r1_[c] + r2_[c]) / n0[c], ev=int(cnt.get(c, 0)),
                                       note=f'前日淨買 {n0[c]:,.0f} 張，昨今共淨賣 {r1_[c] + r2_[c]:,.0f} 張'))
    return pair_rows, brs, alerts, timeline

def build_s05():
    pairs, brs, alerts, tl = daytrade_stats()
    pairs.sort(key=lambda r: (-r['mr'], -r['ev']))
    data = dict(pairs=pairs[:120], brs=brs, alerts=alerts, tl=tl, d=[x.strftime('%Y/%m/%d') for x in IDX[:-1][-120:]], th=DT_ALERT_RATIO, mn=DT_MIN_LOTS)
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>⚠️ 隔日沖警報</h2><div id="t0"></div></div>
<div class="grid"><div class="card"><h2>各分點 T+1 隔日沖比例／2、3 日內淨賣回比例（全期）</h2><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>近 120 日逐日隔日沖比例（隔日賣出 ÷ 前日買進）</h2><div class="ch"><canvas id="c2"></canvas></div></div></div>
<div class="card"><h2>分點 × 個股 隔日沖比例（事件 ≥ 2 次）</h2><div id="t1"></div></div>'''
    js = r'''
kpis('k',[['隔日沖事件總數',fmtN(D.brs.reduce((a,b)=>a+b.events,0))],['比例 ≥ '+fmtP(D.th,0)+' 次數',fmtN(D.brs.reduce((a,b)=>a+b.hit,0)),'up'],
 ['今日警報',D.alerts.length,D.alerts.length?'up':''],['隔兩日沖配對',D.pairs.filter(p=>p.pt==='隔兩日沖').length],['三日短波段配對',D.pairs.filter(p=>p.pt==='三日短波段').length],['最像隔日沖的分點',(D.brs.filter(b=>b.ratio!=null).sort((a,b)=>b.ratio-a.ratio)[0]||{}).branch||'—']]);
table('t0',[{k:'type',t:'類型',f:'tag',tc:'tc'},{k:'branch',t:'分點'},{k:'stk',t:'股票'},{k:'q',t:'張數',f:'n'},{k:'mr',t:'比例',f:'p'},{k:'ev',t:'歷史事件數',f:'n'},{k:'note',t:'說明'}],D.alerts,{h:320});
mk('c1',{type:'bar',data:{labels:D.brs.map(b=>b.branch),datasets:[{label:'T+1 隔日沖比例',data:D.brs.map(b=>b.ratio),backgroundColor:UP},{label:'2 日內淨賣回',data:D.brs.map(b=>b.ratio2),backgroundColor:AC},{label:'3 日內淨賣回',data:D.brs.map(b=>b.ratio3),backgroundColor:BL}]},
 options:{scales:{x:ax(),y:ax({ticks:{color:MU,callback:v=>fmtP(v,0)}})}}});
const pal=[UP,AC,BL,DN,'#b07cff'];
mk('c2',{type:'line',data:{labels:D.d,datasets:Object.keys(D.tl).map((b,i)=>({label:b,data:D.tl[b],borderColor:pal[i%5],pointRadius:0,spanGaps:true,borderWidth:1.5}))},
 options:{scales:{x:ax(),y:ax({ticks:{color:MU,callback:v=>fmtP(v,0)}})}}});
table('t1',[{k:'branch',t:'分點'},{k:'stk',t:'股票'},{k:'ev',t:'事件數',f:'n'},{k:'lb',t:'事件日買進合計(張)',f:'n'},{k:'mr',t:'平均隔日沖比例',f:'p',heat:'pos'},{k:'mr2',t:'2日內淨賣回',f:'p',heat:'pos'},{k:'mr3',t:'3日內淨賣回',f:'p',heat:'pos'},{k:'pt',t:'短線客類型',f:'tag',tc:'ptc'},{k:'h90',t:'≥門檻次數',f:'n'},{k:'hr',t:'≥門檻占比',f:'p'}],D.pairs);'''
    method = f'''<ul><li>事件：某分點某檔股票單日淨買超 ≥ <code>{DT_MIN_LOTS}</code> 張。</li>
<li>隔日沖比例 = 隔日賣出張數 ÷ 前一日買進張數（上限 200%）。分點整體比例 = Σ隔日賣出 ÷ Σ事件日買進。</li>
<li><b>V1.1 多維短線客特徵庫</b>：另計 2 日、3 日內<b>淨賣回比例</b> = max(0, −Σ T+1～T+k 淨買超) ÷ T 日淨買超（用淨額而非賣出張數，避免每天雙向大量進出的活躍分點被誤判），每個「分點 × 個股」配對依序判定：
T+1 平均 ≥ {DT_PAIR_RATIO:.0%} → <b>隔日沖</b>；2 日內 ≥ {DT_CLEAR_RATIO:.0%} → <b>隔兩日沖</b>；3 日內 ≥ {DT_CLEAR_RATIO:.0%} → <b>三日短波段</b>；否則非短沖。
躲避隔日沖追蹤、改成 T+2／T+3 出場的短線客也會被抓出來，並發出「2／3 日內賣壓警示」與「隔兩日沖確認」。</li>
<li>全模型的隔日沖扣分與排除（S06、S12～S15）改用 {DT_HORIZON} 日內出清比例。</li>
<li>比例 ≥ <code>{DT_ALERT_RATIO:.0%}</code> 視為高度隔日沖；警報分「明日賣壓警示」（今日大買且該配對歷史平均 ≥ {DT_PAIR_RATIO:.0%}）與「隔日沖確認」（昨日大買、今日賣出 ≥ 門檻）。</li>
<li>這類買超不能解讀成長線布局。</li></ul>'''
    return save_page(5, 'day_trade', '五', '隔日沖券商辨識', '今日大量買進 → 隔日大量賣出', method, body, js, data, f'{len(alerts)} 則今日警報')

build_s05()

In [ ]:
# ============ S06 長線／短線主力類型分類 ============
def pos_runs(a):
    out, cur = [], 0
    for x in a:
        if x > 0:
            cur += 1
        else:
            if cur: out.append(cur)
            cur = 0
    if cur: out.append(cur)
    return out

def kmeans_np(X, k, n_init=10, iters=100, seed=7):
    """純 numpy 的 K-Means++（不依賴 sklearn）"""
    rng = np.random.default_rng(seed); best = (np.inf, None, None)
    for _ in range(n_init):
        cen = [X[rng.integers(len(X))]]
        for _ in range(1, k):
            d2 = np.min(((X[:, None, :] - np.array(cen)[None]) ** 2).sum(-1), axis=1)
            cen.append(X[rng.choice(len(X), p=d2 / d2.sum())] if d2.sum() > 0 else X[rng.integers(len(X))])
        cen = np.array(cen)
        for _ in range(iters):
            lab = ((X[:, None, :] - cen[None]) ** 2).sum(-1).argmin(1)
            new = np.array([X[lab == j].mean(0) if (lab == j).any() else cen[j] for j in range(k)])
            if np.allclose(new, cen): break
            cen = new
        # 以最終中心重新分派標籤，避免達到迭代上限時 lab 仍對應更新前的中心
        lab = ((X[:, None, :] - cen[None]) ** 2).sum(-1).argmin(1)
        inertia = ((X - cen[lab]) ** 2).sum()
        if inertia < best[0]: best = (inertia, lab, cen)
    return best[1], best[2]

CL_FEATS = [('lturn', '周轉(log)'), ('act', '20日活躍度'), ('br', '買超日占比'), ('run', '平均連買天數'), ('flips', '方向翻轉'),
            ('dtr', 'T+1出清'), ('dtr3', '3日內淨賣回'), ('nr', '20日淨買/總量'), ('nlr', '60日淨買/總量')]

def name_cluster(m):
    if m['dtr'] >= 0.6 or m['dtr3'] >= 0.75: return '短沖群'
    if m['lturn'] >= np.log1p(6): return '高周轉群'
    if m['nr'] <= -0.2: return '賣方群'
    if m['br'] >= 0.65 and m['nr'] > 0.3: return '趨勢累積群'
    if m['nr'] > 0.1: return '波段買方群'
    return '混合/雜訊群'

def build_s06():
    pairs_dt, _, _, _ = daytrade_stats(window=LBN)     # 用近 60 日判斷「目前」是否為隔日沖，與分類視窗一致
    dtm = {(p['branch'], p['code']): p for p in pairs_dt}
    rows = []
    # 歷史「跟單」績效：該分點對該股出現有效買超後，未來 10 日同日超額報酬（全部為已實現的歷史事件）
    _R10 = FWD[10][UNIV]; XR10 = _R10.sub(_R10.mean(axis=1), axis=0).reindex(columns=CODES)
    for b in BR:
        P = NET[b]; Bq = BUY[b]; Sq = SELL[b]
        E = (P >= np.maximum(MATR * VOLD.fillna(0), MAT_MIN_LOTS)) & XR10.notna()
        fw_ex, fw_win, fw_n = XR10.where(E).mean(), (XR10 > 0).where(E).mean(), E.sum()
        # V1.4：拆成前／後半段，兩段都正才算穩定優勢（避免用全期績效在樣本內挑出「最佳群」）
        _early = pd.Series(IDX < IDX[len(IDX) // 2], index=IDX)
        E1, E2 = E & _early.values[:, None], E & ~_early.values[:, None]
        fx1, fn1 = XR10.where(E1).mean(), E1.sum()
        fx2, fn2 = XR10.where(E2).mean(), E2.sum()
        act = (P.iloc[-LBN:] != 0).sum()          # 只分類近 60 日仍有交易的配對（反映目前行為）
        for c in act[act >= 3].index:
            if c not in UNIV:
                continue
            a = P[c].values
            n5, n10, n20, nlb = a[-5:].sum(), a[-10:].sum(), a[-20:].sum(), a[-LBN:].sum()
            gross20 = (Bq[c].values[-20:] + Sq[c].values[-20:]).sum()
            turn = gross20 / abs(n20) if n20 else np.inf
            active20 = (a[-20:] != 0).mean()
            act30 = a[-30:][a[-30:] != 0]
            bratio = (act30 > 0).mean() if len(act30) else 0
            runs = pos_runs(a[-LBN:]); avg_run = np.mean(runs) if runs else 0
            sg = np.sign(a[-20:]); sg = sg[sg != 0]; flips = int((sg[1:] != sg[:-1]).sum()) if len(sg) > 1 else 0
            d = dtm.get((b, c)); dtr = d['mr'] if d else None; nev = d['ev'] if d else 0
            dtr3 = d['mr3'] if d else None
            gross60 = (Bq[c].values[-LBN:] + Sq[c].values[-LBN:]).sum()
            if nev >= 2 and dtr is not None and dtr >= DT_PAIR_RATIO: t = '隔日沖'
            elif active20 >= 0.5 and turn >= 6: t = '高周轉'
            elif n20 > 0 and nlb > 0 and bratio >= 0.65 and len(act30) >= 10: t = '趨勢主力'
            elif n5 > 0 and n10 > 0 and n20 > 0: t = '波段主力'
            elif 2 <= avg_run <= 5 and flips >= 3: t = '短線主力'
            elif n20 < 0 and n10 < 0: t = '持續賣方'
            else: t = '不明顯'
            rows.append(dict(branch=b, stk=stk(c), t=t, tc=tone(t), n5=n5, n10=n10, n20=n20, nlb=nlb, dtr=dtr, turn=turn if np.isfinite(turn) else None,
                             act=active20, br=bratio, run=avg_run, flips=flips, dtr3=dtr3,
                             lturn=float(np.log1p(min(turn, 50))) if np.isfinite(turn) else np.log1p(50),
                             nr=n20 / gross20 if gross20 else 0.0, nlr=nlb / gross60 if gross60 else 0.0,
                             fex=f_(fw_ex[c]), fwin=f_(fw_win[c]), fn=int(fw_n[c]),
                             fex1=f_(fx1[c]), fn1=int(fn1[c]), fex2=f_(fx2[c]), fn2=int(fn2[c])))
    # ---- V1.1：K-Means 機器分群（標準化特徵），以歷史跟單超額報酬評估各群 ----
    clusters = []
    if len(rows) >= N_CLUSTERS * 5:
        X = np.array([[0.0 if r[f] is None or not np.isfinite(r[f]) else r[f] for f, _ in CL_FEATS] for r in rows])
        Z = (X - X.mean(0)) / np.where(X.std(0) > 0, X.std(0), 1)
        lab, _ = kmeans_np(Z, N_CLUSTERS)
        names, used = {}, Counter()
        for j in range(N_CLUSTERS):
            idx = np.where(lab == j)[0]
            if not len(idx): continue
            m = dict(zip([f for f, _ in CL_FEATS], X[idx].mean(0)))
            nm = name_cluster(m); used[nm] += 1
            nm = nm if used[nm] == 1 else f'{nm}{chr(64 + used[nm])}'
            names[j] = nm
            ev = [rows[i] for i in idx if rows[i]['fn'] > 0 and rows[i]['fex'] is not None]
            wn = sum(r['fn'] for r in ev)
            _wavg = lambda fk, nk: (lambda e_, w_: sum(r[fk] * r[nk] for r in e_) / w_ if w_ else None)(
                [rows[i] for i in idx if rows[i][nk] > 0 and rows[i][fk] is not None],
                sum(rows[i][nk] for i in idx if rows[i][nk] > 0 and rows[i][fk] is not None))
            fe1, fe2 = _wavg('fex1', 'fn1'), _wavg('fex2', 'fn2')
            clusters.append(dict(k=nm, n=len(idx), fex=sum(r['fex'] * r['fn'] for r in ev) / wn if wn else None,
                                 fwin=sum(r['fwin'] * r['fn'] for r in ev) / wn if wn else None, fn=wn,
                                 fex1=fe1, fex2=fe2, stab=('前後皆正' if (fe1 or 0) > 0 and (fe2 or 0) > 0 else '前後皆負' if (fe1 or 0) < 0 and (fe2 or 0) < 0 else '不一致'),
                                 **{f: float(m[f]) for f, _ in CL_FEATS}))
        for i, r in enumerate(rows):
            r['k'] = names.get(lab[i])
        clusters.sort(key=lambda c_: -(c_['fex'] if c_['fex'] is not None else -9))
        for i, c_ in enumerate(clusters):
            c_['rk'] = i + 1
            c_['kc'] = 'u' if (c_['fex'] or 0) > 0 and c_['stab'] == '前後皆正' else ('d' if (c_['fex'] or 0) < 0 else 'n')
        kc = {c_['k']: c_['kc'] for c_ in clusters}
        for r in rows:
            r['kc'] = kc.get(r.get('k'), 'g')
    else:
        for r in rows:
            r['k'], r['kc'] = None, 'g'
    rows.sort(key=lambda r: -abs(r['n20']))
    rows = rows[:1200]
    types = ['波段主力', '趨勢主力', '短線主力', '隔日沖', '高周轉', '持續賣方', '不明顯']
    mat = {t: [sum(1 for r in rows if r['branch'] == b and r['t'] == t) for b in BR] for t in types}
    # 個股層級的「籌碼集中」：單一分點占追蹤分點 20 日成交的比重
    knames = [c_['k'] for c_ in clusters]
    cross = [dict(t=t, **{k_: sum(1 for r in rows if r['t'] == t and r.get('k') == k_) for k_ in knames}) for t in types]
    data = dict(rows=rows, types=types, mat=mat, br=BR, cl=clusters, kn=knames, cross=cross,
                feats=[[f, n_] for f, n_ in CL_FEATS], sc={k_: [[r['nr'], r['dtr3'] or 0] for r in rows if r.get('k') == k_][:300] for k_ in knames})
    body = '''
<div class="kpis" id="k"></div>
<div class="grid"><div class="card"><h2>各分點行為類型組成</h2><div class="ch lg"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>類型定義</h2><div id="t0"></div></div></div>
<div class="card"><h2>🤖 K-Means 機器分群（依歷史跟單超額報酬排序）</h2><p class="note">特徵經標準化後分群；「跟單超額」= 該配對出現有效買超後，未來 10 日個股報酬 − 同日股票池平均，按事件數加權。排第一且為正者即機器找出的「高勝率主力樣貌」。</p><div id="t2"></div></div>
<div class="grid"><div class="card"><h2>分群散佈：20日淨買/總量（x）× 3日內淨賣回（y）</h2><div class="ch lg"><canvas id="c2"></canvas></div></div>
<div class="card"><h2>規則分類 × 機器分群 對照</h2><div id="t3"></div></div></div>
<div class="card"><div class="row"><h2 style="margin:0">分點 × 個股 類型明細</h2><select id="s1"></select></div><div id="t1"></div></div>'''
    js = r'''
const pal={'波段主力':UP,'趨勢主力':'#ff8a65','短線主力':AC,'隔日沖':'#b07cff','高周轉':BL,'持續賣方':DN,'不明顯':'#3b4a5a'};
kpis('k',D.types.slice(0,6).map(t=>[t,D.rows.filter(r=>r.t===t).length,t==='持續賣方'?'dn':t.includes('主力')?'up':'']));
mk('c1',{type:'bar',data:{labels:D.br,datasets:D.types.map(t=>({label:t,data:D.mat[t],backgroundColor:pal[t]}))},options:{indexAxis:'y',scales:{x:ax({stacked:true}),y:ax({stacked:true})}}});
table('t0',[{k:'t',t:'類型',f:'tag',tc:'tc'},{k:'d',t:'判斷規則（依序）'}],[
 {t:'隔日沖',tc:'n',d:'事件 ≥2 次且平均隔日沖比例 ≥ __DTPAIR__'},{t:'高周轉',tc:'g',d:'20日活躍 ≥50% 且 (買+賣)/|淨買| ≥ 6'},
 {t:'趨勢主力',tc:'u',d:'20日、全期淨買 >0，近30日活躍日中 ≥65% 為買超（10～30 日）'},{t:'波段主力',tc:'u',d:'5／10／20 日皆淨買超，持續買進'},
 {t:'短線主力',tc:'n',d:'連續買超平均 2～5 天且 20 日內方向翻轉 ≥3 次'},{t:'持續賣方',tc:'d',d:'10、20 日皆淨賣超'},{t:'不明顯',tc:'g',d:'不符合以上'}],{h:340});
const cols=[{k:'branch',t:'分點'},{k:'stk',t:'股票'},{k:'t',t:'類型',f:'tag',tc:'tc'},{k:'n5',t:'5日',f:'n',c:1},{k:'n10',t:'10日',f:'n',c:1},{k:'n20',t:'20日',f:'n',c:1},{k:'nlb',t:'60日',f:'n',c:1},
 {k:'dtr',t:'隔日沖比例',f:'p'},{k:'turn',t:'周轉倍數',f:'f',d:1},{k:'act',t:'20日活躍度',f:'p',d:0},{k:'br',t:'買超日占比',f:'p',d:0},{k:'run',t:'平均連買天數',f:'f',d:1},{k:'flips',t:'方向翻轉',f:'n'},{k:'k',t:'機器分群',f:'tag',tc:'kc'},{k:'fex',t:'跟單10日超額',f:'p',d:2,c:1},{k:'fwin',t:'跟單勝率',f:'p'},{k:'fn',t:'跟單事件',f:'n'}];
const KP=[UP,AC,BL,DN,'#b07cff','#ff8a65','#4dd0e1','#9ccc65'];
table('t2',[{k:'rk',t:'#',f:'n'},{k:'k',t:'機器分群',f:'tag',tc:'kc'},{k:'n',t:'配對數',f:'n'},{k:'fex',t:'跟單10日超額',f:'p',d:2,c:1,heat:1},{k:'fex1',t:'前半段超額',f:'p',d:2,c:1},{k:'fex2',t:'後半段超額',f:'p',d:2,c:1},{k:'stab',t:'穩定性'},{k:'fwin',t:'跟單勝率',f:'p'},{k:'fn',t:'事件數',f:'n'}]
 .concat(D.feats.map(f=>({k:f[0],t:f[1],f:'f',d:2}))),D.cl,{sort:null});
if(D.kn.length)mk('c2',{type:'scatter',data:{datasets:D.kn.map((k,i)=>({label:k,data:D.sc[k].map(p=>({x:p[0],y:p[1]})),backgroundColor:KP[i%8],pointRadius:3}))},
 options:{scales:{x:ax({title:{display:true,text:'20日淨買/總量',color:MU}}),y:ax({title:{display:true,text:'3日內淨賣回',color:MU}})}}});
table('t3',[{k:'t',t:'規則類型'}].concat(D.kn.map(k=>({k:k,t:k,f:'n',heat:'pos'}))),D.cross,{sort:null});
sel('s1',[['ALL','全部類型']].concat(D.types.map(t=>[t,t])),v=>table('t1',cols,v==='ALL'?D.rows:D.rows.filter(r=>r.t===v),{sort:'n20'}));'''
    js = js.replace('__DTPAIR__', f'{DT_PAIR_RATIO:.0%}')
    method = f'''依每個「分點 × 個股」配對的持有行為分類：波段主力（持續買進）、趨勢主力（10～30 日）、短線主力（2～5 日）、隔日沖（今日買、明日賣）、高周轉（買賣頻繁）、持續賣方。
同一分點可能對不同個股呈現不同類型，分類結果作為後續評分「排除隔日沖」與「偏好波段/趨勢主力」的依據。
<ul><li><b>V1.1 機器分群</b>：除了原本的 If-Else 規則樹，另以 9 個標準化特徵（周轉、活躍度、買超日占比、連買天數、方向翻轉、T+1 隔日沖／3 日淨賣回、20／60 日淨買占總量）做 <b>K-Means（k={N_CLUSTERS}）</b>分群，群名依群中心特徵自動命名。</li>
<li>每群以「歷史跟單績效」評估：配對出現<b>有效買超</b>後，未來 10 日同日超額報酬與勝率（事件數加權）。超額最高的群就是資料自己找出的高勝率主力樣貌，可與規則分類交叉比對，檢查人為門檻是否有偏誤。</li>
<li><b>V1.4 穩定性</b>：跟單績效拆成期間前半段與後半段；只有兩段皆為正超額的群才以紅色標示為優勢群。只在單一段有效的群多半是特定行情的巧合，不宜追蹤。</li></ul>'''
    n = sum(1 for r in rows if r['t'] in ('波段主力', '趨勢主力'))
    return save_page(6, 'player_type', '六', '長線／短線主力類型分類', '波段、趨勢、短線、隔日沖、高周轉', method, body, js, data, f'{n} 組波段/趨勢主力' + (f'；最佳機器群：{clusters[0]["k"]}（{clusters[0]["stab"]}）' if clusters else ''))

build_s06()

In [ ]:
# ============ S07 分點集中度 ============
def conc_tier(x):
    if x is None or not np.isfinite(x): return None
    return '高度集中' if x >= CONC_TIER_HIGH else '中度集中' if x >= CONC_TIER_MID else '輕度集中' if x >= CONC_MID else '賣方集中' if x <= -CONC_MID else '分散'

def build_s07():
    rows = []
    for c in UNIV:
        v5 = VOL5[c].iloc[-1]
        if not np.isfinite(v5) or v5 <= 0 or N20[c].iloc[-1] == 0 and N5[c].iloc[-1] == 0:
            continue
        r = dict(code=c, stk=stk(c), n1=N1[c].iloc[-1], n5=N5[c].iloc[-1], n20=N20[c].iloc[-1], vol5=v5,
                 c1=f_(CONC1[c].iloc[-1]), c5=f_(CONC5[c].iloc[-1]), c20=f_(CONC20[c].iloc[-1]),
                 r5=f_(CONC5_RAW[c].iloc[-1]), dts=f_(DTSH[c].iloc[-5:].mean()), rv5=f_(RVOL5[c].iloc[-1]),
                 dsrc=(lambda x: '實際' if len(x) and np.isfinite(x).all() else '混合' if np.isfinite(x).any() else '估計')
                      (DT_SHARE[c].reindex(IDX[-5:]).values))
        rv = RVOL5[c].iloc[-1] if np.isfinite(RVOL5[c].iloc[-1]) and RVOL5[c].iloc[-1] > 0 else v5
        for b in BR:
            r[b] = NB5[b][c].iloc[-1] / rv
        r['t'] = conc_tier(r['c5']); r['tc'] = tone(r['t'])
        rows.append(r)
    rows.sort(key=lambda r: -(r['c5'] if r['c5'] is not None else -9))
    data = dict(rows=rows, br=BR, src=DT_SRC)
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>5 日集中度前 25：各分點貢獻（占 5 日實質留倉量 %）</h2><div class="ch lg"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>集中度總表</h2><div id="t1"></div></div>'''
    js = r'''
const R=D.rows;const n=t=>R.filter(r=>r.t===t).length;
kpis('k',[['高度集中 ≥__CH__',n('高度集中'),'up'],['中度集中 __CM__～__CH__',n('中度集中'),'up'],['輕度集中',n('輕度集中')],['賣方集中',n('賣方集中'),'dn'],['分析個股',R.length]]);
const tp=R.filter(r=>r.c5>0).slice(0,25);const pal=[UP,AC,BL,DN,'#b07cff'];
mk('c1',{type:'bar',data:{labels:tp.map(r=>r.stk),datasets:D.br.map((b,i)=>({label:b,data:tp.map(r=>r[b]),backgroundColor:pal[i%5]}))},
 options:{indexAxis:'y',scales:{x:ax({stacked:true,ticks:{color:MU,callback:v=>fmtP(v,0)}}),y:ax({stacked:true})},plugins:{tooltip:{callbacks:{label:c=>c.dataset.label+': '+fmtP(c.raw)}}}}});
table('t1',[{k:'stk',t:'股票'},{k:'c1',t:'1日集中度',f:'p',c:1,heat:1},{k:'c5',t:'5日集中度',f:'p',c:1,heat:1},{k:'c20',t:'20日集中度',f:'p',c:1,heat:1},{k:'t',t:'等級',f:'tag',tc:'tc'},{k:'n5',t:'5日淨買(張)',f:'n',c:1},{k:'n20',t:'20日淨買(張)',f:'n',c:1},{k:'r5',t:'5日集中度(未扣當沖)',f:'p',c:1},{k:'vol5',t:'5日成交量(張)',f:'n'},{k:'dts',t:'當沖占比',f:'p',d:0},{k:'rv5',t:'5日留倉量(張)',f:'n'},{k:'dsrc',t:'當沖來源'}].concat(D.br.map(b=>({k:b,t:b,f:'p',c:1}))),R);'''
    method = f'''<ul><li>文件定義：前 5 大分點買超 ÷ 全市場成交量。此處「前 5 大」= 本模型追蹤的 5 個分點。</li>
<li><b>V1.1 實質留倉量</b>：集中度 = 追蹤分點淨買超 ÷ <b>（成交量 − 當沖成交量）</b>，計算 1／5／20 日。當沖與高頻來回交易不會留下部位，放在分母會稀釋集中度。
當沖量取自 TWSE 當日沖銷交易統計（近 {DAYTRADE_YEARS} 年、上市股）；上櫃股或更早日期以該股歷史當沖占比中位數、全市場中位數或 0 補值。來源欄依最近五日原始占比覆蓋標示：「實際」= 五日皆有、「混合」= 部分有、「估計」= 全無。表格保留未扣當沖的舊版數值供對照。
對敲量無公開資料，無法扣除。（全域資料狀態：{DT_SRC}；逐股窗口完整性見來源欄）</li>
<li>等級：≥{CONC_TIER_HIGH:.0%} 高度集中、{CONC_TIER_MID:.0%}～{CONC_TIER_HIGH:.0%} 中度、{CONC_MID:.0%}～{CONC_TIER_MID:.0%} 輕度、≤ −{CONC_MID:.0%} 賣方集中。</li></ul>'''
    js = js.replace('__CH__', f'{CONC_TIER_HIGH:.0%}').replace('__CM__', f'{CONC_TIER_MID:.0%}')
    n = sum(1 for r in rows if r['t'] in ('高度集中', '中度集中'))
    return save_page(7, 'concentration', '七', '分點集中度', '少數分點大量交易＝籌碼集中', method, body, js, data, f'{n} 檔中度以上集中')

build_s07()

In [ ]:
# ============ S08 買賣家數差 ============
def breadth_pattern(b5, s5, n5):
    nb = len(BR)
    if b5 >= max(nb - 1, 2): return '買盤廣泛'
    if s5 >= max(nb - 1, 2): return '賣盤廣泛'
    if n5 > 0 and b5 < s5: return '買方集中、賣方分散'
    if n5 < 0 and s5 < b5: return '賣方集中、買方分散'
    return '分歧'

def n_(P, c):
    return f_(P[c].iloc[-1]) if c in P else None

def build_s08():
    rows = []
    for c in UNIV:
        if (NET_T[c].iloc[-5:] == 0).all():
            continue
        b5, s5, n5 = int(BUYERS5[c].iloc[-1]), int(SELLERS5[c].iloc[-1]), N5[c].iloc[-1]
        p = breadth_pattern(b5, s5, n5)
        rows.append(dict(stk=stk(c), b1=int(BUYERS[c].iloc[-1]), s1=int(SELLERS[c].iloc[-1]), d1=int(BRD[c].iloc[-1]), n1=N1[c].iloc[-1],
                         b5=b5, s5=s5, d5=b5 - s5, n5=n5, a5=AMT5[c].iloc[-1], p=p, pc=tone(p)))
    rows.sort(key=lambda r: (-r['d5'], -r['n5']))
    pb = ((NET_T > 0).astype(int)).sum(axis=1); ps = ((NET_T < 0).astype(int)).sum(axis=1)
    pbn = sum((NET[b] > 0).sum(axis=1) for b in BR); psn = sum((NET[b] < 0).sum(axis=1) for b in BR)
    dd = IDX[-40:]
    # ---- V1.1：全市場分點家數（所有券商分點，非只有 5 個追蹤分點）----
    mrows, mser = [], {}
    MB = MKT_BREADTH.copy() if len(MKT_BREADTH) else pd.DataFrame()
    if len(MB):
        # 報表欄位若缺失，保守視為不可用；不可把缺漏的前 15 大欄位當成 0。
        for _col in ('nbuy', 'nsell', 'top_buy', 'top_sell'):
            if _col not in MB:
                MB[_col] = np.nan
        MB['date'] = pd.to_datetime(MB['date']).dt.normalize(); MB['code'] = MB['code'].astype(str)
        MB = (MB.dropna(subset=['nbuy', 'nsell'])
                .drop_duplicates(['code', 'date'], keep='last')
                .sort_values(['code', 'date']))
        MB['diff'] = MB['nbuy'] - MB['nsell']
        _big_ok = MB['top_buy'].notna() & MB['top_sell'].notna()
        MB['big'] = np.nan
        MB.loc[_big_ok, 'big'] = (MB.loc[_big_ok, 'top_buy'].abs()
                                  - MB.loc[_big_ok, 'top_sell'].abs())
        # 只有完整的最近五個 IDX 交易日才標成「今日／5日」；不足時回退到原追蹤分點表。
        _recent5 = pd.DatetimeIndex(IDX[-5:]) if len(IDX) >= 5 else pd.DatetimeIndex([])
        _prior5 = pd.DatetimeIndex(IDX[-10:-5]) if len(IDX) >= 10 else pd.DatetimeIndex([])
        for c, g in MB.groupby('code'):
            g = g.sort_values('date')
            g5 = g[g['date'].isin(_recent5)].copy()
            if len(_recent5) != 5 or len(g5) != 5:
                continue
            gp = g[g['date'].isin(_prior5)]
            d5 = float(g5['diff'].mean())
            big5 = float(g5['big'].sum()) if g5['big'].notna().all() else None
            vol5 = np.nan
            if c in VOL5:
                _v5 = VOL5[c].reindex(_recent5)
                if _v5.notna().all():
                    vol5 = float(_v5.iloc[-1])
            if big5 is not None and d5 < 0 and big5 > 0: p_ = '買方集中、賣方分散'
            elif big5 is not None and d5 > 0 and big5 < 0: p_ = '賣方集中、買方分散'
            elif big5 is not None and d5 > 0 and big5 > 0: p_ = '買盤廣泛'
            elif big5 is not None and d5 < 0 and big5 < 0: p_ = '賣盤廣泛'
            else: p_ = '分歧'
            mrows.append(dict(code=c, stk=stk(c), nb=g5['nbuy'].iloc[-1], ns=g5['nsell'].iloc[-1], d1=g5['diff'].iloc[-1], d5=d5,
                              dchg=d5 - float(gp['diff'].mean()) if len(gp) == 5 else None, big5=big5,
                              bigr=big5 / vol5 if big5 is not None and np.isfinite(vol5) and vol5 != 0 else None,
                              tr5=n_(N5, c), r5=f_(RET5[c].iloc[-1]) if c in RET5 else None, p=p_, pc=tone(p_)))
            mser[c] = dict(d=[x.strftime('%m/%d') for x in g['date']], diff=g['diff'].tolist(), big=g['big'].tolist())
        mrows.sort(key=lambda r: (r['p'] != '買方集中、賣方分散', -(r['bigr'] or 0)))
    data = dict(rows=rows, mrows=mrows, mser=mser, d=[x.strftime('%m/%d') for x in dd], pb=pbn.reindex(dd).tolist(), ps=psn.reindex(dd).tolist())
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>🌐 全市場分點家數差（所有券商分點）</h2><p class="note" id="mn"></p><div id="t2"></div></div>
<div class="card"><div class="row"><h2 style="margin:0">全市場家數差 × 前 15 大淨流向</h2><select id="s2"></select></div><div class="ch"><canvas id="c2"></canvas></div></div>
<div class="card"><h2>追蹤分點整體：每日「買超筆數 vs 賣超筆數」（分點×個股）</h2><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>追蹤 5 分點：個股買賣家數差</h2><div id="t1"></div></div>'''
    js = r'''
const R=D.mrows.length?D.mrows:D.rows;const n=t=>R.filter(r=>r.p===t).length;
$('mn').textContent=D.mrows.length?'家數差 = 全市場買進家數 − 賣出家數（5 日平均）；前15大淨流向 = 前 15 大買超分點合計 − 前 15 大賣超分點合計（5 日）。家數差為負且前15大淨流向為正＝報表上的分散賣出／集中買進代理，不等同散戶賣給大戶。':'⚠️ 未取得全市場家數（BREADTH_STOCKS=0 或網頁格式不符），以下 KPI 改用 5 個追蹤分點。';
table('t2',[{k:'stk',t:'股票'},{k:'p',t:'型態',f:'tag',tc:'pc'},{k:'nb',t:'今日買進家數',f:'n'},{k:'ns',t:'今日賣出家數',f:'n'},{k:'d1',t:'今日家數差',f:'n',c:1,heat:1},{k:'d5',t:'5日平均家數差',f:'n',c:1,heat:1},
 {k:'dchg',t:'家數差變化(較前5日)',f:'n',c:1},{k:'big5',t:'前15大淨流向(張)',f:'n',c:1,heat:1},{k:'bigr',t:'占5日量',f:'p',c:1},{k:'tr5',t:'追蹤分點5日淨買',f:'n',c:1},{k:'r5',t:'5日漲跌',f:'p',c:1}],D.mrows,{sort:null});
if(D.mrows.length)sel('s2',D.mrows.map(r=>[r.code,r.stk+'｜'+r.p]),c=>{const x=D.mser[c];mk('c2',{data:{labels:x.d,datasets:[
 {type:'bar',label:'家數差(買−賣)',data:x.diff,backgroundColor:x.diff.map(v=>v>=0?'rgba(255,77,79,.5)':'rgba(34,197,94,.5)')},
 {type:'line',label:'前15大淨流向(張)',data:x.big,borderColor:AC,pointRadius:2,yAxisID:'y1'}]},options:{scales:{x:ax(),y:ax(),y1:ax({position:'right',grid:{display:false}})}}})});
kpis('k',[['買盤廣泛',n('買盤廣泛'),'up'],['買方集中、賣方分散',n('買方集中、賣方分散'),'up'],['賣盤廣泛',n('賣盤廣泛'),'dn'],['賣方集中、買方分散',n('賣方集中、買方分散'),'dn'],['分歧',n('分歧')]]);
mk('c1',{type:'bar',data:{labels:D.d,datasets:[{label:'買超筆數',data:D.pb,backgroundColor:UP},{label:'賣超筆數',data:D.ps.map(v=>-v),backgroundColor:DN},
 {type:'line',label:'差額',data:D.pb.map((v,i)=>v-D.ps[i]),borderColor:AC,pointRadius:0}]},options:{scales:{x:ax({stacked:true}),y:ax({stacked:true})}}});
table('t1',[{k:'stk',t:'股票'},{k:'b1',t:'今日買方家數',f:'n'},{k:'s1',t:'今日賣方家數',f:'n'},{k:'d1',t:'今日家數差',f:'n',c:1,heat:1},{k:'n1',t:'今日淨買(張)',f:'n',c:1},
 {k:'b5',t:'5日買方家數',f:'n'},{k:'s5',t:'5日賣方家數',f:'n'},{k:'d5',t:'5日家數差',f:'n',c:1,heat:1},{k:'n5',t:'5日淨買(張)',f:'n',c:1},{k:'a5',t:'5日淨額(仟元)',f:'n',c:1},{k:'p',t:'型態',f:'tag',tc:'pc'}],R);'''
    method = f'''<ul><li>家數差 = 買超分點數 − 賣超分點數，並搭配買賣超張數／金額判讀。</li>
<li>型態：多數分點同向買＝買盤廣泛；淨買超但買方家數少於賣方＝買方集中、賣方分散（文件的 25 vs 80 例子）。</li>
<li><b>V1.1 全市場家數</b>：對追蹤分點最活躍的前 {BREADTH_STOCKS} 檔，逐日抓取富邦「個股主力進出」頁的<b>全市場買進家數／賣出家數</b>與前 15 大買超／賣超合計（近 {BREADTH_DAYS} 日），
作為報表上買賣分點分布的集中代理；不可直接辨認散戶／大戶或最終投資人身分。家數差變化為負且前 15 大淨流向為正，只能描述為「報表上的分散賣出／集中買進同時出現」的待驗證訊號。</li>
<li>下方保留原本「5 個追蹤分點」的共識度表（範圍 −5～+5）。</li></ul>'''
    n = sum(1 for r in (mrows or rows) if r['p'] == '買方集中、賣方分散')
    return save_page(8, 'breadth', '八', '買賣家數差', '買方集中、賣方分散 vs 市場買盤廣泛', method, body, js, data, f'{n} 檔買方集中、賣方分散')

build_s08()

In [ ]:
# ============ S09 吸籌偵測 ============
def mini_series(c, n=30):
    dd = IDX[-n:]
    return dict(d=[x.strftime('%m/%d') for x in dd], px=C[c].reindex(dd).tolist(), cn=cum(NET_T[c].reindex(dd).tolist()))

def cum(a):
    s, out = 0.0, []
    for v in a:
        s += 0 if v is None or not np.isfinite(v) else v
        out.append(s)
    return out

def ck(x):
    return None if x is None else bool(x)

def build_s09():
    rows, mini = [], {}
    for c in UNIV:
        cl = CJ[c].iloc[-10:]
        if cl.isna().all():
            continue
        r20 = RET20[c].iloc[-1]; rng10 = (cl.max() - cl.min()) / cl.mean()
        sg = VOLA[c].iloc[-1]
        sg = sg if np.isfinite(sg) else 0.02
        th20, th10 = max(0.05, 0.7 * sg * np.sqrt(20)), max(0.08, 1.5 * sg * np.sqrt(10))   # 依個股波動度放寬（高波動股不會永遠不符合）
        c1 = bool(abs(r20) <= th20 and rng10 <= th10) if np.isfinite(r20) else None
        vr = VR[c].iloc[-1]; c2 = bool(vr >= 1.1) if np.isfinite(vr) else None
        nd = int((NET_T[c].iloc[-5:] > 0).sum()); c3 = bool(nd >= 4 and MAT5[c].iloc[-1])
        mc = MARCHG5[c].iloc[-1]; c4 = bool(mc < 0) if np.isfinite(mc) else None
        f5 = FOR5[c].iloc[-1]; v5 = VOL5[c].iloc[-1]
        c5 = bool(f5 >= -0.05 * v5) if np.isfinite(f5) and np.isfinite(v5) else None
        # ⑥ V1.1 隱性吸籌：盤中賣壓（收黑 K）卻收長下影線、股價跌不下去 → 有人在下檔承接
        o_, h_, l_, cr = [X[c].reindex(IDX[-10:]) for X in (OA, HA, LA_, CA)]
        rng_ = (h_ - l_).replace(0, np.nan)
        # 只把完整且合法的 OHLC K 棒納入；10 日視窗不完整時，隱性吸籌應為 N/A，
        # 不以 skipna 後的少數 K 棒或未知跌幅通過條件。
        valid = (np.isfinite(o_) & np.isfinite(h_) & np.isfinite(l_) & np.isfinite(cr) & (rng_ > 0)
                 & (h_ >= np.maximum(o_, cr)) & (l_ <= np.minimum(o_, cr)))
        shd = ((np.minimum(o_, cr) - l_) / rng_).where(valid).mean()
        pres = valid & (cr < o_)                              # 收盤 < 開盤：盤中賣壓的 K 線代理（非內外盤觀測）
        ret_ = CJ[c].pct_change().reindex(IDX[-10:])
        pret = ret_.where(pres).mean()
        valid_n = int(valid.sum())
        if len(rng_) < 10 or valid_n < len(rng_) or bool((pres & ~np.isfinite(ret_)).any()):
            c6 = None
        else:
            c6 = bool(np.isfinite(shd) and shd >= ABSORB_SHADOW and pres.sum() >= 3 and np.isfinite(pret) and pret >= -0.005) if np.isfinite(shd) else None
        cs = [c1, c2, c3, c4, c5, c6]
        avail = [x for x in cs if x is not None]; passed = sum(avail)
        if c3 and c1 and passed >= 5: t = '吸籌'
        elif c3 and passed >= 4: t = '疑似吸籌'
        else: t = None
        if not c3 and passed < 4:
            continue
        rows.append(dict(code=c, stk=stk(c), c1=c1, c2=c2, c3=c3, c4=c4, c5=c5, c6=c6, shd=f_(shd), pd_=int(pres.sum()), pret=f_(pret), ps=passed, av=len(avail), r20=r20, th20=th20, rg=rng10, th10=th10, vr=vr, nd=nd, n5=N5[c].iloc[-1],
                         mc=mc, f5=f5, t=t, tc=tone(t) if t else 'g'))
    rows.sort(key=lambda r: (r['t'] != '吸籌', r['t'] is None, -r['ps'], -r['n5']))
    for r in rows[:12]:
        mini[r['code']] = mini_series(r['code'])
    data = dict(rows=rows, mini=mini)
    body = MINI_BODY
    js = MINI_JS.replace('__COLS__', r'''[{k:'stk',t:'股票'},{k:'t',t:'判定',f:'tag',tc:'tc'},{k:'ps',t:'符合條件',f:'n'},{k:'av',t:'可判斷',f:'n'},
 {k:'c1',t:'①股價持平',f:'ck'},{k:'c2',t:'②量增',f:'ck'},{k:'c3',t:'③分點5日續買',f:'ck'},{k:'c4',t:'④融資下降',f:'ck'},{k:'c5',t:'⑤外資未大賣',f:'ck'},{k:'c6',t:'⑥隱性吸籌',f:'ck'},
 {k:'r20',t:'20日漲跌',f:'p',c:1},{k:'th20',t:'持平門檻±',f:'p'},{k:'rg',t:'10日振幅',f:'p'},{k:'th10',t:'振幅門檻',f:'p'},{k:'vr',t:'量比(5/20)',f:'f'},{k:'nd',t:'5日中買超天數',f:'n'},{k:'n5',t:'5日淨買(張)',f:'n',c:1},{k:'mc',t:'融資5日增減',f:'n',c:1},{k:'f5',t:'外資5日(張)',f:'n',c:1},{k:'shd',t:'10日下影線比',f:'p',d:0},{k:'pd_',t:'收黑天數',f:'n'},{k:'pret',t:'收黑日平均漲跌',f:'p',d:2,c:1}]''').replace('__KP__', r'''[['吸籌',R.filter(r=>r.t==='吸籌').length,'up'],['疑似吸籌',R.filter(r=>r.t==='疑似吸籌').length],['分點5日續買',R.filter(r=>r.c3).length],['候選總數',R.length]]''')
    method = f'''股價沒有大漲，但特定分點持續吸收賣單 → 可能形成低檔吸籌。六個條件：
<ul><li>① 股價持平：20 日漲跌在 ±max(5%, 0.7×20日波動) 內，且 10 日振幅 ≤ max(8%, 1.5×10日波動)。門檻隨個股 60 日波動度調整，避免高波動股永遠不符合、低波動股過度寬鬆；價格用還原價，除息不影響判斷</li><li>② 成交量逐步增加：5 日均量 ÷ 20 日均量 ≥ 1.1</li>
<li>③ 分點持續買超：追蹤分點合計近 5 日至少 4 日買超；「有效買超」完全沿用 Cell 9 <code>material()</code>：N5 &gt; 0 且 N5 ≥ max(MATR×VOL5, MAT_MIN_LOTS)，其中 MATR 依 60 日均量百分位，在 {MAT_RATIO_ILLIQ:.1%}～{MAT_RATIO_LIQUID:.1%} 間內插（缺值才回到 MATERIAL_RATIO）</li><li>④ 融資下降：5 日融資餘額減少</li><li>⑤ 外資沒有大幅賣超：外資 5 日賣超不超過 5 日成交量的 5%</li>
<li>⑥ <b>V1.1 隱性吸籌</b>：近 10 個完整交易日平均下影線比例 ≥ {ABSORB_SHADOW:.0%}，且至少 3 天「收盤 &lt; 開盤」（僅是賣壓 K 線代理，非內外盤觀測），那些日子平均還原收盤跌幅不超過 0.5%——以 K 線結構代理下檔承接。若 OHLC 或跌幅不足，c6 為 N/A。
（交易所五檔與內外盤歷史需付費資料；若日後取得內外盤歷史，可另建獨立特徵，不應直接宣稱等同。）</li></ul>
判定：①③ 成立且合計 ≥5 項 → 吸籌；③ 成立且 ≥4 項 → 疑似吸籌。資料不足（上櫃無融資/外資或 K 線不完整）標 N/A，不把未知當成通過。'''
    n = sum(1 for r in rows if r['t'] == '吸籌')
    return save_page(9, 'accumulation', '九', '吸籌偵測', '股價沒大漲、量增、分點持續吸收賣單', method, body, js, data, f'{n} 檔吸籌、{sum(1 for r in rows if r["t"] == "疑似吸籌")} 檔疑似')

MINI_BODY = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>訊號總表</h2><div id="t1"></div></div>
<div class="card"><h2>前 12 檔：股價（藍）× 追蹤分點累積淨買（金）</h2><div class="mini" id="mini"></div></div>'''
MINI_JS = r'''
const R=D.rows;kpis('k',__KP__);
table('t1',__COLS__,R);
$('mini').innerHTML=Object.keys(D.mini).map(c=>`<div class="card"><div class="note">${(R.find(r=>r.code===c)||{}).stk||c}</div><div class="ch sm"><canvas id="m${c}"></canvas></div></div>`).join('');
Object.keys(D.mini).forEach(c=>{const x=D.mini[c];mk('m'+c,{type:'line',data:{labels:x.d,datasets:[{label:'股價',data:x.px,borderColor:BL,pointRadius:0,yAxisID:'y'},{label:'累積淨買',data:x.cn,borderColor:AC,pointRadius:0,yAxisID:'y1'}]},
 options:{plugins:{legend:{display:false}},scales:{x:ax({ticks:{display:false}}),y:ax(),y1:ax({position:'right',grid:{display:false}})}}})});'''

build_s09()


In [ ]:
# ============ S10 出貨偵測 ============
def build_s10():
    rows, mini = [], {}
    MSURGE = MARCHG5 / MAR_D.shift(5).replace(0, np.nan)                    # 融資 5 日增幅
    if len(ODD_RAW):
        _o = ODD_RAW.copy(); _o.index = _naive(_o.index)
        ODD = _o.reindex(columns=CODES).reindex(IDX)
    else:
        ODD = pd.DataFrame(np.nan, index=IDX, columns=CODES)
    for c in UNIV:
        r20, r10 = RET20[c].iloc[-1], RET10[c].iloc[-1]
        sg = VOLA[c].iloc[-1]
        sg = sg if np.isfinite(sg) else 0.02
        c1 = bool((np.isfinite(r20) and r20 >= max(0.15, 1.5 * sg * np.sqrt(20))) or (np.isfinite(r10) and r10 >= max(0.10, 1.5 * sg * np.sqrt(10))))
        turned = []
        # 不足完整「前 6～20 日 + 近 5 日」視窗時，不觸發原買方轉賣。
        if len(IDX) >= 20:
            for b in BR:
                prior = NET[b][c].iloc[-20:-5].sum(); now = NET[b][c].iloc[-5:].sum()
                if prior > 0 and now < 0:
                    turned.append(f'{b}（前 {prior:,.0f} → 近5日 {now:,.0f}）')
        c2 = bool(turned) and N5[c].iloc[-1] < 0
        v20 = VOL20[c].iloc[-4] / 20 if len(IDX) > 4 else np.nan     # 以爆量「之前」的 20 日均量為基準，不讓爆量本身拉高基準
        vmax = VOLD[c].iloc[-3:].max()
        c3 = bool(vmax >= 2 * v20) if np.isfinite(vmax) and np.isfinite(v20) and v20 > 0 else None
        ps = sum(x for x in (c1, c2, c3) if x)
        # V1.1 散戶接盤：融資暴增、零股成交量創近期新高
        ms = MSURGE[c].iloc[-1]
        c4 = bool(ms >= MARGIN_SURGE) if np.isfinite(ms) else None
        # 保留交易日位置；缺最新日或中間日不可當作「近三日／近 N 日」的零股新高。
        od_aligned = ODD[c].reindex(IDX)
        odd_window = max(20, int(ODD_LOT_DAYS or 0))
        odd_hist = od_aligned.iloc[-odd_window:] if len(od_aligned) >= odd_window else od_aligned.iloc[:0]
        odd_recent = odd_hist.iloc[-3:]
        odd_prior = odd_hist.iloc[:-3]
        c5 = (bool(odd_recent.max() >= odd_prior.max())
              if len(odd_prior) >= 17 and odd_recent.notna().all() and odd_prior.notna().all()
              else None)
        # 顯示訊號日對齊值；缺最新資料時顯示 N/A，不退回舊日期觀測。
        od_latest = od_aligned.iloc[-1] if len(od_aligned) else np.nan
        retail = bool(c4 or c5)
        if ps < 2:
            continue
        if ps == 3 and retail: t = '出貨警報(散戶接盤)'
        elif ps == 3 or retail: t = '出貨警報'
        else: t = '疑似出貨'
        rows.append(dict(code=c, stk=stk(c), c1=c1, c2=c2, c3=c3, ps=ps, r10=r10, r20=r20, n5=N5[c].iloc[-1], n20=N20[c].iloc[-1],
                         vm=vmax / v20 if v20 else None, c4=c4, c5=c5, ms=f_(ms), od=f_(od_latest), tb='；'.join(turned) or '—', t=t, tc=tone(t)))
    rows.sort(key=lambda r: (-r['ps'], -(r['t'] == '出貨警報(散戶接盤)'), r['n5']))
    for r in rows[:12]:
        mini[r['code']] = mini_series(r['code'])
    data = dict(rows=rows, mini=mini)
    js = MINI_JS.replace('__COLS__', r'''[{k:'stk',t:'股票'},{k:'t',t:'判定',f:'tag',tc:'tc'},{k:'ps',t:'符合條件',f:'n'},{k:'c1',t:'①股價強漲',f:'ck'},{k:'c2',t:'②原買方轉賣',f:'ck'},{k:'c3',t:'③爆大量',f:'ck'},{k:'c4',t:'④融資暴增',f:'ck'},{k:'c5',t:'⑤零股創高',f:'ck'},
 {k:'r10',t:'10日漲跌',f:'p',c:1},{k:'r20',t:'20日漲跌',f:'p',c:1},{k:'n5',t:'5日淨買',f:'n',c:1},{k:'n20',t:'20日淨買',f:'n',c:1},{k:'vm',t:'近3日最大量/均量',f:'f'},{k:'ms',t:'融資5日增幅',f:'p',c:1},{k:'od',t:'今日零股(股)',f:'n'},{k:'tb',t:'轉賣分點'}]''').replace('__KP__', r'''[['散戶接盤確認',R.filter(r=>r.t==='出貨警報(散戶接盤)').length,'dn'],['出貨警報',R.filter(r=>r.t==='出貨警報').length,'dn'],['疑似出貨',R.filter(r=>r.t==='疑似出貨').length],['原買方轉賣',R.filter(r=>r.c2).length],['爆量',R.filter(r=>r.c3).length]]''')
    method = f'''股價看起來很強，但原本持續買進的分點開始賣出，同時爆大量 → 可能高檔派發／出貨。
<ul><li>① 股價強漲（還原價）：20 日漲幅 ≥ max(15%, 1.5×20日波動) 或 10 日漲幅 ≥ max(10%, 1.5×10日波動)——高波動股需要更大漲幅才算「強漲」</li><li>② 原買方轉賣：某追蹤分點在前 6～20 日淨買超、近 5 日轉為淨賣超，且合計 5 日淨賣超</li>
<li>③ 爆大量：近 3 日最大量 ≥ 爆量前 20 日均量的 2 倍</li></ul><b>V1.1 散戶接盤確認</b>：④ 融資 5 日增幅 ≥ {MARGIN_SURGE:.0%}（融資暴增）；⑤ 零股成交量近 3 日創近 {ODD_LOT_DAYS} 日新高（散戶小額追價）。上櫃或無資料標 N/A。
<ul><li>①②③ 皆成立且 ④⑤ 任一成立 → <b>出貨警報(散戶接盤)</b>（信心最高）</li><li>①②③ 皆成立，或兩項成立＋散戶接盤 → 出貨警報</li><li>其餘兩項成立 → 疑似出貨</li></ul>'''
    n = sum(1 for r in rows if r['t'].startswith('出貨警報'))
    return save_page(10, 'distribution', '十', '出貨偵測', '股價強、原買方轉賣、爆大量', method, MINI_BODY, js, data, f'{n} 檔出貨警報')

build_s10()

In [ ]:
# ============ S11 主力反轉訊號 ============
def build_s11():
    # S11 需要訊號日前至少 20 個完整交易日；不改共用因子引擎，只在本頁遮罩暖機區。
    _bear = BEAR.astype(bool).copy()
    _bull = BULL.astype(bool).copy()
    _warmup_idx = IDX[:min(20, len(IDX))]
    if len(_warmup_idx):
        _bear.loc[_warmup_idx, :] = False
        _bull.loc[_warmup_idx, :] = False
    rows = []
    for k in range(max(20, len(IDX) - 20), len(IDX)):
        d = IDX[k]
        for kind, M in (('主力轉空', _bear), ('籌碼反轉(轉多)', _bull)):
            hit = M.iloc[k]
            for c in hit[hit].index:
                if c not in UNIV:
                    continue
                rows.append(dict(date=d.strftime('%Y-%m-%d'), stk=stk(c), t=kind, tc=tone(kind), today=NET_T[c].iloc[k],
                                 p5=n5p[c].iloc[k], p10=n10p[c].iloc[k], p20=n20p[c].iloc[k], px=CJ[c].iloc[k], ma10=MA10[c].iloc[k],
                                 stop=f_((STOP_LONG if kind == '主力轉空' else STOP_SHORT)[c].iloc[k]), zone=f_((ZONE_LO if kind == '主力轉空' else ZONE_HI)[c].iloc[k]), craw=f_(C[c].iloc[k]),
                                 since=(CJ[c].iloc[-1] / CJ[c].iloc[k] - 1) if k < len(IDX) - 1 else None))
    rows.sort(key=lambda r: r['date'], reverse=True)
    br_rows = []
    # 單一分點層級也必須有完整的前 20 日窗口；正常三年資料的最新日邏輯不變。
    if len(IDX) >= 21:
        for b in BR:
            P = NET[b]; a5, a10, a20 = rs(P, 5).shift(1).iloc[-1], rs(P, 10).shift(1).iloc[-1], rs(P, 20).shift(1).iloc[-1]; t = P.iloc[-1]
            bdn, bup = BRK_DN.iloc[-1], BRK_UP.iloc[-1]                           # 與個股層級相同的防守線判定
            mp = np.maximum(DT_MIN_LOTS, MATR.iloc[-2] * VOL5.iloc[-2].fillna(0))
            bear = (a5 > 0) & (a10 > 0) & (a20 > 0) & (a5 >= mp) & (t < 0) & (t.abs() >= 0.3 * a5) & bdn
            bull = (a5 < 0) & (a10 < 0) & (a20 < 0) & (a5.abs() >= mp) & (t > 0) & (t >= 0.3 * a5.abs()) & bup
            for c in bear[bear].index:
                if c in UNIV: br_rows.append(dict(branch=b, stk=stk(c), t='主力轉空', tc='d', today=t[c], p5=a5[c], p10=a10[c], p20=a20[c]))
            for c in bull[bull].index:
                if c in UNIV: br_rows.append(dict(branch=b, stk=stk(c), t='籌碼反轉(轉多)', tc='u', today=t[c], p5=a5[c], p10=a10[c], p20=a20[c]))
    dd = IDX[-60:]
    U = [c for c in UNIV]
    data = dict(rows=rows, br=br_rows, d=[x.strftime('%m/%d') for x in dd], nb=_bear[U].reindex(dd).sum(axis=1).tolist(), nu=_bull[U].reindex(dd).sum(axis=1).tolist(),
                last=L.strftime('%Y-%m-%d'))
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>每日訊號數</h2><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>近 20 日個股訊號（追蹤分點合計）</h2><div id="t1"></div></div>
<div class="card"><h2>今日單一分點層級的轉向</h2><div id="t2"></div></div>'''
    js = r'''
const T=D.rows.filter(r=>r.date===D.last);
kpis('k',[['今日主力轉空',T.filter(r=>r.t==='主力轉空').length,'dn'],['今日籌碼反轉(轉多)',T.filter(r=>r.t!=='主力轉空').length,'up'],['近20日轉空',D.rows.filter(r=>r.t==='主力轉空').length],['近20日轉多',D.rows.filter(r=>r.t!=='主力轉空').length],['分點層級今日訊號',D.br.length]]);
mk('c1',{type:'bar',data:{labels:D.d,datasets:[{label:'籌碼反轉(轉多)',data:D.nu,backgroundColor:UP},{label:'主力轉空',data:D.nb.map(v=>-v),backgroundColor:DN}]},options:{scales:{x:ax({stacked:true}),y:ax({stacked:true})}}});
const pc=[{k:'today',t:'今日淨買',f:'n',c:1},{k:'p5',t:'前5日',f:'n',c:1},{k:'p10',t:'前10日',f:'n',c:1},{k:'p20',t:'前20日',f:'n',c:1}];
table('t1',[{k:'date',t:'日期'},{k:'stk',t:'股票'},{k:'t',t:'訊號',f:'tag',tc:'tc'}].concat(pc).concat([{k:'px',t:'收盤(還原)',f:'f'},{k:'ma10',t:'MA10(還原)',f:'f'},{k:'stop',t:'ATR停損/反壓(還原)',f:'f'},{k:'craw',t:'收盤(原始)',f:'f'},{k:'zone',t:'成本區下緣/上緣(還原)',f:'f'},{k:'since',t:'訊號後漲跌',f:'p',c:1}]),D.rows);
table('t2',[{k:'branch',t:'分點'},{k:'stk',t:'股票'},{k:'t',t:'訊號',f:'tag',tc:'tc'}].concat(pc),D.br);'''
    _mode = {'MA10': '收盤跌破／站上 10MA（原法）', 'ATR': 'ATR 移動停損', 'ZONE': '主力成本密集區下緣／上緣', 'ATR_ZONE': 'ATR 移動停損 或 成本密集區（任一）'}[REV_BREAK_MODE]
    method = f'''<ul><li><b>主力轉空警報</b>：前 5／10／20 日累積皆為淨買超，今日轉為淨賣超（賣超量 ≥ 前 5 日累積的 30%），且<b>跌破防守線</b>。</li>
<li><b>籌碼反轉（轉多）</b>：前 5／10／20 日累積皆為淨賣超，今日轉為淨買超（≥ 前 5 日賣超量的 30%），且<b>站上反壓線</b>。</li>
<li><b>V1.1 防守線</b>（目前：{_mode}，參數 <code>REV_BREAK_MODE</code>）：單純跌破 10MA 在震盪市容易洗出假訊號，改用
① <b>Chandelier ATR 移動停損</b>：近 22 日最高價（還原） − {ATR_MULT:g} × ATR{ATR_N}（轉多用近 22 日最低價 + {ATR_MULT:g} × ATR）；ATR 先由原始高低收盤計算真實波幅，再依當日還原／原始收盤比例換到還原尺度。
② <b>主力成本區</b>：近 60 日正淨買超權重的加權均值 ± 0.674 × 加權標準差（不是精確 P25／P75），跌破下緣代表成本區失守。任一成立即判定趨勢破壞；缺少成本區時依引擎既有路徑退回 ATR 或 MA10。</li>
<li>先前的累積量須有份量：前 5 日累積 ≥ max(50 張, 5 日成交量 × 動態流動性門檻)，避免前幾天只買 1、2 張就觸發。均線、ATR 停損、成本區（V1.5 起買進價已依除權息還原）一律採還原價。</li>
<li>同時提供「追蹤分點合計」與「單一分點」兩個層級；<code>訊號後漲跌</code>是截至最新資料日的變動持有期描述，不是固定持有期回測或績效估計。</li></ul>'''
    t = [r for r in rows if r['date'] == L.strftime('%Y-%m-%d')]
    return save_page(11, 'reversal', '十一', '主力反轉訊號', '主力轉空警報 ／ 籌碼反轉', method, body, js, data, f'今日 {len(t)} 則反轉訊號')

build_s11()

In [ ]:
# ============ S12 主力行為矩陣 ============
def matrix_rows(codes):
    out = []
    for c in codes:
        sig = '轉多' if BULL[c].iloc[-3:].any() else '轉空' if BEAR[c].iloc[-3:].any() else ''
        out.append(dict(code=c, stk=stk(c), n5=N5[c].iloc[-1], n10=N10[c].iloc[-1], n20=N20[c].iloc[-1], n60=N60[c].iloc[-1], cons=CONS[c].iloc[-1],
                        conc=f_(CONC5[c].iloc[-1]), brd=int(BRD5[c].iloc[-1]), cost=f_(COST[c].iloc[-1]), dtr=f_(DTR[c].iloc[-1]), sig=sig, sigc='u' if sig == '轉多' else 'd',
                        dist=f_(DIST[c].iloc[-1]), vr=f_(VR[c].iloc[-1]), mc=f_(MARCHG5[c].iloc[-1]), f5=f_(FOR5[c].iloc[-1]), t5=f_(TRU5[c].iloc[-1]), px=f_(C[c].iloc[-1])))
    return out

PCA_FEATS = [  # (key, 名稱, 群組, 面板)
    ('n5r', '5日買超/量', '籌碼動能', lambda: N5 / VOL5.replace(0, np.nan)),
    ('n10r', '10日買超/量', '籌碼動能', lambda: N10 / VOL10.replace(0, np.nan)),
    ('n20r', '20日買超/量', '中長線佈局', lambda: N20 / VOL20.replace(0, np.nan)),
    ('n60r', f'{LBN}日買超/量', '中長線佈局', lambda: N60 / VA.rolling(LBN, min_periods=20).sum().reindex(IDX).replace(0, np.nan)),
    ('cons', '連續買超', '籌碼動能', lambda: CONS),
    ('conc', '集中度', '集中', lambda: CONC5),
    ('brd', '家數差', '集中', lambda: BRD5.astype(float)),
    ('dist', '股價/成本', '成本位置', lambda: DIST),
    ('dtr', '隔日沖率', '短線投機', lambda: DTR),
    ('vr', '量比', '短線投機', lambda: VR),
    ('mar', '融資5日%', '散戶籌碼', lambda: MARCHG5 / MAR_D.shift(5).replace(0, np.nan)),
    ('for', '外資5日/量', '法人', lambda: FOR5 / VOL5.replace(0, np.nan)),
    ('tru', '投信5日/量', '法人', lambda: TRU5 / VOL5.replace(0, np.nan)),
]

def pca_analysis():
    """V1.1：橫斷面排名 Z 分數 → 堆疊最近 PCA_FIT_DAYS 日 → SVD 主成分，檢查 13 因子的共線性"""
    days = IDX[-min(PCA_FIT_DAYS, len(IDX)):]
    act = ACTIVE20[UNIV].reindex(days)
    Zs = {}
    for k, _, _, fn in PCA_FEATS:
        P = fn()[UNIV].reindex(days)
        Zs[k] = panel_z(P, act)
    keys = [k for k, *_ in PCA_FEATS]
    # 整個特徵在期間內幾乎無資料（例如上櫃股票池沒有外資/融資）就剔除
    keys = [k for k in keys if Zs[k].notna().values.mean() > 0.2]
    # 沒有任何可用因子時，交由既有 build_s12 的空 PCA 分支呈現，不讓 np.stack([]) 中斷整頁。
    if not keys:
        return None
    M = np.stack([Zs[k].values.ravel() for k in keys], axis=1)
    rowok = act.values.ravel() & (np.isfinite(M).sum(1) >= len(keys) * 0.6)
    X = np.nan_to_num(M[rowok])
    X = X - X.mean(0)
    if len(X) < 30:
        return None
    U_, S_, Vt = np.linalg.svd(X, full_matrices=False)
    total_var = float(np.square(S_).sum())
    # 零變異或非有限奇異值沒有可解釋的 PCA；回傳既有空結果而非輸出 NaN EV。
    if not np.isfinite(total_var) or total_var <= 0:
        return None
    ev = S_ ** 2 / total_var
    k_ = min(PCA_COMPONENTS, len(keys))
    meta = {k: (n, g) for k, n, g, _ in PCA_FEATS}
    comps = []
    for i in range(k_):
        v = Vt[i]
        j = np.argmax(np.abs(v)); v = v if v[j] > 0 else -v          # 方向：最大載荷為正
        g = Counter()
        for kk, w in zip(keys, v):
            g[meta[kk][1]] += w ** 2
        top = sorted(zip(keys, v), key=lambda x: -abs(x[1]))[:3]
        comps.append(dict(name=f'PC{i + 1}：{g.most_common(1)[0][0]}', ev=float(ev[i]), vec=v,
                          top='、'.join(f'{meta[kk][0]}({w:+.2f})' for kk, w in top)))
    # np.corrcoef 對單一因子回傳 scalar；轉成 1×1 以維持下游二維索引契約。
    corr = np.atleast_2d(np.corrcoef(X, rowvar=False))
    # 最新一日分數
    zl = pd.DataFrame({k: Zs[k].iloc[-1] for k in keys}).fillna(0)
    scores = {f'pc{i + 1}': pd.Series(zl.values @ comps[i]['vec'], index=zl.index) for i in range(k_)}
    return dict(keys=keys, meta=meta, comps=comps, corr=corr, scores=scores, n=len(X))

def build_s12():
    act = N20.iloc[-1][UNIV].abs().sort_values(ascending=False)
    codes = [c for c in act.index if act[c] > 0][:100]
    rows = matrix_rows(codes)
    pa = pca_analysis()
    pcs, load, corr = [], [], []
    if pa:
        for r in rows:
            for k_, sv in pa['scores'].items():
                r[k_] = f_(sv.get(r['code']))
        pcs = [dict(k=f'pc{i + 1}', name=c_['name'], ev=c_['ev'], top=c_['top']) for i, c_ in enumerate(pa['comps'])]
        load = [dict(f=pa['meta'][k][0], g=pa['meta'][k][1], **{f'pc{i + 1}': float(c_['vec'][j]) for i, c_ in enumerate(pa['comps'])}) for j, k in enumerate(pa['keys'])]
        corr = [dict(f=pa['meta'][k][0], **{pa['meta'][k2][0]: float(pa['corr'][j, j2]) for j2, k2 in enumerate(pa['keys'])}) for j, k in enumerate(pa['keys'])]
    data = dict(rows=rows, lb=LBN, pcs=pcs, load=load, corr=corr, cf=[pa['meta'][k][0] for k in pa['keys']] if pa else [], npca=pa['n'] if pa else 0)
    body = '''<div class="kpis" id="k"></div>
<div class="grid"><div class="card"><h2>🧭 主成分（PCA）解釋變異</h2><div class="ch"><canvas id="c1"></canvas></div><div id="t2"></div></div>
<div class="card"><h2>因子載荷（|值|越大＝該成分越由此因子決定）</h2><div id="t3"></div></div></div>
<div class="card"><h2>因子相關矩陣（高度相關＝資訊重複）</h2><div id="t4"></div></div>
<div class="card"><h2>主力行為矩陣（依 20 日淨買超絕對值前 100 檔，色塊＝相對強度）</h2><div id="t1"></div></div>'''
    js = r'''
table('t1',[{k:'stk',t:'股票'},{k:'n5',t:'5日買超｜短期資金',f:'n',c:1,heat:1},{k:'n10',t:'10日｜波段',f:'n',c:1,heat:1},{k:'n20',t:'20日｜中期',f:'n',c:1,heat:1},{k:'n60',t:`${D.lb}日｜長期`,f:'n',c:1,heat:1},
 {k:'cons',t:'連續性',f:'n',heat:'pos'},{k:'conc',t:'集中度',f:'p',c:1,heat:1},{k:'brd',t:'家數差',f:'n',c:1,heat:1},{k:'cost',t:'庫存成本',f:'f'},{k:'px',t:'現價',f:'f'},{k:'dist',t:'股價/成本',f:'p',c:1,heat:1},
 {k:'dtr',t:'隔日沖率',f:'p',heat:'pos'},{k:'sig',t:'轉向(3日內)',f:'tag',tc:'sigc'},{k:'vr',t:'量比',f:'f',heat:'pos'},{k:'mc',t:'融資5日',f:'n',c:1,heat:1},{k:'f5',t:'外資5日',f:'n',c:1,heat:1},{k:'t5',t:'投信5日',f:'n',c:1,heat:1}].concat(D.pcs.map(p=>({k:p.k,t:p.name,f:'f',d:2,c:1,heat:1}))),D.rows,{h:760});
if(D.pcs.length){
 kpis('k',D.pcs.map(p=>[p.name,fmtP(p.ev,0)]).concat([['前 '+D.pcs.length+' 成分合計',fmtP(D.pcs.reduce((a,p)=>a+p.ev,0),0),'up'],['擬合樣本',fmtN(D.npca)]]));
 mk('c1',{type:'bar',data:{labels:D.pcs.map(p=>p.name),datasets:[{label:'解釋變異比例',data:D.pcs.map(p=>p.ev),backgroundColor:[UP,AC,BL,DN,MU]}]},options:{plugins:{legend:{display:false}},scales:{x:ax(),y:ax({ticks:{color:MU,callback:v=>fmtP(v,0)}})}}});
 table('t2',[{k:'name',t:'成分'},{k:'ev',t:'解釋變異',f:'p'},{k:'top',t:'主要因子（載荷）'}],D.pcs,{sort:null});
 table('t3',[{k:'f',t:'因子'},{k:'g',t:'群組'}].concat(D.pcs.map(p=>({k:p.k,t:p.name,f:'f',d:2,c:1,heat:1}))),D.load,{sort:null});
 table('t4',[{k:'f',t:''}].concat(D.cf.map(f=>({k:f,t:f,f:'f',d:2,c:1,heat:1}))),D.corr,{sort:null,h:520});
}else{$('k').innerHTML='<p class=note>樣本不足，未執行 PCA。</p>'}'''
    method = f'''把單一「主力買賣超」擴充成多因子矩陣：5／10／20／60 日買超（短期資金、波段、中期、長期）、買超連續性、集中度、買賣家數差、分點成本、股價/成本距離、隔日沖率、主力轉向、量比（資金確認）、融資變化（散戶籌碼）、外資/投信（法人確認）。
色塊為同欄相對強度：紅＝偏多、綠＝偏空、金＝數值高低。
<ul><li><b>V1.1 主成分分析（PCA）</b>：13 個數值因子彼此高度相關（例如 5／10 日買超、集中度與家數差），直接加總會重複計分。
每日把 13 個數值因子做橫斷面排名 Z 分數（對極端值穩健），堆疊最近 {PCA_FIT_DAYS} 日後以 SVD 萃取前 {PCA_COMPONENTS} 個互相獨立的主成分，
並依載荷最大的因子群組自動命名（如「籌碼動能」「短線投機」「法人」）。表格最右欄為各股最新的成分分數，判讀時看這幾個分數即可。</li>
<li>相關矩陣可檢查哪些因子資訊重複；買超類因子改用「÷ 同期成交量」以消除大小型股的規模差異。</li></ul>'''
    return save_page(12, 'behavior_matrix', '十二', '主力行為矩陣', '13 個數值因子一次看', method, body, js, data, f'{len(rows)} 檔' + (f'；前 {len(pcs)} 成分解釋 {sum(p["ev"] for p in pcs):.0%} 變異' if pcs else ''))

build_s12()

In [ ]:
# ============ S13 主力籌碼 Score ============
def tier(s):
    if s is None or not np.isfinite(s): return None
    return '強勢主力布局' if s >= 80 else '偏多' if s >= 65 else '中性' if s >= 50 else '偏空' if s >= 35 else '主力撤退/弱勢'

# ---- V1.1：機器學習版籌碼 Score（walk-forward 擴張視窗，嚴格無前視）----
ML_FEATS = [('a', '5日買超', S_A, 18), ('b', '20日買超', S_B, 20), ('c', '集中度', S_C, 12), ('d', '追蹤分點共識', S_D, 8),
            ('e', '成本支撐', S_E, 15), ('f', '外資同步', S_F, 10), ('g', '融資下降', S_G, 8), ('h', '技術突破', S_H, 9)]
ML_AUX = [('dtr', '隔日沖率', DTR.fillna(0)), ('ret20', '20日漲跌', RET20), ('vola', '波動度', VOLA), ('liq', '流動性百分位', LIQP)]

def run_ml_score():
    U = UNIV
    T = len(IDX)
    names = [n for _, n, _, _ in ML_FEATS] + [n for _, n, _ in ML_AUX]
    try:
        h = int(ML_HORIZON)
        if isinstance(ML_HORIZON, (bool, np.bool_)) or float(ML_HORIZON) != h:
            h = 0
    except (TypeError, ValueError, OverflowError):
        h = 0
    if h < 1:
        info = dict(ok=False, names=names, imp=[], ic_ml=None, ic_rule=None, n_fit=0,
                    msg='ML_HORIZON 必須是正整數')
        return pd.DataFrame(np.nan, index=IDX, columns=U).reindex(columns=CODES), info
    warm = min(60, T // 4)
    valid = (C[U].notna() & ACTIVE20[U]).to_numpy(copy=True)
    valid[:warm] = False
    # FWD 目前預先提供 5/10/20 日；其他正整數在本 cell 內計算同一標籤，
    # 避免因字典鍵不存在而中斷。
    fwd_h = FWD.get(h)
    if fwd_h is None:
        fwd_h = (AA.shift(-(1 + h)) / ENT - 1).reindex(IDX)
    R_ = fwd_h[U].where(valid)
    XR = R_.sub(R_.mean(axis=1), axis=0).values                     # 同日超額報酬
    F = np.stack([(P[U].values / w) for _, _, P, w in ML_FEATS] + [P[U].values for _, _, P in ML_AUX], axis=-1)   # T × N × k
    out = np.full((T, len(U)), np.nan)
    info = dict(ok=False, names=names, imp=[], ic_ml=None, ic_rule=None, n_fit=0, msg='')
    if not ML_ENABLE:
        info['msg'] = 'ML_ENABLE=False'; return pd.DataFrame(out, index=IDX, columns=U).reindex(columns=CODES), info
    try:
        from sklearn.ensemble import RandomForestClassifier
    except ImportError:
        info['msg'] = '未安裝 scikit-learn'; return pd.DataFrame(out, index=IDX, columns=U).reindex(columns=CODES), info
    gap = h + 2                                                        # 標籤需要 t+1 進場、t+1+h 出場才知道
    start = warm + ML_MIN_TRAIN + gap                              # 前 gap 日的標籤尚未完結，不列入有效訓練日
    if start >= T:
        info['msg'] = f'樣本不足（需至少 {start} 交易日）'; return pd.DataFrame(out, index=IDX, columns=U).reindex(columns=CODES), info
    imp_hist, model, last_tr_end = [], None, None
    for i in range(start, T, ML_RETRAIN):
        tr_end = i - gap
        m = valid[warm:tr_end] & np.isfinite(XR[warm:tr_end])
        X = np.nan_to_num(F[warm:tr_end][m]); y = (XR[warm:tr_end][m] > 0).astype(int)
        if len(y) < 500 or y.min() == y.max():
            continue
        ms = min(1.0, 120_000 / len(y))
        model = RandomForestClassifier(n_estimators=120, max_depth=6, min_samples_leaf=100, max_samples=ms if ms < 1 else None,
                                       n_jobs=-1, random_state=7).fit(X, y)
        imp_hist.append(dict(d=IDX[i].strftime('%Y/%m/%d'), imp=model.feature_importances_.tolist()))
        info['n_fit'] += 1; last_tr_end = tr_end
        blk = slice(i, min(i + ML_RETRAIN, T))                         # 整段一次預測（比逐日快）
        vb = valid[blk]
        if vb.sum():
            pr = model.predict_proba(np.nan_to_num(F[blk][vb]))[:, 1]
            ob = out[blk]; ob[vb] = pr; out[blk] = ob
    MLP = pd.DataFrame(out, index=IDX, columns=U)
    MLS = (MLP.rank(axis=1, pct=True) * 100)                            # 0～100：當日機率的橫斷面百分位
    # 樣本外 Rank IC 比較（同一批預測日）
    ic_m, ic_r, ic_d = [], [], []
    for t in range(start, T):
        a, b_, r_ = MLP.iloc[t], SCORE[U].iloc[t], pd.Series(XR[t], index=U)
        ok = a.notna() & b_.notna() & r_.notna()
        if ok.sum() >= 8:
            ic_m.append(a[ok].rank().corr(r_[ok].rank())); ic_r.append(b_[ok].rank().corr(r_[ok].rank())); ic_d.append(IDX[t])
    tstat = lambda x: tstat_of(nonoverlap(pd.Series(x, index=ic_d), h))   # V1.5：間隔 ≥ h 日的不重疊抽樣
    pred_rows = np.isfinite(out).any(axis=1)
    info.update(ok=bool(pred_rows.any()), imp=imp_hist, ic_ml=float(np.nanmean(ic_m)) if ic_m else None, ic_rule=float(np.nanmean(ic_r)) if ic_r else None,
                ic_ml_t=tstat(ic_m), ic_rule_t=tstat(ic_r),
                first=(IDX[np.flatnonzero(pred_rows)[0]].strftime('%Y-%m-%d') if pred_rows.any() else None))
    if model is not None:
        fi_all = model.feature_importances_
        fi = fi_all[:len(ML_FEATS)]
        total_fi = fi_all.sum()
        # 僅顯示八大構面在全部 12 個特徵中的重要度占比；不把輔助特徵重新歸一成 100。
        info['w'] = (fi / total_fi * 100).tolist() if total_fi > 0 else None
        # V1.4：permutation importance（AUC 下降）——用最近一次模型「沒訓練過、且標籤已實現」的列評估，
        # 比 impurity importance 不偏好高基數特徵，也直接反映樣本外的預測貢獻
        lo_, hi_ = last_tr_end, T - h - 1
        info['pimp_msg'] = '驗證樣本不足（最近一次重訓後已實現標籤的資料太少）'
        if hi_ > lo_:
            mv = valid[lo_:hi_] & np.isfinite(XR[lo_:hi_])
            Xv = np.nan_to_num(F[lo_:hi_][mv]); yv = (XR[lo_:hi_][mv] > 0).astype(int)
            if len(yv) >= 300 and yv.min() != yv.max():
                from sklearn.inspection import permutation_importance
                from sklearn.metrics import roc_auc_score
                pi = permutation_importance(model, Xv, yv, scoring='roc_auc', n_repeats=5, random_state=7, n_jobs=-1)
                info['pimp'] = [dict(f=n_, m=float(a), s=float(b)) for n_, a, b in zip(names, pi.importances_mean, pi.importances_std)]
                info['auc'] = float(roc_auc_score(yv, model.predict_proba(Xv)[:, 1])); info['n_val'] = int(len(yv))
                info['pimp_msg'] = ''
    fm = lambda v: '—' if v is None else f'{v:+.3f}'
    print(f"ML Score：walk-forward 訓練 {info['n_fit']} 次；樣本外 Rank IC  ML {fm(info['ic_ml'])}  vs 規則版 {fm(info['ic_rule'])}" if info['n_fit'] else 'ML Score 未執行：樣本不足')
    return MLS.reindex(columns=CODES), info

ML_SCORE, ML_INFO = run_ml_score()

def build_s13():
    s = SCORE.iloc[-1][UNIV].dropna().sort_values(ascending=False)
    comp = dict(a=S_A, b=S_B, c=S_C, d=S_D, e=S_E, f=S_F, g=S_G, h=S_H)
    rows = []
    for c in s.index:
        t = tier(s[c])
        r = dict(stk=stk(c), score=s[c], t=t, tc=tone(t), chip=bool(np.isfinite(FOR5[c].iloc[-1]) or np.isfinite(MARCHG5[c].iloc[-1])))
        for k, P in comp.items():
            r[k] = float(P[c].iloc[-1])
        r['ml'] = f_(ML_SCORE[c].iloc[-1]); r['mlt'] = tier(r['ml']); r['mltc'] = tone(r['mlt']) if r['mlt'] else 'g'
        rows.append(r)
    tiers = ['強勢主力布局', '偏多', '中性', '偏空', '主力撤退/弱勢']
    hist = [sum(1 for r in rows if r['t'] == t) for t in tiers]
    trend = {c: SCORE[c].iloc[-20:].tolist() for c in s.index[:10]}
    mli = {k: v for k, v in ML_INFO.items() if k != 'imp'}
    mli['imp'] = ML_INFO['imp'][-12:]
    mli['fixed'] = [w for *_, w in ML_FEATS]; mli['fn'] = [n for _, n, _, _ in ML_FEATS]
    data = dict(mlh=ML_HORIZON, ml=mli, rows=rows, tiers=tiers, hist=hist, trend=trend, d=[x.strftime('%m/%d') for x in IDX[-20:]], names={c: stk(c) for c in s.index[:10]})
    body = '''
<div class="kpis" id="k"></div>
<div class="grid"><div class="card"><h2>分數等級分布</h2><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>前 10 名近 20 日分數走勢</h2><div class="ch"><canvas id="c3"></canvas></div></div></div>
<div class="card"><h2>前 20 名分數組成</h2><div class="ch lg"><canvas id="c2"></canvas></div></div>
<div class="card"><h2>🤖 機器學習版 Score：權重對照與樣本外驗證</h2><div class="kpis" id="k2"></div><p class="note" id="mlnote"></p>
<div class="grid"><div class="ch"><canvas id="c4"></canvas></div><div class="ch"><canvas id="c5"></canvas></div></div>
<h3>Permutation importance（最近一次模型，在未訓練過的已實現樣本上：打亂該特徵後 AUC 下降多少）</h3><div id="t2"></div></div>
<div class="card"><h2>主力籌碼 Score 排名（規則版 ＋ ML 版）</h2><div id="t1"></div></div>'''
    js = r'''
const R=D.rows;kpis('k',D.tiers.map((t,i)=>[t,D.hist[i],i<2?'up':i>2?'dn':'']));
mk('c1',{type:'doughnut',data:{labels:D.tiers,datasets:[{data:D.hist,backgroundColor:[UP,'#ff8a65',AC,'#4ade80',DN],borderColor:'#111821'}]}});
const parts=[['a','5日買超(18)',UP],['b','20日買超(20)','#ff8a65'],['c','集中度(12)',AC],['d','追蹤分點共識(8)','#ffd54f'],['e','成本支撐(15)',BL],['f','外資同步(10)','#b07cff'],['g','融資下降(8)','#4dd0e1'],['h','技術突破(9)','#9ccc65']];
const tp=R.slice(0,20);
mk('c2',{type:'bar',data:{labels:tp.map(r=>r.stk),datasets:parts.map(p=>({label:p[1],data:tp.map(r=>r[p[0]]),backgroundColor:p[2]}))},options:{indexAxis:'y',scales:{x:ax({stacked:true,max:100}),y:ax({stacked:true})}}});
const pal=[UP,AC,BL,DN,'#b07cff','#ff8a65','#4dd0e1','#9ccc65','#ffd54f','#f48fb1'];
mk('c3',{type:'line',data:{labels:D.d,datasets:Object.keys(D.trend).map((c,i)=>({label:D.names[c],data:D.trend[c],borderColor:pal[i%10],pointRadius:0,borderWidth:1.5}))},options:{scales:{x:ax(),y:ax({min:0,max:100})}}});
const M=D.ml;
if(M.ok){kpis('k2',[['樣本外 IC：ML',fmtN(M.ic_ml,3),(M.ic_ml||0)>(M.ic_rule||0)?'up':''],['ML IC t 值',fmtN(M.ic_ml_t,2)],['樣本外 IC：規則',fmtN(M.ic_rule,3)],['規則 IC t 值',fmtN(M.ic_rule_t,2)],['重新訓練次數',M.n_fit],['首個預測日',M.first],['最近模型驗證 AUC',fmtN(M.auc,3),(M.auc||0)>0.52?'up':''],['驗證樣本',fmtN(M.n_val)]]);
 $('mlnote').textContent='左：固定權重 vs 最近一次模型的八大特徵重要度占比（四項輔助特徵未畫入，八項合計可小於 100）；右：各次 walk-forward 訓練的特徵重要度變化。ML Score = 模型預測「未來 '+D.mlh+' 日跑贏同日股票池」機率的當日百分位（0～100）。';
 mk('c4',{type:'bar',data:{labels:M.fn,datasets:[{label:'固定權重',data:M.fixed,backgroundColor:'#33465a'},{label:'ML八大特徵重要度占比',data:M.w,backgroundColor:UP}]},options:{indexAxis:'y'}});
 const pal2=[UP,'#ff8a65',AC,'#ffd54f',BL,'#b07cff','#4dd0e1','#9ccc65','#f48fb1','#7d8b99','#ce93d8','#a1887f'];
 mk('c5',{type:'line',data:{labels:M.imp.map(x=>x.d),datasets:M.names.map((n,i)=>({label:n,data:M.imp.map(x=>x.imp[i]),borderColor:pal2[i%12],pointRadius:2,borderWidth:1.5}))}});
 if(M.pimp)table('t2',[{k:'f',t:'特徵'},{k:'m',t:'AUC 下降（平均）',f:'f',d:4,c:1,heat:1},{k:'s',t:'標準差',f:'f',d:4}],M.pimp,{sort:'m'});else $('t2').innerHTML='<p class=note>'+(M.pimp_msg||'未計算')+'</p>';
}else{$('k2').innerHTML='<p class=note>ML Score 未執行：'+(M.msg||'樣本不足')+'</p>'}
table('t1',[{k:'stk',t:'股票'},{k:'score',t:'總分',f:'f',d:1,heat:'pos'},{k:'t',t:'等級',f:'tag',tc:'tc'},{k:'ml',t:'ML Score',f:'f',d:1,heat:'pos'},{k:'mlt',t:'ML 等級',f:'tag',tc:'mltc'}].concat(parts.map(p=>({k:p[0],t:p[1],f:'f',d:1}))).concat([{k:'chip',t:'有法人/融資資料',f:'ck'}]),R);'''
    method = f'''依文件權重組成 100 分：
<ul><li>分點 5 日買超 <code>18</code>、20 日買超 <code>20</code>（淨買超個股間的百分位排名）</li>
<li>分點集中度 <code>12</code>（5 日集中度 {CONC_SCORE_FULL:.0%} 滿分）、追蹤分點 5 日淨買分點差 <code>8</code></li>
<li>主力成本支撐 <code>15</code>（現價在成本 0～+8% 滿分；−3%～0 或 +8～15% 得 10；+15～25% 得 5）</li>
<li>外資同步買超 <code>10</code>（5 日買超且連買 ≥3 日滿分）、融資下降 <code>8</code>（無資料給一半中性分）</li>
<li>技術面突破 <code>9</code>：站上 MA20、MA5&gt;MA20、接近 20 日高點各 3 分（還原價）</li>
<li>優化：5/20 日買超排名只在股票池內比較；主力成本改用<b>庫存成本</b>（含賣出扣減、半衰期 {COST_HALFLIFE} 日）；{f'隔日沖比例 ≥{DT_FLAG_RATIO:.0%} 的個股，5 日買超與集中度不給分（文件第五層：隔日沖買超不能視為布局）' if DT_PENALTY else '未啟用隔日沖扣分'}。</li></ul>
等級：80～100 強勢主力布局／65～79 偏多／50～64 中性／35～49 偏空／0～34 主力撤退/弱勢。
<ul><li><b>V1.1 機器學習版 Score</b>：固定權重是人為設定。把上述 8 個構面（各除以滿分標準化）加上隔日沖率、20 日漲跌、波動度、流動性 4 個輔助特徵，
以 <b>Random Forest</b> 預測「未來 {ML_HORIZON} 日報酬是否跑贏同日股票池」。採 <b>walk-forward 擴張視窗</b>：每 {ML_RETRAIN} 個交易日重新訓練，訓練資料只用到預測日前 {ML_HORIZON + 2} 天（標籤已實現），
完全沒有前視偏差；至少累積 {ML_MIN_TRAIN} 個交易日才開始預測。</li>
<li>ML Score = 預測機率的當日橫斷面百分位 × 100；圖中的「ML 八大特徵重要度占比」= 最近一次模型對八大構面的 impurity importance ÷ 全部 12 個特徵重要度總和 × 100，四項輔助特徵未畫入圖中，因此八項合計可能小於 100；這不是因果係數或可直接採用的最優權重。</li>
<li>樣本外 Rank IC 同時計算 ML 與規則版，ML 不一定勝出——兩者都列出，由數據決定要用哪一版。S15 回測同時驗證兩者。資料越長越可靠（文件建議 5 年，可把 LOOKBACK_YEARS 調高）。</li></ul>'''
    return save_page(13, 'chip_score', '十三', '主力籌碼 Score', '八大構面加權評分 0～100', method, body, js, data, f'{hist[0]} 檔強勢主力布局')

build_s13()

In [ ]:
# ============ S14 分點量化選股漏斗 ============
def build_s14():
    U = [c for c in UNIV if (NET_T[c].iloc[-20:] != 0).any()]
    def v(P, c):
        # 缺欄、非數值或 NaN 均視為 N/A；避免缺資料被誤轉成 False/True。
        try:
            x = P[c].iloc[-1]
            return float(x) if np.isfinite(x) else None
        except (KeyError, TypeError, ValueError):
            return None
    def volume_pass(M, V, c):
        # Cell 9 的 material() 對 V.fillna(0) 計算；零/缺成交量在 S14 應標成 N/A。
        vol, flag = v(V, c), v(M, c)
        return None if vol is None or vol <= 0 or flag is None else bool(flag)
    def ma_pass(c):
        x, m20, m5 = v(CJ, c), v(MA20, c), v(MA5, c)
        return None if any(q is None for q in (x, m20, m5)) else bool(x > m20 and m5 > m20)
    steps = [
        ('5日有效買超', lambda c: volume_pass(MAT5, VOL5, c)),
        ('20日有效買超', lambda c: volume_pass(MAT20, VOL20, c)),
        (f'集中度 ≥ {CONC_MID:.0%}', lambda c: None if v(CONC5, c) is None else v(CONC5, c) >= CONC_MID),
        ('成本位置 −5%～+10%', lambda c: None if v(DIST, c) is None else -0.05 <= v(DIST, c) <= 0.10),
        (f'排除隔日沖（≥{DT_FLAG_RATIO:.0%}）', lambda c: not (v(DTR, c) is not None and v(DTR, c) >= DT_FLAG_RATIO)),
        ('外資連買 ≥3日', lambda c: None if v(FORCONS, c) is None else v(FORCONS, c) >= 3),
        ('融資下降', lambda c: None if v(MARCHG5, c) is None else v(MARCHG5, c) < 0),
        ('MA 趨勢多頭', ma_pass),
        ('RSI 40～75', lambda c: None if v(RSI14, c) is None else 40 <= v(RSI14, c) <= 75),
        ('量能放大', lambda c: None if v(VR, c) is None else v(VR, c) >= 1.0),
    ]
    # ---- V1.1 Z-Score 軟性評分：每一關改成連續指標，排名常態化後取平均 ----
    L_ = IDX[-1]
    MP = MAR_D.shift(5).replace(0, np.nan)
    metrics = [  # 與 steps 一一對應，數值越大越好
        (N5 / VOL5.replace(0, np.nan)).loc[L_], (N20 / VOL20.replace(0, np.nan)).loc[L_], CONC5.loc[L_],
        -(DIST.loc[L_] - 0.025).abs(), -DTR.loc[L_], FORCONS.loc[L_], -(MARCHG5 / MP).loc[L_],
        ((CJ / MA20 - 1) + (MA5 / MA20 - 1)).loc[L_] / 2, -(RSI14.loc[L_] - 57.5).abs(), VR.loc[L_]]
    ZM = pd.DataFrame({i: rank_z(m[U]) for i, m in enumerate(metrics)})
    COMP = ZM.mean(axis=1, skipna=True)
    DTV_ = DTR.loc[L_].reindex(U).fillna(0) >= DT_FLAG_RATIO                # 隔日沖仍為硬性排除
    SOFT = (COMP >= SOFT_Z_PASS) & ~DTV_
    rows, alive, funnel = [], set(U), [dict(s='股票池（有分點交易）', n=len(U))]
    for name, fn in steps:
        alive = {c for c in alive if fn(c) is not False}
        funnel.append(dict(s=name, n=len(alive)))
    for c in U:
        r = dict(stk=stk(c), score=f_(SCORE[c].iloc[-1]))
        res = [fn(c) for _, fn in steps]
        for i, x in enumerate(res):
            r[f's{i}'] = None if x is None else bool(x)
        r['ps'] = sum(1 for x in res if x); r['ok'] = c in alive
        r['fail'] = sum(1 for x in res if x is False)
        r['z'] = f_(COMP.get(c)); r['soft'] = bool(SOFT.get(c, False))
        for i in range(len(steps)):
            r[f'z{i}'] = f_(ZM.loc[c, i]) if c in ZM.index else None
        r['t'] = '全數通過' if c in alive else ('軟性入選' if r['soft'] else None); r['tc'] = tone(r['t']) if r['t'] else 'g'
        r['near'] = bool(r['soft'] and not r['ok'] and r['fail'] == 1)
        rows.append(r)
    rows.sort(key=lambda r: (-(r['z'] if r['z'] is not None else -9), -r['ps']))
    for i, r in enumerate(rows, 1):
        r['rk'] = i
    funnel.append(dict(s=f'Z 軟性入選（平均 Z ≥ {SOFT_Z_PASS}）', n=int(SOFT.sum())))
    data = dict(rows=rows, funnel=funnel, steps=[s for s, _ in steps], mode=UNIVERSE_MODE, zp=SOFT_Z_PASS)
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>選股漏斗（逐層過濾，N/A 不淘汰）</h2><div class="ch lg"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>最後排名（依 Z 軟性綜合分數）</h2><div class="row"><div class="tabs" id="tb"></div></div><div id="t1"></div></div>'''
    js = r'''
const F=D.funnel;kpis('k',[['股票池',F[0].n],['硬漏斗全數通過',F[F.length-2].n,'up'],['Z 軟性入選',F[F.length-1].n,'up'],['僅差一關落馬但軟性入選',D.rows.filter(r=>r.near).length],['模式',D.mode==='0050'?'0050 成分股':'全部上市櫃'],['通過 ≥8 項',D.rows.filter(r=>r.ps>=8).length]]);
mk('c1',{type:'bar',data:{labels:F.map(f=>f.s),datasets:[{label:'剩餘檔數',data:F.map(f=>f.n),backgroundColor:F.map((_,i)=>i===F.length-1?BL:i===F.length-2?UP:i===0?MU:AC)}]},options:{indexAxis:'y',plugins:{legend:{display:false}},scales:{x:ax(),y:ax()}}});
const base=[{k:'rk',t:'#',f:'n'},{k:'stk',t:'股票'},{k:'t',t:'結果',f:'tag',tc:'tc'},{k:'z',t:'平均Z',f:'f',d:2,c:1,heat:1},{k:'ps',t:'硬性通過數',f:'n',heat:'pos'},{k:'score',t:'Score',f:'f',d:1}];
tabs('tb',['Z 分數明細','硬性過關明細','僅差一關'],t=>{
 if(t==='硬性過關明細')table('t1',base.concat(D.steps.map((s,i)=>({k:'s'+i,t:s,f:'ck'}))),D.rows,{sort:null});
 else table('t1',base.concat(D.steps.map((s,i)=>({k:'z'+i,t:s,f:'f',d:2,c:1,heat:1}))),t==='僅差一關'?D.rows.filter(r=>r.near):D.rows,{sort:null})});'''
    method = f'''對應文件第十四層：<code>{'0050 成分股' if UNIVERSE_MODE == '0050' else '股票池'} → 券商分點 → 5 日買超 → 20 日買超 → 主力集中度 → 主力成本 → 隔日沖排除 → 外資連買 → 融資下降 → MA 趨勢 → RSI → 成交量 → 最後排名</code>。
逐層過濾畫成漏斗；因逐層全通過往往很少，另依「通過條件數 → 主力籌碼 Score」給出完整排名。上櫃股無外資/融資資料者標 N/A，不淘汰。
（可在參數區把 <code>UNIVERSE_MODE</code> 改成 '0050'。）
<ul><li><b>V1.1 Z-Score 軟性評分</b>：二元過濾會因單一指標差 0.1% 就淘汰好股票。每一關改用連續指標（如 5 日買超÷成交量、集中度、距成本區中心的距離、RSI 距 57.5 的距離、量比…），
當日在股票池內做<b>排名常態化 Z 分數</b>（對極端值穩健），取平均為綜合分數；平均 Z ≥ <code>{SOFT_Z_PASS}</code> 即「軟性入選」。缺資料的指標不計入平均。</li>
<li>「排除隔日沖」保留為硬性條件（隔日沖買超不是布局）。漏斗圖最後一條為軟性入選數，「僅差一關」分頁列出硬漏斗只差一關、但綜合分數達標的個股。</li></ul>'''
    return save_page(14, 'stock_funnel', '十四', '分點量化選股漏斗', '籌碼＋技術 選股模型', method, body, js, data, f'{funnel[-2]["n"]} 檔全數通過、{funnel[-1]["n"]} 檔軟性入選')

build_s14()

In [ ]:
# ============ S15 十大分點因子 × 回測驗證 ============
def build_s15():
    U = UNIV
    warm = min(60, len(IDX) // 4)   # 讓 60 日視窗因子有足夠資料再開始計分
    valid = C[U].notna() & ACTIVE20[U]
    valid.iloc[:warm] = False
    F = {
        '①5/10/20日累積有效買超': MAT5 & MAT10 & MAT20,
        f'②主力分點集中度≥{CONC_HIGH:.0%}': CONC5 >= CONC_HIGH,
        '③現價在主力庫存成本之上': CJ >= COST,
        '④連續買超≥3日': CONS >= 3,
        '⑤有效買超且非隔日沖': MAT5 & ~(DTR.fillna(0) >= DT_FLAG_RATIO),
        '⑥買賣家數差≥2': BRD5 >= 2,
        '⑦主力轉向(轉多)3日內': BULL.astype(int).rolling(3, min_periods=1).max() > 0,
        '⑧庫存成本與現價距離≤5%': DIST.abs() <= 0.05,
        '⑨分點強度≥70': STR >= 70,
        '⑩分點+外資+融資交叉': MAT5 & (FOR5 > 0) & (MARCHG5 < 0),
        '✕主力轉空(反向)': BEAR.astype(int).rolling(3, min_periods=1).max() > 0,
        'Score≥80': SCORE >= 80,
        'ML Score≥80(樣本外)': ML_SCORE >= 80,
    }
    # 同日超額報酬：個股報酬 − 同一天股票池等權平均。排除「訊號剛好集中在多頭日」的大盤效應
    # V1.5：進場日（t+1）收漲停 → 實務上買不到，因子統計也排除這些訊號（與組合回測一致）
    CANBUY = ~(((CA / CA.shift(1) - 1).reindex(IDX).shift(-1) >= LIMIT_TH).reindex(columns=U).fillna(False))
    valid = valid & CANBUY
    XR = {}
    for h in (5, 10, 20):
        R_ = FWD[h][U].where(valid)
        XR[h] = R_.sub(R_.mean(axis=1), axis=0)

    def stat(mask, h, vmask=None):
        M_ = mask[U].fillna(False).astype(bool) & (valid if vmask is None else vmask)
        m = M_.values
        r = FWD[h][U].values[m]; x = XR[h].values[m]; mae = MAE[h][U].values[m]
        ok = np.isfinite(r) & np.isfinite(x); r, x, mae = r[ok], x[ok], mae[ok]
        if len(r) == 0:
            return dict(n=0, win=None, avg=None, ex=None, t=None, mae=None)
        # t 值：每 h 日取一次（不重疊）的「當日訊號平均超額」，避免重疊持有期高估顯著性
        daily = XR[h].where(M_).mean(axis=1)
        t = tstat_of(nonoverlap(daily, h))   # V1.5：間隔 ≥ h 日的不重疊抽樣
        return dict(n=int(len(r)), win=float((r > 0).mean()), avg=float(r.mean()), ex=float(x.mean()), t=t, mae=float(np.nanmean(mae)))
    ALL = pd.DataFrame(True, index=IDX, columns=CODES)
    base = {h: stat(ALL, h) for h in (5, 10, 20)}
    rows = []
    for name, M in F.items():
        r = dict(f=name)
        for h in (5, 10, 20):
            s_ = stat(M, h)
            r[f'n{h}'] = s_['n']; r[f'w{h}'] = s_['win']; r[f'a{h}'] = s_['avg']; r[f'm{h}'] = s_['mae']
            r[f'e{h}'] = s_['ex']; r[f't{h}'] = s_['t']
            r[f'ew{h}'] = None if s_['win'] is None or base[h]['win'] is None else s_['win'] - base[h]['win']
            r[f'sig{h}'] = '樣本不足' if s_['n'] < 30 else ''
        rows.append(r)
    # V1.4：Benjamini-Hochberg 多重檢定（全部 因子 × 天期 一起校正）。t 值近似常態 → 雙尾 p 值
    from math import erfc, sqrt
    tests = [(i, h, erfc(abs(r[f't{h}']) / sqrt(2))) for i, r in enumerate(rows) for h in (5, 10, 20)
             if r[f't{h}'] is not None and r[f'n{h}'] >= 30]
    m_ = len(tests); qmin = 1.0
    for rank_, (i, h, p) in sorted(enumerate(sorted(tests, key=lambda x: x[2]), 1), key=lambda x: -x[0]):
        qmin = min(qmin, p * m_ / rank_)
        rows[i][f'q{h}'] = qmin
    for r in rows:
        for h in (5, 10, 20):
            q = r.get(f'q{h}')
            if r[f'sig{h}'] == '樣本不足':
                continue
            r[f'sig{h}'] = '顯著' if q is not None and q <= FDR_Q else ('名目顯著' if r[f't{h}'] is not None and abs(r[f't{h}']) >= 2 else '')
    N_TESTS = m_
    # V1.4：保留期驗證——樣本內（保留期前，再空出 11 日避免標籤重疊）vs 最後 HOLDOUT_DAYS 日
    dmask = lambda cond: pd.DataFrame(np.repeat(np.asarray(cond)[:, None], len(U), axis=1), index=IDX, columns=U)
    ho_rows, HO = [], None
    if HOLDOUT_DAYS and len(IDX) > warm + HOLDOUT_DAYS + 120:
        HO = IDX[-HOLDOUT_DAYS]
        _p = IDX.get_loc(HO)
        v_in, v_out = valid & dmask(IDX < IDX[_p - 11]), valid & dmask(IDX >= HO)
        for name, M in F.items():
            si, so = stat(M, 10, v_in), stat(M, 10, v_out)
            same = None if si['ex'] is None or so['ex'] is None else bool(np.sign(si['ex']) == np.sign(so['ex']))
            ho_rows.append(dict(f=name, ei=si['ex'], ti=si['t'], ni=si['n'], eo=so['ex'], wo=so['win'], no=so['n'], same=same))
    # 逐年穩定度
    years = sorted(set(IDX[warm:].year))
    yr_rows = []
    for name, M in F.items():
        r = dict(f=name)
        pos = 0
        for y in years:
            vy = valid & pd.DataFrame(np.repeat((IDX.year == y)[:, None], len(U), axis=1), index=IDX, columns=U)
            sy = stat(M, 10, vy)
            r[f'e{y}'] = sy['ex']; r[f'w{y}'] = sy['win']; r[f'n{y}'] = sy['n']
            pos += 1 if (sy['ex'] is not None and sy['ex'] > 0) else 0
        r['pos'] = f'{pos}/{len(years)}'
        r['stable'] = '穩定' if pos == len(years) else ('不穩定' if pos <= len(years) // 2 else '大致穩定')
        r['sc'] = 'u' if r['stable'] == '穩定' else ('d' if r['stable'] == '不穩定' else 'n')
        yr_rows.append(r)
    # Score 分組（同日超額報酬）
    tiers = ['強勢主力布局', '偏多', '中性', '偏空', '主力撤退/弱勢']
    bounds = [(80, 101), (65, 80), (50, 65), (35, 50), (0, 35)]
    buckets = []
    for t_, (lo, hi) in zip(tiers, bounds):
        M = (SCORE >= lo) & (SCORE < hi)
        b = dict(t=t_)
        for h in (5, 10, 20):
            s_ = stat(M, h); b[f'w{h}'] = s_['win']; b[f'a{h}'] = s_['ex']; b[f'n{h}'] = s_['n']
        buckets.append(b)
    # Rank IC（Score vs 10 日報酬）與 IC t 值（每 10 日取一次）
    ics = []
    for d in IDX[warm:]:
        s_, r_ = SCORE.loc[d, U], FWD[10].loc[d, U]
        ok = s_.notna() & r_.notna()
        if ok.sum() >= 8 and s_[ok].nunique() > 1:
            ics.append((d, s_[ok].rank().corr(r_[ok].rank())))
    ic_t = tstat_of(nonoverlap(pd.Series([x[1] for x in ics], index=[x[0] for x in ics]), 10))
    # 組合回測：每 HOLD_DAYS 日換股，持有 Score 前 N 名；扣除換股的來回交易成本
    # V1.1：另依投入資金計算「價差＋市場衝擊」滑價（中小型股、資金越大越貴），每個資金規模一條淨值
    TICKS = [(10, 0.01), (50, 0.05), (100, 0.1), (500, 0.5), (1000, 1.0), (np.inf, 5.0)]
    def tick(p):
        return next(t for lim, t in TICKS if p < lim)
    def slip(c, d, cap, signal_date=None, topn=None):
        """單邊滑價（比例）與參與率：半個跳動點價差 + K × 日波動 × √(下單張數 ÷ 20 日均量)。cap = 當期實際資金（已複利）"""
        price_date = d
        signal_date = d if signal_date is None else signal_date
        p = CA.at[price_date, c] if price_date in CA.index else np.nan
        if not np.isfinite(p) or p <= 0:
            return 0.002, None
        lots = cap / (topn or TOP_N_PORTFOLIO) / (p * 1000)
        adv = ADV20.at[signal_date, c] if signal_date in ADV20.index else np.nan
        sg = VOLA.at[signal_date, c] if signal_date in VOLA.index else np.nan
        sg = sg if np.isfinite(sg) else 0.025
        part = lots / adv if np.isfinite(adv) and adv > 0 else None
        impact = IMPACT_K * sg * np.sqrt(part) if part is not None else 0.01
        return tick(p) / 2 / p + impact, part

    # V1.4：漲跌停可成交性（以原始收盤漲跌幅近似：收漲停 → 收盤買不到；收跌停 → 收盤賣不掉）
    RAWRET = (CA / CA.shift(1) - 1).reindex(IDX)
    def _limit(c, d, up):
        v = RAWRET.at[d, c] if c in RAWRET.columns else np.nan
        return bool(np.isfinite(v) and (v >= LIMIT_TH if up else v <= -LIMIT_TH))
    def _delay_exit(c, k_):
        """第 k_ 日跌停賣不掉 → 順延到第一個非跌停日（最多 5 日）；回傳順延期間的額外報酬"""
        j = k_
        while j < min(k_ + 5, len(IDX) - 1) and _limit(c, IDX[j], False):
            j += 1
        a0, a1 = AA.at[IDX[k_], c], AA.at[IDX[j], c]
        return (a1 / a0 - 1) if np.isfinite(a0) and np.isfinite(a1) and a0 > 0 else 0.0

    def run_port(SC, thr, k0, hold=None, topn=None, caps=True, limits=True, k_end=None):
        hold, topn = hold or HOLD_DAYS, topn or TOP_N_PORTFOLIO
        eq, eqg, bm, ed, picks, prev, per = [1.0], [1.0], [1.0], [], [], set(), []
        CAPS = CAPITAL_LEVELS if caps else ()
        eqc = {cap: [1.0] for cap in CAPS}; slips = {cap: [] for cap in CAPS}; over = {cap: 0 for cap in CAPS}
        n_lu = n_ld = 0
        k = k0
        while k + 1 + hold < (k_end if k_end is not None else len(IDX)):   # k_end：出場日不得超過此位置（保留期切分用）
            d = IDX[k]
            e0, e1 = IDX[k + 1], IDX[k + 1 + hold]
            cand = SC.loc[d, U].dropna()
            cand = cand[cand >= thr].sort_values(ascending=False)
            sel_ = []
            for c in cand.index:
                if limits and c not in prev and _limit(c, e0, True):   # 新買進遇漲停 → 買不到，改買次一名
                    n_lu += 1; continue
                sel_.append(c)
                if len(sel_) >= topn: break
            # 上期持股本期要賣出、但進場日（＝上期出場日）跌停 → 順延出場，額外報酬計入本期
            drag = 0.0
            if limits:
                for c in prev - set(sel_):
                    if _limit(c, e0, False):
                        n_ld += 1; drag += _delay_exit(c, k + 1) / topn
            allr = (AA.loc[e1, U] / AA.loc[e0, U] - 1).where(valid.loc[d, U]).dropna()
            pr = (AA.loc[e1, sel_] / AA.loc[e0, sel_] - 1).dropna() if sel_ else pd.Series(dtype=float)
            gross = (pr.mean() if len(pr) else 0.0) + drag
            cur = set(sel_)
            chg = (cur - prev) | (prev - cur)
            turnover = len(chg) / (2 * max(topn, 1))   # 換掉的部位比例
            net = gross - turnover * TC_ROUNDTRIP
            for cap in CAPS:
                sl = [slip(c, e0, cap * eqc[cap][-1], d, topn) for c in chg]       # V1.4：資金隨淨值複利
                cost = sum(TC_ROUNDTRIP / 2 + x for x, _ in sl) / max(topn, 1)
                eqc[cap].append(eqc[cap][-1] * (1 + gross - cost))
                slips[cap] += [x for x, _ in sl]
                over[cap] += sum(1 for _, pt in sl if pt is not None and pt > MAX_PARTICIPATION)
            eqg.append(eqg[-1] * (1 + gross)); eq.append(eq[-1] * (1 + net))
            bm.append(bm[-1] * (1 + (allr.mean() if len(allr) else 0)))
            ed.append(e1.strftime('%Y/%m/%d')); per.append(net)
            picks.append(dict(date=d.strftime('%Y-%m-%d'), stks='、'.join(stk(c) for c in sel_) or f'（無 ≥{thr} 分）', ret=gross, net=net, to=turnover,
                              bm=allr.mean() if len(allr) else None))
            prev = cur
            k += hold
        return dict(eq=eq, eqg=eqg, bm=bm, ed=ed, picks=picks, per=per, eqc=eqc, slips=slips, over=over, n_lu=n_lu, n_ld=n_ld, hold=hold)

    def port_stats(P):
        ppy_ = 245 / P['hold']; e_ = pd.Series(P['eq']); per_ = pd.Series(P['per'])
        return dict(tot=float(e_.iloc[-1] - 1), ann=float(e_.iloc[-1] ** (ppy_ / max(len(per_), 1)) - 1) if len(per_) else None,
                    shp=float(per_.mean() / per_.std(ddof=1) * np.sqrt(ppy_)) if len(per_) > 2 and per_.std(ddof=1) > 0 else None,
                    mdd=float((e_ / e_.cummax() - 1).min()), n=len(per_))

    P0 = run_port(SCORE, 50, warm)
    # V1.4：參數敏感度（持有天數 × 檔數），只扣固定交易成本；目前設定以 ✓ 標示
    sens = [dict(hold=hd, topn=tn, cur=(hd == HOLD_DAYS and tn == TOP_N_PORTFOLIO), **port_stats(run_port(SCORE, 50, warm, hold=hd, topn=tn, caps=False)))
            for hd in SENS_HOLD for tn in SENS_TOPN]
    ho_port = None
    if HO is not None:
        _pi = port_stats(run_port(SCORE, 50, warm, caps=False, k_end=IDX.get_loc(HO)))
        _po = port_stats(run_port(SCORE, 50, IDX.get_loc(HO), caps=False))
        ho_port = dict(i_ann=_pi['ann'], i_shp=_pi['shp'], o_tot=_po['tot'], o_ann=_po['ann'], o_shp=_po['shp'], o_n=_po['n'])
    eq, eqg, bm, ed, picks, per = P0['eq'], P0['eqg'], P0['bm'], P0['ed'], P0['picks'], P0['per']
    ppy = 245 / HOLD_DAYS
    def perf(e, n):
        return float(e[-1] ** (ppy / max(n, 1)) - 1) if n else None
    caps = []
    for cap in CAPITAL_LEVELS:
        e = P0['eqc'][cap]; es = pd.Series(e); pr_ = pd.Series(e).pct_change().dropna()
        caps.append(dict(cap=f'{cap / 1e4:,.0f} 萬', tot=e[-1] - 1, ann=perf(e, len(e) - 1), mdd=float((es / es.cummax() - 1).min()),
                         shp=float(pr_.mean() / pr_.std(ddof=1) * np.sqrt(ppy)) if len(pr_) > 2 and pr_.std(ddof=1) > 0 else None,
                         slip=float(np.mean(P0['slips'][cap])) if P0['slips'][cap] else None, over=P0['over'][cap], eq=e))
    # ML 版 Score 組合（與規則版在同一段樣本外期間比較）
    mlc = None
    if ML_INFO.get('ok'):
        k_ml = IDX.get_loc(pd.Timestamp(ML_INFO['first']))
        PM, PR = run_port(ML_SCORE, 50, k_ml), run_port(SCORE, 50, k_ml)
        mlc = dict(ed=PM['ed'], ml=PM['eq'], rule=PR['eq'], bm=PM['bm'], ml_tot=PM['eq'][-1] - 1, rule_tot=PR['eq'][-1] - 1, bm_tot=PM['bm'][-1] - 1,
                   ml_ann=perf(PM['eq'], len(PM['per'])), rule_ann=perf(PR['eq'], len(PR['per'])))
    eqs = pd.Series(eq); mdd = float((eqs / eqs.cummax() - 1).min())
    bms = pd.Series(bm); bmdd = float((bms / bms.cummax() - 1).min())
    wr = float(np.mean([p['net'] > 0 for p in picks])) if picks else None
    per_s = pd.Series(per)
    ann = perf(eq, len(per)) if per else None
    shp = float(per_s.mean() / per_s.std(ddof=1) * np.sqrt(ppy)) if len(per) > 2 and per_s.std(ddof=1) > 0 else None
    avg_to = float(np.mean([p['to'] for p in picks])) if picks else None
    data = dict(rows=rows, base=base, buckets=buckets, ic=[x[1] for x in ics], icd=[x[0].strftime('%Y/%m/%d') for x in ics],
                icm=float(np.mean([x[1] for x in ics])) if ics else None, ic_t=ic_t, eq=eq, eqg=eqg, bm=bm, ed=['起點'] + ed, mdd=mdd, bmdd=bmdd, wr=wr,
                ann=ann, shp=shp, to=avg_to, tc=TC_ROUNDTRIP, caps=caps, mlc=mlc, ik=IMPACT_K, picks=picks[::-1], hold=HOLD_DAYS, topn=TOP_N_PORTFOLIO, years=years, yr=yr_rows,
                ho=ho_rows, ho_from=HO.strftime('%Y-%m-%d') if HO is not None else None, ho_port=ho_port, sens=sens, fdr=FDR_Q, ntest=N_TESTS, n_lu=P0['n_lu'], n_ld=P0['n_ld'])
    body = '''
<div class="kpis" id="k"></div>
<div class="card"><h2>十大分點因子回測（訊號日收盤後確認 → 隔日收盤進場）</h2>
<div class="row"><div class="tabs" id="tb"></div><span class="note">超額＝個股報酬 − 同日股票池平均（排除大盤漲跌）；「顯著」= 多重檢定校正後 FDR q ≤ <span id="fq"></span>，「名目顯著」= 只有 |t|≥2（未校正，多半是巧合）；MAE＝持有期間平均最大不利幅度</span></div><div id="t1"></div></div>
<div class="card"><h2>🔒 樣本內 vs 保留期（10 日超額）</h2><p class="note" id="hon"></p><div class="kpis" id="k4"></div><div id="t5"></div></div>
<div class="card"><h2>逐年穩定度：10 日超額報酬（每一年都為正才算穩定）</h2><div id="t3"></div></div>
<div class="card"><h2>🎛️ 參數敏感度：持有天數 × 持股檔數（Score ≥ 50 前 N 名，扣固定交易成本）</h2><p class="note">結果若只在單一組合特別好、鄰近組合明顯變差，代表參數過度擬合。</p><div id="t6"></div></div>
<div class="grid"><div class="card"><h2>因子超額報酬</h2><div class="ch lg"><canvas id="c1"></canvas></div></div>
<div class="card"><h2>Score 等級 × 未來超額報酬</h2><div class="ch lg"><canvas id="c2"></canvas></div></div></div>
<div class="grid"><div class="card"><h2>組合淨值：每期 Score 前 N 名（扣交易成本）vs 股票池等權</h2><div class="ch"><canvas id="c3"></canvas></div></div>
<div class="card"><h2>Rank IC（Score vs 未來 10 日報酬）</h2><div class="ch"><canvas id="c4"></canvas></div></div></div>
<div class="card"><h2>💸 資金規模 × 滑價：同一策略在不同資金下的淨值</h2><p class="note" id="capn"></p><div class="grid"><div class="ch"><canvas id="c5"></canvas></div><div id="t4"></div></div></div>
<div class="card"><h2>🤖 規則版 vs ML 版 Score（同一段樣本外期間，固定交易成本）</h2><div class="kpis" id="k3"></div><div class="ch"><canvas id="c6"></canvas></div></div>
<div class="card"><h2>每期持股明細</h2><div id="t2"></div></div>'''
    js = r'''
kpis('k',[['組合淨報酬(扣成本)',fmtP(D.eq[D.eq.length-1]-1),cls(D.eq[D.eq.length-1]-1)],['毛報酬',fmtP(D.eqg[D.eqg.length-1]-1)],['基準報酬',fmtP(D.bm[D.bm.length-1]-1)],
 ['年化(淨)',fmtP(D.ann),cls(D.ann)],['Sharpe(淨)',fmtN(D.shp,2)],['最大回撤',fmtP(D.mdd),'dn'],['基準最大回撤',fmtP(D.bmdd),'dn'],['每期勝率(淨)',fmtP(D.wr)],
 ['平均換手',fmtP(D.to,0)],['來回成本',fmtP(D.tc,2)],['平均 Rank IC',fmtN(D.icm,3)],['IC t 值',fmtN(D.ic_t,2),Math.abs(D.ic_t||0)>=2?'up':''],['漲停買不到(改買次名)',fmtN(D.n_lu)],['跌停順延出場',fmtN(D.n_ld)]]);
if(D.ho.length){$('hon').textContent='保留期自 '+D.ho_from+' 起；樣本內截至保留期前 11 個交易日（避免 10 日標籤重疊）。調整參數時只看樣本內，保留期結果才有參考價值。';
 const P=D.ho_port;if(P)kpis('k4',[['樣本內年化',fmtP(P.i_ann),cls(P.i_ann)],['樣本內 Sharpe',fmtN(P.i_shp,2)],['保留期報酬',fmtP(P.o_tot),cls(P.o_tot)],['保留期年化',fmtP(P.o_ann),cls(P.o_ann)],['保留期 Sharpe',fmtN(P.o_shp,2)],['保留期期數',P.o_n]]);
 table('t5',[{k:'f',t:'因子'},{k:'ei',t:'樣本內超額',f:'p',d:2,c:1,heat:1},{k:'ti',t:'樣本內 t',f:'f',d:2,c:1},{k:'ni',t:'樣本內 n',f:'n'},{k:'eo',t:'保留期超額',f:'p',d:2,c:1,heat:1},{k:'wo',t:'保留期勝率',f:'p'},{k:'no',t:'保留期 n',f:'n'},{k:'same',t:'方向一致',f:'ck'}],D.ho,{sort:null});}
else{$('hon').textContent='期間太短，未切保留期（需 > 暖機 + HOLDOUT_DAYS + 120 日）';}
table('t6',[{k:'hold',t:'持有天數',f:'n'},{k:'topn',t:'持股檔數',f:'n'},{k:'tot',t:'總報酬(淨)',f:'p',c:1,heat:1},{k:'ann',t:'年化(淨)',f:'p',c:1,heat:1},{k:'shp',t:'Sharpe',f:'f',d:2,c:1,heat:1},{k:'mdd',t:'最大回撤',f:'p',c:1},{k:'n',t:'期數',f:'n'},{k:'cur',t:'目前設定',f:'ck'}],D.sens,{sort:null});
function show(h){D.rows.forEach(r=>r['sc'+h]=r['sig'+h]==='顯著'?(r['e'+h]>0?'u':'d'):'g');
 table('t1',[{k:'f',t:'因子'},{k:'n'+h,t:'樣本數',f:'n'},{k:'w'+h,t:h+'日勝率',f:'p',heat:'pos'},{k:'ew'+h,t:'勝率差',f:'p',c:1},{k:'a'+h,t:h+'日平均報酬',f:'p',d:2,c:1},{k:'e'+h,t:'同日超額',f:'p',d:2,c:1,heat:1},{k:'t'+h,t:'t 值',f:'f',d:2,c:1},{k:'q'+h,t:'FDR q 值',f:'f',d:3,tip:'Benjamini-Hochberg 校正後；≤ 門檻才標「顯著」'},{k:'sig'+h,t:'顯著性',f:'tag',tc:'sc'+h},{k:'m'+h,t:'平均MAE',f:'p',d:2,c:1}],D.rows,{sort:null});
 mk('c1',{type:'bar',data:{labels:D.rows.map(r=>r.f),datasets:[{label:h+'日超額報酬',data:D.rows.map(r=>r['e'+h]),backgroundColor:D.rows.map(r=>sc(r['e'+h]||0))}]},
  options:{indexAxis:'y',plugins:{legend:{display:false}},scales:{x:ax({ticks:{color:MU,callback:v=>fmtP(v,1)}}),y:ax()}}});}
$('fq').textContent=D.fdr+'（共 '+D.ntest+' 個檢定）';tabs('tb',['5','10','20'],h=>show(+h));
mk('c2',{type:'bar',data:{labels:D.buckets.map(b=>b.t),datasets:[5,10,20].map((h,i)=>({label:h+'日超額報酬',data:D.buckets.map(b=>b['a'+h]),backgroundColor:[UP,AC,BL][i]}))},options:{scales:{x:ax(),y:ax({ticks:{color:MU,callback:v=>fmtP(v,1)}})}}});
mk('c3',{type:'line',data:{labels:D.ed,datasets:[{label:'Score 前'+D.topn+'名・扣成本',data:D.eq,borderColor:UP,pointRadius:0},{label:'未扣成本',data:D.eqg,borderColor:'rgba(255,77,79,.4)',borderDash:[4,3],pointRadius:0},{label:'股票池等權',data:D.bm,borderColor:MU,pointRadius:0}]}});
mk('c4',{type:'bar',data:{labels:D.icd,datasets:[{label:'Rank IC',data:D.ic,backgroundColor:D.ic.map(sc)}]},options:{plugins:{legend:{display:false}}}});
table('t3',[{k:'f',t:'因子'},{k:'stable',t:'判定',f:'tag',tc:'sc'},{k:'pos',t:'正超額年數'}].concat(D.years.flatMap(y=>[{k:'e'+y,t:y+' 超額',f:'p',d:2,c:1,heat:1},{k:'w'+y,t:y+' 勝率',f:'p'},{k:'n'+y,t:y+' 樣本',f:'n'}])),D.yr,{sort:null});
const CP=[UP,AC,BL,'#b07cff'];
$('capn').textContent='淨值已扣：手續費＋證交稅（'+fmtP(D.tc,2)+' 來回）＋半個跳動點價差＋市場衝擊（'+D.ik+' × 日波動 × √(下單張數 ÷ 20日均量)）。參與率 >10% 的交易次數列於右表。';
mk('c5',{type:'line',data:{labels:D.ed,datasets:D.caps.map((c,i)=>({label:c.cap,data:c.eq,borderColor:CP[i%4],pointRadius:0})).concat([{label:'未扣成本',data:D.eqg,borderColor:'rgba(255,255,255,.25)',borderDash:[4,3],pointRadius:0}])}});
table('t4',[{k:'cap',t:'投入資金'},{k:'tot',t:'淨報酬',f:'p',c:1},{k:'ann',t:'年化',f:'p',c:1},{k:'shp',t:'Sharpe',f:'f',d:2},{k:'mdd',t:'最大回撤',f:'p',c:1},{k:'slip',t:'平均單邊滑價',f:'p',d:2},{k:'over',t:'參與率>10%次數',f:'n'}],D.caps,{sort:null});
if(D.mlc){const m=D.mlc;kpis('k3',[['ML 版報酬',fmtP(m.ml_tot),cls(m.ml_tot)],['規則版報酬',fmtP(m.rule_tot),cls(m.rule_tot)],['股票池等權',fmtP(m.bm_tot)],['ML 年化',fmtP(m.ml_ann)],['規則年化',fmtP(m.rule_ann)]]);
 mk('c6',{type:'line',data:{labels:['起點'].concat(m.ed),datasets:[{label:'ML 版 Score',data:m.ml,borderColor:BL,pointRadius:0},{label:'規則版 Score',data:m.rule,borderColor:UP,pointRadius:0},{label:'股票池等權',data:m.bm,borderColor:MU,pointRadius:0}]}});
}else{$('k3').innerHTML='<p class=note>ML Score 未執行（樣本不足或未安裝 scikit-learn）。</p>'}
table('t2',[{k:'date',t:'訊號日'},{k:'stks',t:'持股'},{k:'ret',t:'毛報酬',f:'p',d:2,c:1},{k:'to',t:'換手',f:'p',d:0},{k:'net',t:'扣成本',f:'p',d:2,c:1},{k:'bm',t:'股票池等權',f:'p',d:2,c:1}],D.picks,{h:360,sort:null});'''
    _fi = FOR_D.dropna(how='all').index
    chip_from = f'{_fi.min():%Y-%m-%d}' if len(_fi) else '（未抓取）'
    method = f'''回答文件最後的核心問題：「哪些券商分點行為，歷史上真的能提高未來 5～20 日上漲機率？」
<ul><li>對文件第十五層列出的 10 個因子（另加主力轉空與 Score≥80）逐日產生訊號，統計未來 5／10／20 日的勝率、平均報酬、<b>同日超額報酬</b>（個股 − 同一天股票池平均，排除「訊號剛好出現在多頭日」的大盤效應）與平均最大不利幅度（MAE）。</li>
<li><b>t 值</b>：每 h 日取一次（不重疊）的當日訊號平均超額來計算，避免重疊持有期把顯著性灌水。</li>
<li><b>V1.4 多重檢定</b>：{len(F)} 個因子 × 3 個天期同時檢定，單看 |t| ≥ 2 必然出現假陽性。改以 Benjamini-Hochberg 控制偽發現率，q ≤ {FDR_Q} 才標「顯著」；只有 |t| ≥ 2 者標「名目顯著」。</li>
<li><b>V1.4 保留期與敏感度</b>：最後 {HOLDOUT_DAYS} 個交易日為保留期，樣本內與保留期分開統計；另列持有天數 {SENS_HOLD} × 檔數 {SENS_TOPN} 的績效表檢查參數是否過度擬合。</li>
<li><b>V1.4 可成交性</b>：新買進標的若進場日收漲停（≥ {LIMIT_TH:.1%}）視為買不到、改買次一名；要賣出的持股若當日收跌停，順延至第一個非跌停日（最多 5 日）出場。滑價依當期淨值（複利後）計算部位大小。V1.5 起因子統計表也排除進場日漲停的訊號。</li>
<li><b>V1.5 存活者偏誤</b>：{SURV['n']}/{SURV['n_all']} 檔普通股有分點交易但抓不到股價（多為下市／轉板），占追蹤分點淨買賣量 {SURV['share']:.1%}，未納入回測{('（量最大：' + '、'.join(SURV['codes'][:8]) + '）') if SURV['codes'] else ''}。占比越高，回測越可能高估（下市股多半是虧損收場）。</li>
<li><b>V1.5 Top50 截尾</b>：各分點列滿 {TRUNC_ROWS} 檔的日子比例：{'、'.join(f'{b} {v:.0%}' for b, v in TRUNC_BR.items())}。觸頂越頻繁，未上榜的交易被當成 0 的偏誤越大。</li>
<li>報酬一律用<b>還原收盤價</b>（含除權息），進場價＝訊號日隔日收盤（分點資料於收盤後才公布），避免前視偏差。</li>
<li>組合回測：每 {HOLD_DAYS} 個交易日挑 Score ≥ 50 的前 {TOP_N_PORTFOLIO} 名等權持有，依換手比例扣除來回交易成本 {TC_ROUNDTRIP:.2%}，計算淨報酬、年化、Sharpe、最大回撤。</li>
<li><b>V1.1 滑價模型</b>：固定 {TC_ROUNDTRIP:.2%} 只含手續費與稅。實際成本另加 ① 半個跳動點價差（依台股升降單位表）② 市場衝擊 ≈ {IMPACT_K} × 日報酬波動 × √(單檔下單張數 ÷ 20 日均量)（平方根法則），
依投入資金 {'、'.join(f'{c / 1e4:,.0f} 萬' for c in CAPITAL_LEVELS)} 各算一條淨值。籌碼集中的中小型股均量低，資金放大後衝擊成本會明顯侵蝕報酬；單檔下單超過均量 {MAX_PARTICIPATION:.0%} 另行計數警示。
（交易所五檔歷史需付費資料，此處以跳動點與成交量估算；取得五檔後可把 <code>slip()</code> 換成實際買賣價差。）</li>
<li><b>ML 版對照</b>：S13 walk-forward 機器學習 Score 只在樣本外期間有值，與規則版在同一段期間並列比較；因子表另含「ML Score≥80(樣本外)」。</li>
<li>逐年穩定度：把樣本依年份切開，同一因子在每一年都要有正的超額報酬，才視為「穩定」，用來排除只在單一行情有效的條件。</li>
<li>樣本數 &lt; 30 的因子標示「樣本不足」，勝率與報酬僅供參考。</li>
<li>⚠️ 樣本 {len(IDX)} 個交易日（{IDX[0]:%Y-%m-%d} ～ {IDX[-1]:%Y-%m-%d}）、5 個分點，股票池是「這些分點交易過的股票」（有選樣偏誤）；外資/融資資料自 {chip_from} 起。前 {warm} 日為暖機期不計入。</li></ul>'''
    best = max((r for r in rows if r['e10'] is not None and r['sig10'] == '顯著' and r['e10'] > 0), key=lambda r: r['e10'], default=None)
    return save_page(15, 'factor_backtest', '十五', '十大分點因子 × 回測驗證', '勝率／平均報酬／最大回撤', method, body, js, data,
                     f'10日顯著超額最高：{best["f"]} {best["e10"]:+.2%}（t={best["t10"]:.1f}）' if best else '無統計顯著的因子')

build_s15()

In [ ]:
# ============ 00 一頁式重點摘要（純 HTML，不需網路） ============
import html as _html

def build_summary():
    E = lambda x: _html.escape(str(x)) if x is not None else '—'
    D = PAGE_DATA
    def n(v, d=0):
        return '—' if v is None or (isinstance(v, float) and not np.isfinite(v)) else f'{v:,.{d}f}'
    def pc(v, d=1, sign=True):
        if v is None or (isinstance(v, float) and not np.isfinite(v)): return '—'
        return f'{v * 100:+.{d}f}%' if sign else f'{v * 100:.{d}f}%'
    def cl(v):
        return '' if v is None or not np.isfinite(v) or v == 0 else ('up' if v > 0 else 'dn')
    def tag(t, tone_=None):
        return f'<span class="pill {tone_ or tone(t)}">{E(t)}</span>' if t else ''
    def tbl(heads, rows, empty='無符合項目'):
        if not rows:
            return f'<p class="none">{empty}</p>'
        th = ''.join(f'<th>{h}</th>' for h in heads)
        tr = ''.join('<tr>' + ''.join(f'<td>{c}</td>' for c in r) + '</tr>' for r in rows)
        return f'<table><thead><tr>{th}</tr></thead><tbody>{tr}</tbody></table>'
    def bar(v, mx=100, color='var(--ac)'):
        w = 0 if v is None else max(0, min(100, v / mx * 100))
        return f'<span class="bar"><i style="width:{w:.0f}%;background:{color}"></i></span>'
    def stats(items):
        return '<div class="st">' + ''.join(f'<div><b class="{c}">{v}</b><span>{k}</span></div>' for k, v, c in items) + '</div>'
    files = {p['num']: p['file'] for p in PAGES}
    cards = {}

    def card(num, title, kpi, body):
        cards[num] = f'''<section class="card"><h2><span class="no">S{num:02d}</span>{title}<a href="{files.get(num, '#')}">詳細 →</a></h2>{kpi}{body}</section>'''

    # S01
    b1 = []
    for b, v in D[1]['br'].items():
        tb = v['top_buy'][0] if v['top_buy'] else None
        ts = v['top_sell'][0] if v['top_sell'] else None
        b1.append([E(b), (E(tb['stk']) + f' <span class="up">{n(tb["net"])}</span>') if tb else '—',
                   (E(ts['stk']) + f' <span class="dn">{n(ts["net"])}</span>') if ts else '—'])
    top5 = N5.iloc[-1][UNIV].sort_values(ascending=False)
    top5 = [[E(stk(c)), f'<span class="up">{n(top5[c])}</span>'] for c in top5.index[:5] if top5[c] > 0]
    card(1, '分點買賣超', '', tbl(['分點', '今日買超第一（張）', '今日賣超第一（張）'], b1) +
         '<h3>追蹤分點合計 5 日買超前 5</h3>' + tbl(['股票', '5日淨買(張)'], top5))
    # S02
    st2 = D[2]['stock']
    card(2, '分點強度分數', stats([('≥80 分個股', sum(1 for r in st2 if r['score'] >= 80), 'up'),
                                  ('≥60 分個股', sum(1 for r in st2 if r['score'] >= 60), ''),
                                  ('≥80 分配對', sum(1 for r in D[2]['pairs'] if r['score'] >= 80), '')]),
         tbl(['股票', '強度', '', '20日淨買'], [[E(r['stk']), n(r['score']), bar(r['score']), f'<span class="{cl(r["n20"])}">{n(r["n20"])}</span>'] for r in st2[:6]]))
    # S03
    p3 = D[3]['pairs']; sts = ['被套/防守', '成本附近', '已有獲利', '獲利了結風險', '已出清']
    near = [r for r in p3 if r['st'] == '成本附近'][:4]
    risk = [r for r in p3 if r['st'] == '獲利了結風險'][:4]
    row3 = lambda r: [E(r['branch']), E(r['stk']), n(r['q20'] or r['qall']), n(r['cost20'] or r['costall'], 2), n(r['px'], 2), f'<span class="{cl(r["d20"] if r["d20"] is not None else r["dall"])}">{pc(r["d20"] if r["d20"] is not None else r["dall"])}</span>']
    card(3, '主力成本區（庫存法）', stats([(s_, sum(1 for r in p3 if r['st'] == s_), {'被套/防守': 'dn', '獲利了結風險': 'up'}.get(s_, '')) for s_ in sts]
                               + [('實際均價占比', pc(D[3]['diag'].get('real_share'), 0, False), '')]),
         '<h3>成本附近（支撐觀察，仍有庫存）</h3>' + tbl(['分點', '股票', '庫存(張)', '成本', '現價', '距成本'], [row3(r) for r in near]) +
         '<h3>獲利了結風險</h3>' + tbl(['分點', '股票', '庫存(張)', '成本', '現價', '距成本'], [row3(r) for r in risk]))
    # S04
    r4 = D[4]['rows']
    brk = sorted([r for r in r4 if r['st'] == '突破成本區'], key=lambda r: r['bdays'] or 99)[:5]
    card(4, '成本密集區', stats([(s_, sum(1 for r in r4 if r['st'] == s_), c) for s_, c in
                                [('突破成本區', 'up'), ('接近成本上緣', ''), ('成本區內', ''), ('低於成本區', 'dn')]]),
         '<h3>突破成本區（依突破天數，越短越新）</h3>' + tbl(['股票', '密集區', '現價', '突破天數', '突破後'],
            [[E(r['stk']), f'{n(r["zl"], 1)}～{n(r["zh"], 1)}', n(r['px'], 2), n(r['bdays']), f'<span class="{cl(r["bret"])}">{pc(r["bret"])}</span>'] for r in brk]))
    # S05
    al = D[5]['alerts']; brs5 = sorted([b for b in D[5]['brs'] if b['ratio'] is not None], key=lambda b: -b['ratio'])
    card(5, '隔日沖辨識', stats([('今日警報', len(al), 'up' if al else ''),
                                ('最像隔日沖', E(brs5[0]['branch']) if brs5 else '—', ''),
                                ('其隔日沖比例', pc(brs5[0]['ratio'], 0, False) if brs5 else '—', '')]),
         tbl(['類型', '分點', '股票', '說明'], [[tag(a['type'], a['tc']), E(a['branch']), E(a['stk']), E(a['note'])] for a in al[:6]], '今日無隔日沖警報'))
    # S06
    mat, types = D[6]['mat'], D[6]['types']
    r6 = []
    for i, b in enumerate(D[6]['br']):
        cnt = {t: mat[t][i] for t in types if t != '不明顯'}
        main = max(cnt, key=cnt.get) if any(cnt.values()) else '不明顯'
        r6.append([E(b), tag(main), ' '.join(f'{t[:2]}{cnt[t]}' for t in types if t != '不明顯' and cnt[t])])
    card(6, '主力類型分類', '', tbl(['分點', '主要行為', '各類配對數'], r6))
    # S07
    r7 = [r for r in D[7]['rows'] if (r['c5'] or 0) > 0][:6]
    card(7, '分點集中度', stats([(t, sum(1 for r in D[7]['rows'] if r['t'] == t), c) for t, c in
                                [('高度集中', 'up'), ('中度集中', 'up'), ('輕度集中', ''), ('賣方集中', 'dn')]]),
         tbl(['股票', '5日集中度', '', '等級'], [[E(r['stk']), pc(r['c5'], 1, False), bar((r['c5'] or 0) * 100, 30), tag(r['t'])] for r in r7]))
    # S08
    r8 = D[8]['mrows'] or D[8]['rows']           # 有全市場家數就用全市場
    pats = ['買盤廣泛', '買方集中、賣方分散', '賣盤廣泛', '賣方集中、買方分散']
    conc8 = sorted([r for r in r8 if r['p'] in ('買方集中、賣方分散', '買盤廣泛')], key=lambda r: (r['p'] != '買方集中、賣方分散', -(r.get('n5', r.get('tr5')) or 0)))[:5]
    card(8, '買賣家數差' + ('（全市場）' if D[8]['mrows'] else '（5 分點）'), stats([(p_, sum(1 for r in r8 if r['p'] == p_), 'up' if '買' in p_[:2] else 'dn') for p_ in pats]),
         tbl(['股票', '型態', '5日家數差', '追蹤分點5日淨買'], [[E(r['stk']), tag(r['p']), n(r['d5']), f'<span class="{cl(r.get("n5", r.get("tr5")))}">{n(r.get("n5", r.get("tr5")))}</span>'] for r in conc8]))
    # S09
    r9 = [r for r in D[9]['rows'] if r['t']][:6]
    card(9, '吸籌偵測', stats([('吸籌', sum(1 for r in D[9]['rows'] if r['t'] == '吸籌'), 'up'), ('疑似吸籌', sum(1 for r in D[9]['rows'] if r['t'] == '疑似吸籌'), '')]),
         tbl(['股票', '判定', '符合條件', '20日漲跌', '5日淨買'], [[E(r['stk']), tag(r['t']), f'{r["ps"]}/{r["av"]}', pc(r['r20']), n(r['n5'])] for r in r9], '今日無吸籌訊號'))
    # S10
    r10 = D[10]['rows'][:6]
    card(10, '出貨偵測', stats([('散戶接盤確認', sum(1 for r in D[10]['rows'] if r['t'] == '出貨警報(散戶接盤)'), 'dn'), ('出貨警報', sum(1 for r in D[10]['rows'] if r['t'] == '出貨警報'), 'dn'), ('疑似出貨', sum(1 for r in D[10]['rows'] if r['t'] == '疑似出貨'), '')]),
         tbl(['股票', '判定', '20日漲跌', '轉賣分點'], [[E(r['stk']), tag(r['t']), f'<span class="{cl(r["r20"])}">{pc(r["r20"])}</span>', f'<span class="sm">{E(r["tb"])}</span>'] for r in r10], '今日無出貨訊號'))
    # S11
    t11 = [r for r in D[11]['rows'] if r['date'] == D[11]['last']]
    card(11, '主力反轉訊號', stats([('今日轉空', sum(1 for r in t11 if r['t'] == '主力轉空'), 'dn'), ('今日轉多', sum(1 for r in t11 if r['t'] != '主力轉空'), 'up'),
                                  ('分點層級訊號', len(D[11]['br']), '')]),
         tbl(['股票', '訊號', '今日淨買', '前20日'], [[E(r['stk']), tag(r['t']), f'<span class="{cl(r["today"])}">{n(r["today"])}</span>', n(r['p20'])] for r in t11[:6]], '今日無反轉訊號'))
    # S12：多因子共振
    r12 = [r for r in D[12]['rows'] if (r['n5'] or 0) > 0 and (r['n20'] or 0) > 0 and (r['conc'] or 0) > 0
           and r['dist'] is not None and -0.05 <= r['dist'] <= 0.10 and (r['dtr'] is None or r['dtr'] < DT_FLAG_RATIO)]
    card(12, '主力行為矩陣', f'<p class="note">篩選：5/20 日皆買超、集中度&gt;0、距成本 −5%～+10%、非隔日沖（&lt;{DT_FLAG_RATIO:.0%}）</p>',
         tbl(['股票', '5日', '20日', '集中度', '距成本'], [[E(r['stk']), n(r['n5']), n(r['n20']), pc(r['conc'], 1, False), pc(r['dist'])] for r in r12[:6]]))
    # S13
    r13 = D[13]['rows']
    card(13, '主力籌碼 Score', stats([(t, h, 'up' if i < 2 else 'dn' if i > 2 else '') for i, (t, h) in enumerate(zip(D[13]['tiers'], D[13]['hist']))]),
         tbl(['股票', 'Score', '', '等級', 'ML Score'], [[E(r['stk']), n(r['score'], 1), bar(r['score']), tag(r['t']), n(r.get('ml'), 0)] for r in r13[:8]]) +
         (f'<p class="note">ML 樣本外 Rank IC {n(D[13]["ml"]["ic_ml"], 3)}（規則版 {n(D[13]["ml"]["ic_rule"], 3)}）</p>' if D[13]['ml'].get('ok') else ''))
    # S14
    f14 = D[14]['funnel']; ok14 = [r for r in D[14]['rows'] if r['ok'] or r['soft']]
    funnel = '<div class="fn">' + ''.join(f'<div><span>{E(f["s"])}</span>{bar(f["n"], max(f14[0]["n"], 1), "var(--bl)")}<b>{f["n"]}</b></div>' for f in f14) + '</div>'
    card(14, '量化選股漏斗', funnel, '<h3>硬性全數通過／Z 軟性入選</h3>' + tbl(['股票', '結果', '平均Z', 'Score', '通過'], [[E(r['stk']), tag(r['t']), n(r['z'], 2), n(r['score'], 1), f'{r["ps"]}/10'] for r in ok14[:8]], '本日無入選個股'))
    # S15
    d15 = D[15]; yr = {r['f']: r for r in d15['yr']}
    rows15 = sorted(d15['rows'], key=lambda r: -(r['e10'] if r['e10'] is not None else -9))
    card(15, '因子回測驗證', stats([('組合淨報酬(扣成本)', pc(d15['eq'][-1] - 1), cl(d15['eq'][-1] - 1)), ('年化(淨)', pc(d15['ann']), cl(d15['ann'] or 0)),
                                  ('Sharpe', n(d15['shp'], 2), ''), ('最大回撤', pc(d15['mdd']), 'dn'), ('IC t 值', n(d15['ic_t'], 2), 'up' if abs(d15['ic_t'] or 0) >= 2 else '')]),
         tbl(['因子', '10日勝率', '同日超額', 't 值', '逐年'], [[E(r['f']), pc(r['w10'], 1, False), f'<span class="{cl(r["e10"])}">{pc(r["e10"], 2)}</span>',
             (f'<b class="{cl(r["t10"])}">{n(r["t10"], 1)}</b>' if r['sig10'] == '顯著' else f'<span class="sm">{"樣本不足" if r["sig10"] == "樣本不足" else n(r["t10"], 1)}</span>'),
             tag(yr[r['f']]['stable'], yr[r['f']]['sc']) + f' <span class="sm">{yr[r["f"]]["pos"]}</span>'] for r in rows15]) +
         '<p class="note">t 值粗體＝|t|≥2 統計顯著；同日超額已扣除當天股票池平均報酬。</p>' +
         '<h3>資金規模 × 滑價</h3>' + tbl(['資金', '淨報酬', '年化', '平均滑價'], [[E(c_['cap']), f'<span class="{cl(c_["tot"])}">{pc(c_["tot"])}</span>', pc(c_['ann']), pc(c_['slip'], 2, False)] for c_ in d15['caps']]))

    # ---- 多訊號共振 ----
    bull_src = {
        '強度≥80': [r['stk'] for r in D[2]['stock'] if r['score'] >= 80],
        '突破成本區': [r['stk'] for r in D[4]['rows'] if r['st'] == '突破成本區'],
        f'集中度≥{CONC_TIER_MID:.0%}': [r['stk'] for r in D[7]['rows'] if (r['c5'] or 0) >= CONC_TIER_MID],
        '吸籌': [r['stk'] for r in D[9]['rows'] if r['t'] in ('吸籌', '疑似吸籌')],
        '轉多': [r['stk'] for r in t11 if r['t'] != '主力轉空'],
        'Score≥65': [r['stk'] for r in D[13]['rows'] if r['score'] >= 65],
        '漏斗/軟性入選': [r['stk'] for r in ok14],
        'ML≥80': [r['stk'] for r in D[13]['rows'] if (r.get('ml') or 0) >= 80],
        '族群建倉': list({s_.strip() for r in D[1]['sect'] if r['t'] and r['branch'] == '追蹤分點合計' for s_ in r['stks'].split('、') if s_ != '—'}),
        '成本附近': list({r['stk'] for r in D[3]['pairs'] if r['st'] == '成本附近'}),
    }
    bear_src = {
        '出貨': [r['stk'] for r in D[10]['rows']],
        '轉空': [r['stk'] for r in t11 if r['t'] == '主力轉空'],
        '隔日沖警報': [a['stk'] for a in D[5]['alerts']],
        '獲利了結風險': list({r['stk'] for r in D[3]['pairs'] if r['st'] == '獲利了結風險'}),
        '賣方集中': [r['stk'] for r in D[7]['rows'] if r['t'] == '賣方集中'],
    }
    def confl(src):
        hit = {}
        for k, lst in src.items():
            for s_ in set(lst):
                hit.setdefault(s_, []).append(k)
        return sorted(hit.items(), key=lambda kv: -len(kv[1]))
    score_of = {r['stk']: r['score'] for r in D[13]['rows']}
    bull = [(s_, v) for s_, v in confl(bull_src) if len(v) >= 2][:10]
    bear = [(s_, v) for s_, v in confl(bear_src) if len(v) >= 2][:10]
    bull_s = {s_ for s_, _ in confl(bull_src)}; bear_s = {s_ for s_, _ in confl(bear_src)}
    conflict = sorted(bull_s & bear_s)[:10]
    focus = f'''<section class="focus">
<div><h2>🔺 偏多共振（≥2 個策略同時出現）</h2>{tbl(['股票', '訊號數', '出現於', 'Score'], [[E(s_), f'<b class="up">{len(v)}</b>', ' '.join(tag(x, 'u') for x in v), n(score_of.get(s_), 1)] for s_, v in bull], '今日無偏多共振個股')}</div>
<div><h2>🔻 偏空共振（≥2 個策略同時出現）</h2>{tbl(['股票', '訊號數', '出現於', 'Score'], [[E(s_), f'<b class="dn">{len(v)}</b>', ' '.join(tag(x, 'd') for x in v), n(score_of.get(s_), 1)] for s_, v in bear], '今日無偏空共振個股')}</div>
</section>
{('<section class="warn"><b>⚠️ 多空訊號互相矛盾（同時出現在偏多與偏空清單，需人工判讀）：</b>' + '、'.join(E(x) for x in conflict) + '</section>') if conflict else ''}'''

    css = CSS + '''
.wrap{max-width:1500px;margin:0 auto;padding:16px 24px 28px;display:flex;flex-direction:column;gap:14px}
.focus{display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:14px}
.focus>div{background:var(--pn);border:1px solid var(--bd);border-radius:10px;padding:12px 14px}
.warn{background:#2a1f05;border:1px solid #5a4410;border-radius:8px;padding:10px 14px;color:#f5c451;font-size:13px}
.cards{columns:3 420px;column-gap:14px}
.card{break-inside:avoid;margin:0 0 14px;padding:12px 14px}
.card h2{display:flex;align-items:center;gap:8px;font-size:15px;margin:0 0 8px}
.card h2 a{margin-left:auto;font-size:12px;color:var(--mu);text-decoration:none;font-weight:400}.card h2 a:hover{color:var(--ac)}
.card h3{font-size:12px;color:var(--mu);margin:10px 0 4px;font-weight:600}
.no{font-size:12px;padding:0 6px}
table{font-size:12.5px}th{position:static;cursor:default;padding:4px 6px;font-size:11.5px}td{padding:3px 6px;white-space:normal}
.st{display:flex;flex-wrap:wrap;gap:6px 16px;margin:0 0 8px}.st div{display:flex;flex-direction:column}.st b{font-size:17px;font-family:ui-monospace,Menlo,Consolas,monospace}.st span{font-size:11px;color:var(--mu)}
.bar{display:inline-block;width:70px;height:7px;background:#1d2732;border-radius:4px;overflow:hidden;vertical-align:middle}.bar i{display:block;height:100%}
.fn div{display:grid;grid-template-columns:150px 1fr 44px;gap:8px;align-items:center;font-size:12px;margin:2px 0}.fn .bar{width:100%}.fn b{text-align:right;font-family:ui-monospace,Menlo,Consolas,monospace}
.none{color:var(--mu);font-size:12px;margin:4px 0}.sm{font-size:11px;color:var(--mu)}
.pill{font-size:11px;padding:0 6px;margin:1px 0;white-space:nowrap}
@media print{body{background:#fff;color:#111}.card,.focus>div{border-color:#ccc;background:#fff}th{background:#eee;color:#333}header{background:#fff}.cards{columns:2}}
@media(max-width:640px){.wrap{padding:12px}.focus{grid-template-columns:1fr}.fn div{grid-template-columns:110px 1fr 36px}}'''
    demo = '<div class="demo">⚠️ DEMO 模擬資料：所有數字為隨機產生，僅供檢查版面。</div>' if USE_DEMO_DATA else ''
    body = ''.join(cards[k] for k in sorted(cards))
    page = f'''<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>券商分點重點摘要 {L:%Y-%m-%d}</title><style>{css}</style></head><body>
<header><a href="index.html">← 回總覽</a><h1>券商分點重點摘要</h1><p class="sub">15 個子策略的今日重點，一頁看完</p>
<div class="meta">資料日期 {L:%Y-%m-%d} ｜ 期間 {IDX[0]:%Y-%m-%d} ～ {IDX[-1]:%Y-%m-%d}（{len(IDX)} 日）｜ 分點 {'、'.join(BR)} ｜ 股票池 {len(UNIV)} 檔 ｜ 分點日覆蓋 {COV_ALL:.1%}（最新日 {COV_LAST}/{len(BR)}）｜ 產生 {dt.datetime.now(TPE):%Y-%m-%d %H:%M}</div></header>
{demo}<div class="wrap">{focus}<div class="cards">{body}</div></div>
<footer>分點資料為交易通路而非投資人身分；共振清單僅表示多個條件同時成立，並非買賣建議。研究用途，不構成投資建議。</footer></body></html>'''
    with open(os.path.join(OUTPUT_DIR, '00_summary.html'), 'w', encoding='utf-8') as f:
        f.write(page)
    print(f'✅ 00 重點摘要 → 00_summary.html  ({len(page) / 1024:.0f} KB)  偏多共振 {len(bull)} 檔、偏空共振 {len(bear)} 檔')

build_summary()

In [ ]:
# ============ 總覽頁 index.html ============
def build_index():
    cards = ''.join(f'''<a class="card ic" href="{p['file']}"><div class="row" style="margin:0"><span class="no">S{p['num']:02d}</span><span class="mu">第{p['chapter']}層</span></div>
<h2 style="margin:8px 0 4px">{p['title']}</h2><p class="note" style="margin:0 0 8px">{p['sub']}</p><div class="hl">{p['headline']}</div></a>''' for p in sorted(PAGES, key=lambda x: x['num']))
    flow = '股票代碼 → 抓取券商分點 → 5/10/20/60 日 → 分點買賣超 → 主力集中度 → 主力成本 → 隔日沖辨識 → 吸籌/出貨辨識 → 主力轉向 → 外資/投信 → 融資融券 → MA/RSI/成交量 → 主力籌碼分數 0～100（規則版＋ML 版）→ 回測未來 5/10/20 日 → 勝率/平均報酬/最大回撤/資金規模滑價'
    html = f'''<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>券商分點量化研究 V1.6</title>
<style>{CSS}
.ic{{text-decoration:none;color:inherit;display:block;transition:border-color .15s}}.ic:hover{{border-color:var(--ac)}}.sumlink{{border-color:var(--ac);background:linear-gradient(90deg,rgba(240,180,41,.12),var(--pn))}}.hl{{color:var(--ac);font-weight:700;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:13px}}</style></head><body>
<header><h1>券商分點 V1.6 量化研究模型</h1><p class="sub">15 個子策略獨立 Dashboard ｜ 分點：{'、'.join(BR)}</p>
<div class="meta">資料日期 {L:%Y-%m-%d} ｜ 期間 {IDX[0]:%Y-%m-%d} ～ {IDX[-1]:%Y-%m-%d}（{len(IDX)} 日）｜ 股票池 {len(UNIV)} 檔 ｜ 分點日覆蓋 {COV_ALL:.1%}（最新日 {COV_LAST}/{len(BR)}）｜ 產生 {dt.datetime.now(TPE):%Y-%m-%d %H:%M}</div></header>
{'<div class="demo">⚠️ DEMO 模擬資料：所有數字為隨機產生，僅供檢查版面。</div>' if USE_DEMO_DATA else ''}
<main><section class="method"><h3>研究流程</h3>{flow}<br><br><b>V1.1 優化重點：</b>動態流動性門檻、EMA 加權與族群建倉、FIFO/LIFO＋時間衰減成本、KDE 成本密集區、T+2/T+3 短沖特徵、K-Means 主力分群、扣當沖的留倉集中度、全市場家數差、隱性吸籌、散戶接盤、ATR／成本區防守線、PCA 降維、walk-forward ML Score、Z-Score 軟性選股、資金規模滑價回測。<br><br><b>最重要的觀念：</b>不要把「某分點買超」直接等於「主力買進」。分點資料 → 行為特徵 → 量化因子 → 回測 → 找出具有統計優勢的條件。</section>
<a class="card ic sumlink" href="00_summary.html"><h2 style="margin:0">📋 一頁式重點摘要 →</h2><p class="note" style="margin:4px 0 0">15 個策略的今日重點與多訊號共振個股，一頁看完（可列印）</p></a>
<div class="grid">{cards}</div></main>
<footer>研究用途，不構成投資建議。</footer></body></html>'''
    with open(os.path.join(OUTPUT_DIR, 'index.html'), 'w', encoding='utf-8') as f:
        f.write(html)
    print('✅ index.html')

build_index()

In [ ]:
# ============ V1.4 自我測試（RUN_SELF_TESTS = True 時執行；只檢查程式邏輯，不是績效驗證） ============
def run_self_tests():
    res = []
    def check(name, fn):
        try:
            ok, msg = fn()
        except Exception as e:
            ok, msg = False, f'例外：{e!r}'
        res.append((name, bool(ok), msg))

    def t_bsum():
        ix = IDX[:3]; c = CODES[0]
        a = pd.DataFrame({c: [1.0, 1.0, 1.0]}, index=ix); b = pd.DataFrame({c: [2.0, np.nan, 2.0]}, index=ix)
        s = _bsum({'A': a, 'B': b})[c].loc[ix].tolist()
        return s == [3.0, 1.0, 3.0], f'缺一個分點的日子仍加總：{s}'
    check('分點缺漏加總（_bsum）', t_bsum)

    def t_inv():
        k = _inventory(list(IDX[-3:]), [10.0, 10.0, -10.0], [100.0, 110.0, np.nan], ['實際均價'] * 3)
        ok = abs(k['fifo'] - 110) < 1e-9 and abs(k['lifo'] - 100) < 1e-9 and abs(k['avg'] - 105) < 1e-9 and abs(k['pos'] - 10) < 1e-9
        return ok, f"買 10@100、10@110、賣 10 → FIFO {k['fifo']}／LIFO {k['lifo']}／AVG {k['avg']}，剩 {k['pos']} 張"
    check('庫存法 AVG／FIFO／LIFO', t_inv)

    def t_kde():
        rng = np.random.default_rng(1)
        p = np.r_[rng.normal(100, 2, 200), rng.normal(130, 2, 100)]; w = rng.uniform(1, 5, 300)
        x, dens, zl, zh, acc, pk, bw, npk = dense_zone(p, w, 0.6)
        return acc >= 0.6 and zl <= pk <= zh and npk >= 2, f'雙峰資料：涵蓋 {acc:.0%}、密集區 {zl:.1f}～{zh:.1f}、波峰 {npk}'
    check('KDE 成本密集區', t_kde)

    def t_rankz():
        z = rank_z(pd.Series([3, 1, np.nan, 2, 5]))
        return bool(np.isnan(z[2]) and z[4] > z[0] > z[3] > z[1] and abs(z.dropna().mean()) < 1e-9), f'{z.round(3).tolist()}'
    check('排名 Z 分數', t_rankz)

    c_ = next((c for c in UNIV if AA[c].notna().sum() > 60), None)
    def t_fwd():
        t = IDX[len(IDX) // 2]; p = AA.index.get_loc(t); a = AA[c_]
        exp = a.iloc[p + 6] / a.iloc[p + 1] - 1; got = FWD[5].at[t, c_]
        return bool(np.isclose(exp, got, equal_nan=True)), f'{c_} {t:%Y-%m-%d}：t+1 收盤進場、t+6 收盤出場 = {got:.4%}'
    check('前瞻報酬對齊（t+1 進場）', t_fwd)

    def t_mae():
        t = IDX[len(IDX) // 2]; p = AA.index.get_loc(t)
        lo = _adj_low[c_].iloc[p + 2:p + 7].min(); exp = min(lo / AA[c_].iloc[p + 1] - 1, 0)
        got = MAE[5].at[t, c_]
        return bool(np.isclose(exp, got, equal_nan=True)), f'MAE 取 t+2～t+6 最低價：{got:.4%}'
    check('MAE 視窗（持有期間）', t_mae)

    def t_dtr():
        global NET_T, SELL_T, BUY_T, _ev
        bak = (NET_T, SELL_T, BUY_T, _ev); cut = len(IDX) - 30
        try:
            b3, b1 = _clear_ratio(DT_HORIZON, net=True).iloc[:cut].copy(), _clear_ratio(1).iloc[:cut].copy()
            rng = np.random.default_rng(3)
            fut = (np.arange(len(IDX)) >= cut)[:, None]
            noise = lambda P: P.where(~np.broadcast_to(fut, P.shape), P * rng.uniform(0, 3, P.shape))
            NET_T, SELL_T, BUY_T = noise(NET_T), noise(SELL_T), noise(BUY_T); _ev = NET_T >= DT_MIN_LOTS
            ok = _clear_ratio(DT_HORIZON, net=True).iloc[:cut].equals(b3) and _clear_ratio(1).iloc[:cut].equals(b1)
            return ok, f'竄改 {IDX[cut]:%Y-%m-%d} 之後的分點資料，之前的隔日沖比例' + ('完全不變' if ok else '被改變了（有前視！）')
        finally:
            NET_T, SELL_T, BUY_T, _ev = bak
    check('隔日沖比例無前視', t_dtr)

    def t_cov():
        bad = [b for b in BR if not (NET[b].isna().all(axis=1) == ~BR_OBS[b]).all()]
        return not bad, '覆蓋旗標與分點面板缺漏一致' if not bad else f'不一致：{bad}'
    check('分點日覆蓋旗標', t_cov)

    def t_unit():
        r = DIAG.get('price_ratio')
        return (r is None) or (0.6 < r < 1.6) or ('fix' in DIAG), f"買進均價/收盤 中位數 {r if r is None else round(r, 3)}（修正：{DIAG.get('fix', '無')}）"
    check('張數／金額單位', t_unit)

    def t_nonover():
        s = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0], index=IDX[[0, 1, 2, 10, 11]])
        got = nonoverlap(s, 5).tolist()
        return got == [1.0, 4.0], f'間隔 ≥5 日抽樣：{got}'
    check('t 值不重疊抽樣', t_nonover)

    def t_adj():
        m = FL['adjf'].notna() & FL['cost_raw'].notna()
        ok = np.allclose(FL.loc[m, 'cost_px'], FL.loc[m, 'cost_raw'] * FL.loc[m, 'adjf'])
        last = FL[FL['date'] == IDX[-1]]
        ok2 = np.allclose(last['adjf'].dropna(), 1.0, atol=1e-6)
        return ok and ok2, f"買進價 × 還原係數 = 還原成本；最新日係數 = 1（{DIAG.get('adj_rows', 0):,} 筆跨除權息被調整）"
    check('除權息成本換算', t_adj)

    w = max(len(n) for n, _, _ in res)
    print('── V1.6 自我測試 ──')
    for n, ok, msg in res:
        print(f"  {'✅' if ok else '❌'} {n:<{w}}  {msg}")
    nf = sum(1 for _, ok, _ in res if not ok)
    print(f'  {len(res) - nf}/{len(res)} 通過' + ('' if not nf else '  ⚠️ 有測試失敗，請把這段輸出貼給我'))
    return nf == 0

if RUN_SELF_TESTS:
    SELF_TEST_OK = run_self_tests()

# ---- V1.6：排程模式的建置資訊與失敗條件 ----
BUILD_INFO = dict(data_date=f'{L:%Y-%m-%d}', generated=dt.datetime.now(TPE).strftime('%Y-%m-%d %H:%M'), demo=bool(USE_DEMO_DATA),
                  days=len(IDX), branches=BR, coverage=round(COV_ALL, 4), latest_branches=COV_LAST,
                  trunc=round(TRUNC_ALL, 4), survivorship_share=round(SURV['share'], 4),
                  self_test=(globals().get('SELF_TEST_OK') if RUN_SELF_TESTS else None),
                  pages=[p['file'] for p in sorted(PAGES, key=lambda x: x['num'])])
with open(os.path.join(OUTPUT_DIR, 'build_info.json'), 'w', encoding='utf-8') as f:
    json.dump(BUILD_INFO, f, ensure_ascii=False, indent=1)
print('build_info：', json.dumps(BUILD_INFO, ensure_ascii=False))
if RUN_MODE == 'ci':
    _fail = []
    if RUN_SELF_TESTS and not globals().get('SELF_TEST_OK'):
        _fail.append('自我測試失敗')
    if COV_LAST == 0:
        _fail.append(f'最新交易日 {L:%Y-%m-%d} 沒有任何分點資料（富邦尚未更新或被擋）')
    if _fail:
        raise RuntimeError('排程中止，不發佈網頁：' + '；'.join(_fail))


In [ ]:
# ============ 打包下載 & Colab 內預覽 ============
from IPython.display import HTML, display

def show_dashboard(n=0, height=900):
    """n=0 顯示總覽；'summary' 顯示一頁式重點摘要；1～15 顯示對應子策略"""
    fn = 'index.html' if n == 0 else '00_summary.html' if n == 'summary' else next(p['file'] for p in PAGES if p['num'] == n)
    with open(os.path.join(OUTPUT_DIR, fn), encoding='utf-8') as f:
        html = f.read()
    import html as _h
    display(HTML(f'<iframe srcdoc="{_h.escape(html)}" style="width:100%;height:{height}px;border:0;background:#0a0e13"></iframe>'))

# V1.3：只打包 HTML（不含 _cache 數千個快取檔與原始 CSV），zip 放 Colab 本機不寫回雲端硬碟
import zipfile
_zip_dir = '/content' if os.path.isdir('/content') else os.path.dirname(OUTPUT_DIR)
zip_path = os.path.join(_zip_dir, f'broker_dashboards_{L:%Y%m%d}.zip')
_htmls = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.html'))
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in _htmls:
        zf.write(os.path.join(OUTPUT_DIR, fn), fn)
print(f'共 {len(_htmls)} 個 HTML，{os.path.getsize(zip_path) / 1024:.0f} KB')
print('打包完成：', zip_path)
for p in sorted(PAGES, key=lambda x: x['num']):
    print(f"  S{p['num']:02d} {p['title']:<16} {p['headline']}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass

In [ ]:
if RUN_MODE != 'ci':
    show_dashboard('summary')   # 改成 0（總覽）或 1～15 預覽其他頁